<a href="https://colab.research.google.com/github/bogdanparvu18/msc-graduate-project/blob/main/notebooks/phase3/phase3_02_grounded_question_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 3 – Sector 2

## Build or verify the S8 inventory and generate grounded questions

The default `inventory_only` mode runs on a CPU VM and makes zero OpenAI calls. Allow the Drive mount when prompted. The default output root is `/content/drive/MyDrive/MMVQA_Clinical/outputs/phase3/sector2`. Old databases from earlier notebook versions can remain in place.

1. Section 1 imports the seven V13 grammar/semantics pairs, the verified S1 release and the S8 clinical evidence metadata. It verifies the source bytes on the VM. S8 image pixels are read by the eventual multimodal consumer.
2. The execution cell computes `inventory_id` from these inputs and the code that constructs the census. It looks up the manifest in `real_s8/inventories/<inventory_id>/inventory.json`. Every invocation streams the referenced Drive database to a temporary file on the VM and calculates its actual SHA-256.
3. If the manifest, digest, database identity and SQLite integrity checks pass, a complete census is reused. A verified partial census resumes from the recorded recipe count. If the manifest or its database is missing or inconsistent, the census starts at zero. A new release or changed eligibility rules have their own inventory ID.
4. The writer commits progress locally, publishes verified partial SQLite snapshots at the configured interval and always publishes the final complete database. It publishes the snapshot first and the updated manifest last. Read the database path, SHA-256 and recipe count in the manifest. Use one writer per inventory directory.

For generation select `real_openai` in the first cell, enable the Colab secret `OPENAI_API_KEY`, and run all cells. To switch after an inventory-only census on the same VM, rerun the first cell with the new mode and then the execution cell. The model is `gpt-6-luna`. The default per-execution limit is 12 HTTP calls shared with advisory review. Settings control the number of questions per instance and the API budget. Python derives the reference answers and validates each candidate. The model receives only the public wording request.


In [1]:
import ast
import builtins
import copy
import datetime as dt
import errno
import gzip
import hashlib
import inspect
import io
import itertools
import json
import math
import multiprocessing
import os
import re
import shutil
import signal
import sqlite3
import subprocess
import symtable
import sys
import tempfile
import unicodedata
import uuid
import zlib
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from contextlib import ExitStack, contextmanager
from http.client import IncompleteRead
from functools import lru_cache
from importlib.metadata import PackageNotFoundError, version as package_version
from importlib.util import find_spec
from multiprocessing.connection import wait as wait_for_workers
from pathlib import Path, PurePosixPath
from queue import SimpleQueue
from types import CellType, FunctionType, ModuleType
from typing import Callable
from urllib.error import HTTPError
from urllib.parse import quote, unquote, urlparse
from urllib.request import HTTPRedirectHandler, Request, build_opener, urlopen

try:
    import fcntl
except ImportError:
    fcntl = None
    import msvcrt
else:
    msvcrt = None

from IPython import get_ipython

# Both modes use the same verified inputs and Parquet reader.
MODE = "real_openai"  # @param ["inventory_only", "real_openai"]
packages = {"jsonschema": "jsonschema>=4.18,<5", "tqdm": "tqdm>=4", "pyarrow": "pyarrow"}
missing = [dist for module, dist in packages.items() if find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
# Question validation uses a pinned language model during generation.
# Inventory-only runs read evidence and do not install or load the language model.
def _installed_version(distribution):
    try:
        return package_version(distribution)
    except PackageNotFoundError:
        return None


spacy = None
if MODE == "real_openai":
    _language_deps = []
    if _installed_version("spacy") != "3.8.11":
        _language_deps.append("spacy==3.8.11")
    if _installed_version("en-core-web-sm") != "3.8.0":
        _language_deps.append(
            "https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl"
            "#sha256=1932429db727d4bff3deed6b34cfc05df17794f4a52eeb26cf8928f7c1a0fb85"
        )
    if _language_deps:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *_language_deps])
    import spacy

    if spacy.__version__ != "3.8.11":
        raise RuntimeError("The kernel already loaded a different spaCy version. Restart the session.")

from jsonschema import Draft202012Validator
from tqdm.auto import tqdm
from tqdm.std import tqdm as text_progress

import pyarrow.parquet as pq

try:
    from google.colab import drive, userdata
except ImportError:
    drive, userdata = None, None

# All engine components share these explicitly imported dependencies.
ENGINE_IMPORTS = {
    name: globals()[name]
    for name in (
        "ThreadPoolExecutor", "SimpleQueue", "multiprocessing", "wait_for_workers",
        "CellType", "FunctionType",
        "errno", "fcntl", "msvcrt", "zlib", "contextmanager", "io", "ExitStack", "IncompleteRead", "tqdm",
        "ast", "builtins", "symtable", "copy", "dt", "gzip", "hashlib", "inspect", "itertools", "json",
        "math", "os", "re", "shutil", "signal", "sqlite3", "tempfile", "unicodedata", "uuid",
        "Counter", "defaultdict", "lru_cache", "package_version", "Path", "PurePosixPath",
        "ModuleType", "Callable", "HTTPError", "quote",
        "unquote", "urlparse", "HTTPRedirectHandler", "Request", "build_opener", "urlopen",
        "Draft202012Validator", "pq", "userdata", "spacy",
    )
}


In [2]:
# Optional persistence for campaign outputs. An image Drive is unnecessary.
MOUNT_DRIVE = True  # @param {type:"boolean"}
# Inventory and generation enumerate every S8 recipe and every V13 task.
QUESTIONS_PER_INSTANCE = 1  # @param {type:"integer"}
# Within this campaign, each instance may spend this many wording attempts to reach its target.
ATTEMPT_BUDGET = 3  # @param {type:"integer"}
BATCH_SIZE = 100  # @param {type:"integer"}
# Zero allows the run to process consecutive batches until its API budget is reached.
MAX_BATCHES_PER_RUN = 0  # @param {type:"integer"}
# Each quota is in HTTP calls. Zero per run disables dispatch; zero lifetime means uncapped.
MAX_HTTP_CALLS = 12  # @param {type:"integer"}
MAX_TOTAL_API_CALLS = 0  # @param {type:"integer"}
# Advisory review shares both HTTP budgets with generation.
INTERPRETATION_REVIEW_ENABLED = True  # @param {type:"boolean"}
MAX_REVIEW_HTTP_CALLS = 2  # @param {type:"integer"}
REVIEWER_MODEL = "gpt-6-luna"  # @param {type:"string"}
# Recipe windows are divided among CPU processes. Zero selects the worker count for this VM.
INVENTORY_WINDOW_SIZE = 5000  # @param {type:"integer"}
INVENTORY_WORKERS = 0  # @param {type:"integer"}
# Save completed recipes in S8 order. A checkpoint can be smaller than a dispatch window.
CHECKPOINT_RECIPES = 5000  # @param {type:"integer"}
# Publish a resumable full database after this many locally committed recipes and at completion.
SNAPSHOT_EVERY_RECIPES = 50000  # @param {type:"integer"}
# Capture the latest main for a new source release. A normal restart reuses its saved revision.
REFRESH_GITHUB_MAIN = False  # @param {type:"boolean"}
LOCAL_WORK = "/content/mmvqa_phase3_sector2" if Path("/content").is_dir() else "./mmvqa_phase3_sector2"
# Keep LOCAL_WORK on the VM disk. Point persistent output at the same Drive folder on each VM to reuse a verified snapshot.
OUTPUT_OVERRIDE = ""  # @param {type:"string"}


### Runtime setup

Run the cells in order. Each component cell registers its implementation in the shared Python engine. Section 1 ends with the executable import for this session.


In [3]:
engine = {"__name__": "mmvqa_sector2_notebook_engine", **ENGINE_IMPORTS}
component_hashes = {}
component_sources = {}
engine["component_sources"] = component_sources

def sector2_component(line, cell):
    name = line.strip()
    exec(compile(cell, "<mmvqa-sector2/" + name + ">", "exec"), engine)
    component_sources[name] = cell
    component_hashes[name] = hashlib.sha256(cell.encode("utf-8")).hexdigest()

get_ipython().register_magic_function(sector2_component, magic_kind="cell", magic_name="sector2_component")


### 1 · Import V13, Sector 1 and S8 metadata onto the VM

Both `inventory_only` and `real_openai` import the seven grammar/semantics pairs, the pinned S1 release (including the applicable rule matrix and approved terminology) and the published S8 evidence metadata (including recipes, visual descriptors and private ground truth). Run the import cell at the end of this section to prepare the verified inputs for this VM session.


In [4]:
%%sector2_component sources
def _wire_bytes(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"),
                      allow_nan=False).encode("utf-8")


def wire_digest(payload):
    return hashlib.sha256(_wire_bytes(payload)).hexdigest()


# Read immutable GitHub inputs over HTTPS and verify their contents after staging.

SECTOR2_MODES = frozenset({"inventory_only", "real_openai"})
MMVQA_REPO = "bogdanparvu18/msc-graduate-project"
CONTRACT_PREFIX = "outputs/phase3/sector2/contracts/v13"
CONTRACT_MANIFEST_NAME = "contracts.v13.manifest.json"
LATEST_PATH = "outputs/phase3/sector1/latest.json"
V13_TASKS = frozenset({
    "evidence_frame_localization", "finding_category", "finding_identification",
    "observed_anchor_order", "optional_spatial_localization",
    "temporal_relative_evidence_localization", "verified_reference_comparison",
})
V13_CONTRACT_NAMES = frozenset(
    f"{task}.{kind}.v13.json" for task in V13_TASKS for kind in ("grammar", "semantics")
)


def _source_assert(condition, message):
    if not condition:
        raise ValueError("source_preflight: " + message)


def _byte_sha(data):
    return hashlib.sha256(data).hexdigest()


def _s1_json(data):
    def unique(pairs):
        result = {}
        for name, value in pairs:
            _source_assert(name not in result, "duplicate JSON field: " + name)
            result[name] = value
        return result
    def invalid_number(value):
        raise ValueError("Non-finite JSON number: " + value)
    return json.loads(data, object_pairs_hook=unique, parse_constant=invalid_number)


def _clean_repo_path(path):
    _source_assert(isinstance(path, str) and path, "invalid repository path")
    parsed = PurePosixPath(path)
    _source_assert(not parsed.is_absolute() and parsed.as_posix() == path
                   and path != "." and ".." not in parsed.parts, "invalid repository path")
    return path


def _commit_id(value):
    _source_assert(isinstance(value, str) and re.fullmatch("[a-f0-9]{40}", value),
                   "an immutable GitHub commit is required")
    return value


def _raw_at_commit(commit, path):
    url = f"https://raw.githubusercontent.com/{MMVQA_REPO}/{_commit_id(commit)}/{quote(_clean_repo_path(path))}"
    req = Request(url, headers={"User-Agent": "MMVQA-Sector2-Colab"})
    with urlopen(req, timeout=120) as response:
        return response.read()


def _github_metadata(endpoint):
    url = f"https://api.github.com/repos/{MMVQA_REPO}/{endpoint}"
    req = Request(url, headers={"User-Agent": "MMVQA-Sector2-Colab",
                               "Accept": "application/vnd.github+json"})
    with urlopen(req, timeout=60) as response:
        return _s1_json(response.read())


def _file_record(commit, path, github_json):
    endpoint = f"contents/{quote(_clean_repo_path(path))}?ref={_commit_id(commit)}"
    row = (github_json or _github_metadata)(endpoint)
    _source_assert(isinstance(row, dict) and row.get("type") == "file" and row.get("path") == path
                   and type(row.get("size")) is int and row["size"] >= 0
                   and isinstance(row.get("sha"), str) and re.fullmatch("[0-9a-f]{40}", row["sha"]),
                   "invalid GitHub file identity: " + path)
    return row


def _read_git_blob(commit, record, cache_dir, read_bytes, *, destination=None, progress=None):
    """Admit a Git-pinned bootstrap file using its blob ID; compute its audit SHA-256 once."""
    repo_path, size, blob = record["path"], record["size"], record["sha"]
    path = Path(destination) if destination else Path(cache_dir) / "git_blobs" / blob
    item = {"relative_path": path.name, "size_bytes": size, "git_blob_sha": blob, "capture_bytes": True}
    values = []
    def capture(record, local, data):
        values.append(data)
    try:
        sync_s8_files([item], path.parent, lambda _: io.BytesIO((read_bytes or _raw_at_commit)(commit, repo_path)),
                      digest_kind="git_blob_sha1", label="Git · " + PurePosixPath(repo_path).name,
                      on_verified=capture, progress=progress)
    except RuntimeError as exc:
        raise ValueError(f"Git object import failed for {repo_path}: {exc}") from exc
    data = values[0]
    return data, {"path": repo_path, "commit": commit, "git_blob_sha": blob, "bytes": size, "sha256": _byte_sha(data)}


def read_github_snapshot_file(commit, path, cache_dir, *, github_json=None, read_bytes=None, progress=None):
    row = _file_record(commit, path, github_json)
    return _read_git_blob(commit, row, cache_dir, read_bytes, progress=progress)


def _write_atomic(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=path.parent, delete=False) as file:
        staged = Path(file.name)
        try:
            file.write(data)
            file.flush()
            os.fsync(file.fileno())
            os.replace(staged, path)
        finally:
            staged.unlink(missing_ok=True)


def _source_index(rows, keys):
    index = {}
    for row in rows:
        identity = tuple(row[key] for key in keys)
        _source_assert(identity not in index, "duplicate index: " + str(identity))
        index[identity] = row
    return index


def validate_contract_pairs(contracts):
    _source_assert(set(contracts) == V13_TASKS, "expected seven V13 pairs")
    for task, pair in contracts.items():
        sem, gram = pair["semantics"], pair["grammar"]
        _source_assert(sem["task_id"] == gram["task_id"] == task, "task_id mismatch")
        _source_assert(sem["grammar_file"] == f"{task}.grammar.v13.json" and
                       gram["semantics_file"] == f"{task}.semantics.v13.json", "peer mismatch")
        _source_assert(sem["schema_version"] == "mmvqa-task-semantics-v13" and
                       gram["schema_version"] == "mmvqa-task-grammar-v13", "unsupported contract")
        out = gram["output_contract"]
        _source_assert(gram["generation_mode"] == "open_paraphrase" and
                       out["question_generated_by_llm"] is True and out["answer_generated_by_llm"] is False,
                       "unexpected generation policy")
        qschema = out["base_json_schema"]
        _source_assert(qschema["required"] == ["question"] and set(qschema["properties"]) == {"question"}
                       and qschema["additionalProperties"] is False and
                       "enum" not in qschema["properties"]["question"], "invalid question schema")
        Draft202012Validator.check_schema(qschema)
        schema = gram["public_request"]["data_schema"]
        Draft202012Validator.check_schema(schema)
        bindings = gram["public_request"]["field_bindings"]
        _source_assert(set(bindings) == set(schema["required"]) == set(schema["properties"]) and
                       schema["additionalProperties"] is False, "public projection mismatch")
        slot_names = set(gram["slot_bindings"])
        for field, binding in bindings.items():
            ref = binding.get("resolved_slot", binding)
            _source_assert(ref["root"] in gram["binding_roots"] and isinstance(ref["path"], list),
                           "unresolved binding: " + field)
            if "resolved_slot" in binding:
                _source_assert(ref["root"] == "grammar_document" and len(ref["path"]) == 2
                               and ref["path"][0] == "slot_bindings" and ref["path"][1] in slot_names,
                               "unknown public role: " + field)
        profiles = _source_index(sem["profiles"], ["profile_id"])
        common = sem["required_checks"]["common"]
        specific = sem["required_checks"]["additional_by_profile"]
        _source_assert({pid for (pid,) in profiles} == set(specific), "missing profile checks")
        for (pid,), profile in profiles.items():
            if task == "finding_identification":
                _source_assert(
                    profile["public_kind"].get("approved_equivalents") == [],
                    f"{pid}: this runtime supports only fixed KIND labels; "
                    "approved_equivalents must be an empty list",
                )
            _source_assert(set(common + specific[pid]) <= set(sem["check_registry"]), "unknown check")
            _source_assert(profile["terminology_representation"] == {
                "type": "existing_concept_references", "allow_postcoordination": False},
                "postcoordination requires a separate MRCM validator")
            Draft202012Validator.check_schema(profile["parameters_schema"])
    return contracts


def _contract_manifest(raw):
    manifest = _s1_json(raw)
    _source_assert(isinstance(manifest, dict) and manifest.get("schema_version") == "mmvqa-contract-manifest-v1"
                   and manifest.get("algorithm") == "sha256" and manifest.get("contracts_version") == "v13"
                   and isinstance(manifest.get("files"), list), "invalid contract manifest")
    files = {}
    for entry in manifest["files"]:
        _source_assert(isinstance(entry, dict) and set(entry) == {"path", "sha256", "bytes"},
                       "invalid contract manifest entry")
        name, digest, size = entry["path"], entry["sha256"], entry["bytes"]
        _source_assert(isinstance(name, str) and name not in files and PurePosixPath(name).name == name
                       and isinstance(digest, str) and re.fullmatch("[0-9a-f]{64}", digest)
                       and type(size) is int and size > 0, "invalid or duplicate contract manifest entry")
        files[name] = entry
    _source_assert(set(files) == V13_CONTRACT_NAMES, "manifest must identify every grammar and semantics file exactly once")
    return files


def _contract_git_inventory(commit, github_json):
    """Read authoritative blob IDs for the contract directory at the captured commit."""
    rows = (github_json or _github_metadata)(f"contents/{quote(CONTRACT_PREFIX)}?ref={commit}")
    _source_assert(isinstance(rows, list) and len(rows) < 1000, "incomplete GitHub V13 directory listing")
    selected = {}
    for row in rows:
        name = row.get("name") if isinstance(row, dict) else None
        _source_assert(isinstance(name, str), "invalid GitHub V13 directory entry")
        if name != CONTRACT_MANIFEST_NAME and name not in V13_CONTRACT_NAMES:
            _source_assert(not name.endswith((".grammar.v13.json", ".semantics.v13.json")),
                           "unexpected V13 task contract: " + name)
            continue
        _source_assert(name not in selected and row.get("type") == "file"
                       and row.get("path") == f"{CONTRACT_PREFIX}/{name}"
                       and type(row.get("size")) is int and row["size"] > 0
                       and isinstance(row.get("sha"), str) and re.fullmatch(r"[0-9a-f]{40}", row["sha"]),
                       "invalid or duplicate GitHub V13 file identity: " + name)
        selected[name] = row
    published = selected.pop(CONTRACT_MANIFEST_NAME, None)
    missing = V13_CONTRACT_NAMES.difference(selected)
    _source_assert(not missing, "incomplete GitHub V13 contract set: " + ", ".join(sorted(missing)))
    return selected, published


def _parse_contract_pairs(payloads):
    contracts = {}
    for name, data in sorted(payloads.items()):
        task, kind, _, _ = name.split(".")
        contracts.setdefault(task, {})[kind] = _s1_json(data)
    return validate_contract_pairs(contracts)


def _check_contract_directory(directory):
    unexpected = {path.name for path in directory.iterdir()} - (V13_CONTRACT_NAMES | {CONTRACT_MANIFEST_NAME})
    _source_assert(not unexpected, "unexpected staged contract files: " + str(sorted(unexpected)))


def verify_staged_contract_release(staged_directory, expected_manifest_sha256):
    """Explicit read-only check of a previously admitted SHA-256 receipt and its files."""
    directory = Path(staged_directory)
    manifest = (directory / CONTRACT_MANIFEST_NAME).read_bytes()
    _source_assert(_byte_sha(manifest) == expected_manifest_sha256, "staged contract manifest changed")
    entries = _contract_manifest(manifest)
    _check_contract_directory(directory)
    payloads = {}
    for name, row in sorted(entries.items()):
        path = directory / name
        _source_assert(not path.is_symlink(), "contract file is a symlink: " + name)
        data = path.read_bytes()
        _source_assert(len(data) == row["bytes"] and _byte_sha(data) == row["sha256"],
                       "staged contract bytes do not match the manifest: " + name)
        payloads[name] = data
    return {"contracts": _parse_contract_pairs(payloads), "manifest_sha256": expected_manifest_sha256,
            "contract_sha256": {name: row["sha256"] for name, row in entries.items()}, "contract_batch_summary": None}


def open_contract_release(cache_dir, *, online, manifest_path=None, local_contract_dir=None,
                          github_json=None, read_bytes=None, pinned_commit=None, expected_manifest_sha256=None,
                          progress=None):
    """Check GitHub copies against a pinned directory, or use a SHA-256 receipt for local tests."""
    root = Path(cache_dir).expanduser().resolve() / "contract_imports"
    root.mkdir(parents=True, exist_ok=True)
    commit, published = None, None
    git_files = {}
    previous = []
    if online:
        if manifest_path is not None:
            raise ValueError("manifest_path is supported only for offline local tests")
        commit = _commit_id(pinned_commit if pinned_commit is not None
                            else (github_json or _github_metadata)("commits/main").get("sha"))
        git_files, published = _contract_git_inventory(commit, github_json)
        directory = root / ("v13-" + commit)
        if not directory.exists():
            legacy = [p for p in root.glob("v13-" + commit + "-*")
                      if p.is_dir() and not p.is_symlink()
                      and re.fullmatch(r"v13-[0-9a-f]{40}-[0-9a-f]{64}", p.name)
                      and (expected_manifest_sha256 is None or p.name.endswith("-" + expected_manifest_sha256))]
            if len(legacy) == 1:
                directory = legacy[0]
        if not directory.exists():
            previous = sorted((p for p in root.glob("v13-*") if p.is_dir() and not p.is_symlink()),
                              key=lambda p: p.stat().st_mtime_ns, reverse=True)
    else:
        if not local_contract_dir or not Path(local_contract_dir).expanduser().is_dir():
            raise FileNotFoundError("Set LOCAL_CONTRACT_DIR to a directory containing the 14 V13 contracts.")
        local = Path(local_contract_dir).expanduser()
        local_manifest = Path(manifest_path).expanduser() if manifest_path else local / CONTRACT_MANIFEST_NAME
        directory = Path(tempfile.mkdtemp(prefix="v13-offline-", dir=root))
    _source_assert(not directory.is_symlink(), "contract staging directory is a symlink")
    directory.mkdir(parents=True, exist_ok=True)
    _check_contract_directory(directory)
    target = directory / CONTRACT_MANIFEST_NAME
    _source_assert(not target.is_symlink(), "contract manifest is a symlink")
    manifest_data, digest, entries = None, None, {}
    if published is not None:
        manifest_data, ref = _read_git_blob(commit, published, cache_dir, read_bytes,
                                            destination=target, progress=progress)
        digest = ref["sha256"]
    elif not online:
        if not local_manifest.is_file():
            raise FileNotFoundError(f"Offline tests require a V13 SHA-256 receipt: {local_manifest}")
        manifest_data = local_manifest.read_bytes()
        digest = _byte_sha(manifest_data)
    if manifest_data is not None:
        entries = _contract_manifest(manifest_data)
        _source_assert(expected_manifest_sha256 is None or digest == expected_manifest_sha256,
                       "saved contract manifest differs")

    records = []
    for name in sorted(V13_CONTRACT_NAMES):
        row = {"relative_path": name, "capture_bytes": True}
        if online:
            row.update(size_bytes=git_files[name]["size"], git_blob_sha=git_files[name]["sha"], capture_sha256=True)
            if published is not None:
                row["published_sha256"] = entries[name]["sha256"]
                _source_assert(entries[name]["bytes"] == row["size_bytes"],
                               "published byte count disagrees with pinned Git file: " + name)
        else:
            row.update(size_bytes=entries[name]["bytes"], sha256=entries[name]["sha256"])
        records.append(row)

    def content(item):
        # One worker owns this record, including every retry and its transfer origin.
        name = item["relative_path"]
        if not online:
            return (local / name).open("rb")
        if "_origin" not in item:
            for older in previous:
                candidate = older / name
                if not candidate.is_symlink() and candidate.is_file() and candidate.stat().st_size == item["size_bytes"]:
                    item["_origin"] = "vm"
                    return candidate.open("rb")
        item["_origin"] = "github"
        return io.BytesIO((read_bytes or _raw_at_commit)(commit, git_files[name]["path"]))

    payloads, hashes = {}, {}
    def capture(item, path, data):
        name = item["relative_path"]
        sha = item["verified_sha256"] if online else entries[name]["sha256"]
        payloads[name], hashes[name] = data, sha

    try:
        sync = sync_s8_files(records, directory, content, workers=8, progress=progress, on_verified=capture,
                             label="V13 contracts · up to 8 workers",
                             digest_kind="git_blob_sha1" if online else "sha256")
    except RuntimeError as exc:
        raise ValueError(f"V13 contract import failed: {exc}") from exc

    if manifest_data is None:
        # Stable ordering and formatting also reopen receipts from earlier notebook releases.
        receipt = {"schema_version": "mmvqa-contract-manifest-v1", "algorithm": "sha256", "contracts_version": "v13",
                   "files": [{"path": name, "sha256": hashes[name], "bytes": git_files[name]["size"]}
                             for name in sorted(git_files)]}
        manifest_data = (json.dumps(receipt, indent=2, ensure_ascii=False) + "\n").encode("utf-8")
        digest = _byte_sha(manifest_data)
        _source_assert(expected_manifest_sha256 is None or digest == expected_manifest_sha256,
                       "saved contract manifest differs")
    contracts = _parse_contract_pairs(payloads)
    if published is None and (not target.is_file() or target.read_bytes() != manifest_data):
        _write_atomic(target, manifest_data)
    sources = {item["relative_path"]: item.get("_origin", "reused" if online else "vm") for item in records}
    return {
        "contracts": contracts, "registry_commit": commit, "manifest_sha256": digest,
        "manifest_origin": ("github" if published is not None else "github_git_blobs") if online else "local_sidecar",
        "manifest_git_blob_sha": published["sha"] if published else None,
        "contract_git_blob_sha": {name: git_files[name]["sha"] for name in sorted(git_files)},
        "contract_sha256": dict(sorted(hashes.items())), "contract_batch_summary": sync,
        "contract_sync": {"reused_in_place": sorted(n for n, source in sources.items() if source == "reused"),
                          "copied_from_vm": sorted(n for n, source in sources.items() if source == "vm"),
                          "downloaded_from_github": sorted(n for n, source in sources.items() if source == "github")},
        "staged_directory": str(directory), "staged_manifest_path": str(target), "_manifest_bytes": manifest_data,
        "_github_json": github_json, "_read_bytes": read_bytes,
    }


def _source_artifact_record(source, name):
    idx, pubidx = source["_manifest"], source["_published"]
    record, publication = idx[(name,)], pubidx[(name,)]
    _source_assert(all(record[key] == publication[key] for key in ("bytes", "sha256")), "manifest disagreement")
    path = _clean_repo_path(publication["repository_path"])
    _source_assert(path.startswith(source["_layout"] + "/"), "artifact outside run")
    return record, path


def read_sector1_artifact(source, name, *, as_bytes=False):
    """Return a value from this verified import. No filesystem or network access."""
    if as_bytes:
        return source["_admitted_bytes"][name]
    return source["_admitted_json"][name]


def _sync_sector1_group(declarations, commit, cache_dir, reader, progress, label):
    """Stage a dependency group in the existing VM object cache and capture verified bytes."""
    groups = {}
    for name, (remote_path, sha, size) in declarations.items():
        _source_assert(isinstance(sha, str) and re.fullmatch("[0-9a-f]{64}", sha),
                       "missing or malformed published SHA-256 for " + remote_path)
        entry = groups.setdefault(sha, {"relative_path": sha, "sha256": sha, "capture_bytes": True,
                                        "remote": _clean_repo_path(remote_path), "names": []})
        if size is not None:
            _source_assert(type(size) is int and size >= 0 and entry.get("size_bytes", size) == size,
                           "invalid or inconsistent size for " + remote_path)
            entry["size_bytes"] = size
        entry["names"].append(name)
    values = {}
    def open_source(item):
        return io.BytesIO((reader or _raw_at_commit)(commit, item["remote"]))
    def capture(item, path, data):
        for name in item["names"]:
            values[name] = data
    try:
        report = sync_s8_files(groups.values(), cache_dir / "objects", open_source, label=label,
                               progress=progress, on_verified=capture)
    except RuntimeError as exc:
        failed = next((entry["remote"] for sha, entry in groups.items() if sha in str(exc)), label)
        raise ValueError(f"S1 import failed for {failed}: {exc}") from exc
    return values, report


def load_verified_sector1(release, cache_dir=None, *, progress=None):
    """Import the published S1 control manifests, task inputs and binding dependencies once."""
    cache_dir = Path(cache_dir or ".mmvqa-cache")
    commit, contracts = _commit_id(release["registry_commit"]), release["contracts"]
    read_bytes = release["_read_bytes"]
    latest_raw, latest_ref = read_github_snapshot_file(
        commit, LATEST_PATH, cache_dir, github_json=release["_github_json"], read_bytes=read_bytes, progress=progress)
    latest = _s1_json(latest_raw)
    run, data_commit = latest["run_id"], _commit_id(latest["data_commit"])
    layout = f"outputs/phase3/sector1/metadata/{run}"
    _source_assert(re.fullmatch(r"\d{8}T\d{6}_\d+Z", run) and latest["layout_root"] == layout and
                   latest["repository"] == MMVQA_REPO and
                   latest["integrity_status"] == "remote_bytes_sha256_verified" and
                   latest["required_sector2_artifacts_published"] is True, "incomplete or unrelated release")
    digests = {"latest.json": latest_ref["sha256"], CONTRACT_MANIFEST_NAME: release["manifest_sha256"],
               **release["contract_sha256"]}
    declarations = {}
    for key in ("files_manifest", "publication_manifest", "drive_artifacts_manifest"):
        path = _clean_repo_path(latest[key + "_path"])
        _source_assert(path.startswith(layout + "/"), "cross-run manifest")
        digest = latest.get(key + "_sha256")
        declarations[key] = (path, digest, None)
        digests[PurePosixPath(path).name] = digest
    manifests, sync = _sync_sector1_group(declarations, data_commit, cache_dir, read_bytes, progress, "S1 manifests")
    manifest = _source_index(_s1_json(manifests["files_manifest"]), ["relative_path"])
    publication = _s1_json(manifests["publication_manifest"])
    _s1_json(manifests["drive_artifacts_manifest"])
    _source_assert(all(latest[k] == publication[k] for k in
                       ("run_id", "layout_root", "root", "repository", "files_manifest_sha256",
                        "drive_artifacts_manifest_path", "drive_artifacts_manifest_sha256")),
                   "incompatible publication manifests")
    published = _source_index(publication["files"], ["relative_path"])
    source = {
        "run_id": run, "data_commit": data_commit, "registry_commit": commit,
        "contracts": contracts, "artifacts": {}, "selected_cells": {},
        "files_manifest_sha256": latest["files_manifest_sha256"], "verified_digests": digests,
        "verified_references": {"latest.json": latest_ref}, "_github_json": release["_github_json"],
        "qa_generation_authorized": all(a.get("qa_generation_authorized") is True for a in (latest, publication)),
        "corpus_export_authorized": all(a.get("corpus_export_authorized") is True for a in (latest, publication)),
        "_cache": cache_dir, "_reader": read_bytes, "_manifest": manifest, "_published": published, "_layout": layout,
        "_admitted_bytes": {}, "_admitted_json": {}, "import_report": {"manifests": sync},
    }

    def admit(names, label):
        group = {}
        for name in sorted(set(names) - source["_admitted_bytes"].keys()):
            _source_assert(PurePosixPath(name).name == name, "expected a flat S1 input name: " + name)
            record, path = _source_artifact_record(source, name)
            group[name] = (path, record["sha256"], record["bytes"])
        values, stats = _sync_sector1_group(group, data_commit, cache_dir, read_bytes, progress, label)
        source["import_report"][label] = stats
        for name, data in values.items():
            digests[name] = group[name][1]
            source["_admitted_bytes"][name] = data
            if name.endswith(".json"):
                source["_admitted_json"][name] = _s1_json(data)

    interface_name = "sector2_binding_contract.json"
    admit([interface_name], "S1 binding interface")
    interface = source["_admitted_json"][interface_name]
    _source_assert(interface["module"] == "sector2_binding_validator.py"
                   and interface["loader"] == "load_sector1_binding_contract", "unsupported S1 binding interface")
    keys = {key for pair in contracts.values() for key in pair["semantics"]["source_binding"]["required_artifacts"]}
    keys.update(("sector2_binding_contract", "phase2_source_snapshot", "snomed_subset_concepts", "snomed_subset_terms"))
    file_keys = {("task_contract_enriched" if key == "task_contract" else key) + ".json": key for key in keys}
    for name in interface["required_artifacts"]:
        _source_assert(isinstance(name, str) and name.endswith(".json"), "unsupported binding artifact")
        file_keys.setdefault(name, name[:-5])
    admit([*file_keys, interface["module"]], "S1 task and binding inputs")
    source["artifacts"] = {key: source["_admitted_json"][name] for name, key in file_keys.items()}
    matrix = source["artifacts"]["coverage_contract"]
    _source_index(matrix, ["task", "variant", "evidence_state"])
    rules = _source_index(source["artifacts"]["task_contract"]["TASK_SPECIFICATION"], ["task"])
    for task, pair in contracts.items():
        sem = pair["semantics"]
        selector = sem["applicability"]["selection"]
        _source_assert(selector == {"task": task, "original_policy_code": "A", "expected_status": "applicable"},
                       "unsupported applicability selector")
        cells = [row for row in matrix if all(row[k] == v for k, v in selector.items())]
        rule = sem["source_binding"]["rule_id"]
        _source_assert(rules[(task,)]["rule_id"] == rule and all(row["rule_id"] == rule for row in cells),
                       "matrix and semantic rules disagree")
        source["selected_cells"][task] = cells
    return source


def load_observation_binding(source, corpus, *, code=None):
    """Bind already admitted S1 artifacts to their S8 corpus using the published v1 constructor."""
    artifacts = source["artifacts"]
    interface = artifacts["sector2_binding_contract"]
    names = {
        "bindings": "contract_terminology_bindings",
        "mappings": "class_crosswalk",
        "coverage": "coverage_contract",
        "observations": "phase2_annotation_provenance",
        "classes": "phase2_classes",
        "snapshot": "phase2_source_snapshot",
    }
    module_name = "sector2_binding_validator.py"
    _source_assert(
        interface.get("schema_version") == "mmvqa-sector2-binding-contract-v1"
        and interface.get("module") == module_name
        and interface.get("loader") == "load_sector1_binding_contract"
        and sorted(interface["required_artifacts"]) == sorted(name + ".json" for name in names.values()),
        "unsupported S1 binding bundle interface",
    )
    # Use the exact immutable bytes captured during this import. Source and build
    # receipts already bind these bytes to the published module's SHA-256.
    admitted = source["_real_runtime_bytes"]["s1_binder"]
    receipt = source["verified_references"]["s1_binder"]
    _source_assert(type(admitted) is bytes and (code is None or code == admitted)
                   and receipt["sha256"] == source["verified_digests"][module_name]
                   and receipt["bytes"] == len(admitted), "S1 binding module differs from the admitted release")
    module = ModuleType("sector2_binding_validator_" + receipt["sha256"][:16])
    exec(compile(admitted, module_name, "exec"), module.__dict__)
    constructor = module.Sector1BindingContract
    _source_assert(tuple(inspect.signature(constructor).parameters) == (*names, "corpus")
                   and constructor.corpus_interface_specification() == interface["corpus_interface"],
                   "unsupported S1 in-memory binding constructor")
    return constructor(**{argument: artifacts[name] for argument, name in names.items()}, corpus=corpus)


def _role_lexical_key(text):
    return " ".join(unicodedata.normalize("NFKC", text).casefold().split())


def _terminology_indices(source):
    if "_terms_index" in source:
        return source["_terms_index"]
    a = source["artifacts"]
    stem = ("dataset_class_id", "mapping_id", "terminology", "target_id")
    lexical = stem + ("term_key", "term", "source_sha256", "source_description_id", "language_refset_id")
    idx = {"class": _source_index(a["phase2_classes"], ["dataset_class_id"]),
           "concept": _source_index(a["terminology_terms"], ["terminology", "term_id", "source_sha256"]),
           "mapping": _source_index(a["class_crosswalk"], stem),
           "alias": _source_index(a["approved_class_synonyms"], lexical),
           "sct": _source_index(a["snomed_subset_concepts"], ["id"]),
           "description": _source_index(a["snomed_subset_terms"], ["description_id", "refset_id"]),
           "vocabulary": defaultdict(list), "potential": defaultdict(set), "owners": defaultdict(set)}
    for row in a["distinct_class_vocabulary"]:
        if row["language"] == "en":
            idx["vocabulary"][row["dataset_class_id"]].append(row)
            idx["potential"][row["dataset_class_id"]].add(row["display_term"])
            for provenance in _s1_json(row["provenance_json"]):
                idx["potential"][row["dataset_class_id"]].add(provenance["term"])
    for row in a["approved_class_synonyms"]:
        if row["language"] == "en":
            idx["potential"][row["dataset_class_id"]].add(row["term"])
    for row in a["phase2_classes"]:
        idx["potential"][row["dataset_class_id"]].update(row["source_label_forms"])
    for cid, surfaces in idx["potential"].items():
        for s in surfaces:
            idx["owners"][_role_lexical_key(s)].add(cid)
    source["_terms_index"] = idx
    return idx


def resolve_class_lexicon(source, class_id, approved_mapping_ids):
    """Pass only mapping IDs from successful observation receipts for this class and current scope."""
    idx = _terminology_indices(source)
    allowed = {"MST", "CEST", "SNOMED_CT"}
    ids = frozenset(approved_mapping_ids)
    def eligible(mapping):
        if not (mapping["dataset_class_id"] == class_id and mapping["mapping_id"] in ids
                and mapping["terminology"] in allowed and mapping["mapping_status"] == "approved"):
            return False
        concept = idx["concept"].get((mapping["terminology"], mapping["target_id"], mapping["source_sha256"]))
        if concept is None:
            return False
        if mapping["terminology"] == "SNOMED_CT":
            cid = mapping["target_id"]
            _source_assert(isinstance(cid, str) and cid.isdigit() and
                           idx["sct"].get((cid,), {}).get("active") == "1", "invalid/inactive SNOMED concept")
        return True

    mappings = {key: m for key, m in idx["mapping"].items() if eligible(m)}
    cls = idx["class"].get((class_id,))
    forms = cls["source_label_forms"] if cls else []
    canonical = forms[0] if len(forms) == 1 and mappings else None
    potential = sorted(idx["potential"].get(class_id, ()))
    admitted = set()
    for v in idx["vocabulary"].get(class_id, ()):
        for p in _s1_json(v["provenance_json"]):
            target = (class_id, p["mapping_id"], p["terminology"], p["target_id"])
            mapping = mappings.get(target)
            if mapping is None or mapping["relation"] != "exact":
                continue
            key = target + (v["term_key"], p["term"], p["source_sha256"],
                            p["source_description_id"], p["language_refset_id"])
            a = idx["alias"].get(key)
            if not (a and a["language"] == "en" and a["rag_decision"] == "accepted_source_synonym"
                    and a["rag_lexical_supported"] is True and a["source_sha256"] == mapping["source_sha256"]
                    and a["rag_bundle_sha256"] == p["rag_bundle_sha256"]):
                continue
            fields = ("dataset_class_id", "mapping_id", "terminology", "target_id")
            if any(target not in {tuple(r[k] for k in fields) for r in _s1_json(row["concept_contexts_json"])}
                   for row in (v, a)):
                continue
            if p["terminology"] == "SNOMED_CT":
                # Description IDs and language refsets have distinct roles; neither becomes a clinical concept.
                desc = idx["description"].get((p["source_description_id"], p["language_refset_id"]))
                if not (desc and desc["active"] == desc["language_active"] == "1"
                        and desc["languageCode"] == "en" and desc["concept_id"] == p["target_id"]
                        and desc["term"] == p["term"] and desc["acceptability"] in {"acceptable","preferred"}):
                    continue
            # No additional context is asserted in the public prompt to license an ambiguous alias.
            if (v["requires_concept_context"] is not False or a["requires_concept_context"] is not False
                    or _s1_json(mapping["required_attribute_constraints_json"])):
                continue
            admitted.add(p["term"])
    accepted = sorted(admitted, key=lambda x: (_role_lexical_key(x), x))
    pool = ([canonical] if canonical else []) + accepted
    public = [s for s in dict.fromkeys(pool) if idx["owners"][_role_lexical_key(s)] == {class_id}]
    return {"canonical": canonical, "public_surfaces": public, "answer_aliases": accepted,
            "potential_aliases": potential, "aliases": potential}


def open_campaign_release(cache_dir, output_dir, *, refresh=False, progress=None):
    """Resolve main on the first run and reuse that revision when resuming a campaign."""
    directory = Path(output_dir) / "source"
    pointer = directory / "active_release.json"
    previous = _s1_json(pointer.read_bytes()) if pointer.is_file() and not refresh else None
    commit = None
    if previous is not None:
        _source_assert(previous["repository"] == MMVQA_REPO, "saved release points to another repository")
        commit = _commit_id(previous["registry_commit"])
        digest = previous["manifest_sha256"]
        _source_assert(isinstance(digest, str) and re.fullmatch("[a-f0-9]{64}", digest), "invalid saved manifest digest")
    release = open_contract_release(
        cache_dir, online=True, pinned_commit=commit,
        expected_manifest_sha256=previous["manifest_sha256"] if previous else None, progress=progress,
    )
    digest = release["manifest_sha256"]
    backup = directory / (digest + ".json")
    # A newly returned bundle already contains the admitted bytes. Compare the
    # durable backup by equality and repair this one copy without rehashing.
    if not backup.is_file() or backup.read_bytes() != release["_manifest_bytes"]:
        _write_atomic(backup, release["_manifest_bytes"])
    if previous is None:
        _write_atomic(pointer, _wire_bytes({"repository": MMVQA_REPO, "registry_commit": release["registry_commit"],
                                            "manifest_sha256": digest}))
    return release


#### S8 evidence metadata and visual references

The Git revision recorded by S1 supplies the S8 catalog, indexes, recipes, clinical evidence and manifests. Section 1 stages the metadata to the VM with progress, checks its published identities and selectively repairs damaged or missing metadata files. The verified S1 binding validator and S8 reader bind each eligible instance to that evidence.

The visual-reference check reads rows from the admitted source, archive-membership and visual manifests. It matches source frames and derived members by canonical path, storage role, declared SHA-256 and any available byte count. The report counts these references. Generating text requires zero JPEG reads, TAR member reads or image transfers.

Descriptions of views, panel order, locations and temporal anchors come from the evidence. Display holds for assisted overlays or ambiguous bindings still apply. A downstream visual consumer must verify the actual bytes before showing or training on an image.


In [5]:
%%sector2_component real
# Bind S1-approved observations to the original ordered S8 evidence.
_S8_READER_PATH = "notebooks/phase3/phase3_01_terminology_mapping.ipynb"


def resolve_real_runtime_components(source):
    """Bind the reader to this run's Git commit and the binder to the verified S1 release."""
    if "_real_runtime_bytes" not in source:
        commit = source["registry_commit"]
        reader, reader_ref = read_github_snapshot_file(
            commit, _S8_READER_PATH, source["_cache"],
            github_json=source["_github_json"], read_bytes=source["_reader"],
        )
        module = source["artifacts"]["sector2_binding_contract"]["module"]
        if module != "sector2_binding_validator.py":
            raise ValueError("unsupported_S1_binding_module_interface")
        binder = read_sector1_artifact(source, module, as_bytes=True)
        record, path = _source_artifact_record(source, module)
        binder_ref = {
            "path": path, "commit": source["data_commit"], "sha256": record["sha256"],
            "bytes": record["bytes"], "files_manifest_sha256": source["files_manifest_sha256"],
        }
        source["verified_references"].update(s8_reader=reader_ref, s1_binder=binder_ref)
        source["verified_digests"][_S8_READER_PATH] = reader_ref["sha256"]
        source["_real_runtime_bytes"] = {"s8_reader": reader, "s1_binder": binder}
    return {name: dict(source["verified_references"][name]) for name in ("s8_reader", "s1_binder")}


def _s8_reader_env(reader_bytes):
    """Load the supported reader declarations from the exact bytes recorded in the build."""
    if pq is None:
        raise ImportError("Real S8 indexing requires the pyarrow package in the first notebook cell.")
    env = {
        "copy": copy, "gzip": gzip, "math": math, "os": os, "shutil": shutil, "tempfile": tempfile,
        "Counter": Counter, "PurePosixPath": PurePosixPath, "Callable": Callable, "quote": quote,
        "urlparse": urlparse, "Request": Request, "urlopen": urlopen, "pq": pq, "json": json,
        "hashlib": hashlib, "re": re, "Path": Path, "defaultdict": defaultdict,
        "DEFAULT_REPOSITORY": MMVQA_REPO, "S8_ROOT": "outputs/phase2/sector8",
        "CATALOG_PATH": "configs/sector8_archive_state.json", "INDEX_PATH": "configs/sector8_github_storage.json",
        "_BLOCK_SIZE": 1024**2, "_SHA256": re.compile(r"[a-f0-9]{64}\Z"),
        "_HASH": re.compile(r"[a-f0-9]{64}\Z"), "_COMMIT": re.compile(r"[a-f0-9]{40}\Z"),
        "CONFIG": {"progress": True, "demo_variants": []},
    }
    classes = {"IntegrityError", "S8Store", "S8Benchmark", "Corpus"}
    functions = set("safe_relative_path file_digest verify_file _size _hash canonical_clean "
                    "canonical_json canonical_sha256 _canonical _identity _records _clean _decoded "
                    "s8_semantic_hash _frame_key".split())
    expected = classes | functions | {"S8_TABLE_NAMES"}
    # Look up declarations by name. Moving a cell does not change the interface.
    # Other cells may contain Colab magics and are never executed or imported.
    marker = re.compile(r"^(?:(?:class|def)\s+(?:" + "|".join(sorted(classes | functions)) +
                        r")\b|S8_TABLE_NAMES\s*=)", re.MULTILINE)
    document = json.loads(reader_bytes)
    if document.get("nbformat") != 4:
        raise ValueError("unsupported_S8_reader_notebook_format")
    declarations = {}
    for cell in document["cells"]:
        if cell.get("cell_type") != "code":
            continue
        text = "".join(cell["source"])
        if not marker.search(text):
            continue
        for node in ast.parse(text).body:
            if isinstance(node, ast.FunctionDef) and node.name in functions:
                name = node.name
            elif isinstance(node, ast.ClassDef) and node.name in classes:
                name = node.name
            elif (isinstance(node, ast.Assign) and len(node.targets) == 1
                  and getattr(node.targets[0], "id", "") == "S8_TABLE_NAMES"):
                name = "S8_TABLE_NAMES"
                table_names = ast.literal_eval(node.value)
                if not isinstance(table_names, (list, tuple)) or not all(isinstance(x, str) for x in table_names):
                    raise ValueError("unsupported_S8_table_declaration")
            else:
                continue
            if name in declarations:
                raise ValueError("duplicate_S8_reader_declaration:" + name)
            declarations[name] = node
    if set(declarations) != expected:
        raise ValueError("missing_S8_reader_declarations:" + ",".join(sorted(expected - declarations.keys())))
    exec(compile(ast.Module(list(declarations.values()), []), _S8_READER_PATH, "exec"), env)
    return env


def _s8_native_units_reader(binder):
    """Use the very same unit resolver as the verified binder instance."""
    reader = inspect.getclosurevars(binder.evaluate).nonlocals.get("_binding_units")
    if not inspect.isfunction(reader) or list(inspect.signature(reader).parameters) != ["corpus", "recipe"]:
        raise ValueError("unsupported_S1_unit_reader_interface")
    return reader


class _ReadOnlyBindingJSON:
    """Keep verified JSON values shareable with their original JSON types and encoding."""
    __slots__ = ()

    def _readonly(self, *args, **kwargs):
        raise TypeError("Verified S1/S8 binding values are read-only")

    __init__ = __setitem__ = __delitem__ = _readonly

    def __deepcopy__(self, memo):
        return self


class _BindingObject(_ReadOnlyBindingJSON, dict):
    __slots__ = ()
    clear = pop = popitem = setdefault = update = __ior__ = _ReadOnlyBindingJSON._readonly


class _BindingArray(_ReadOnlyBindingJSON, list):
    __slots__ = ()
    append = clear = extend = insert = pop = remove = reverse = sort = _ReadOnlyBindingJSON._readonly
    __iadd__ = __imul__ = _ReadOnlyBindingJSON._readonly


def _freeze_binding_json(value):
    if isinstance(value, dict):
        result = dict.__new__(_BindingObject)
        dict.update(result, {k: _freeze_binding_json(v) for k, v in value.items()})
        return result
    if isinstance(value, (list, tuple)):
        result = list.__new__(_BindingArray)
        list.extend(result, (_freeze_binding_json(v) for v in value))
        return result
    if value is None or type(value) in (str, int, float, bool):
        return value
    raise TypeError(f"Expected JSON in a verified S1/S8 binding, got {type(value).__name__}")


def _s8_binding_preparer(binder):
    """Prepare the pinned reader and isolate this inventory's immutable S1 snapshot."""
    reader = _s8_native_units_reader(binder)
    native_evaluate = binder.evaluate.__func__
    unit_slot = native_evaluate.__code__.co_freevars.index("_binding_units")
    original_cells = native_evaluate.__closure__
    runtime = copy.copy(binder)
    runtime.snapshot = _freeze_binding_json(binder.snapshot)

    def prepare(source_recipe):
        recipe = _freeze_binding_json(source_recipe)
        # Only the release's native resolver may produce these units. It validates
        # recipe identity, geometry, scope, source links and annotations before reuse.
        corpus = runtime.corpus
        units = _freeze_binding_json(reader(corpus, recipe))

        def prepared_units(request_corpus, request_recipe):
            if request_corpus is not corpus or request_recipe is not recipe:
                raise RuntimeError("Prepared S8 units requested for a different corpus or recipe")
            return units

        # Use the pinned function body and all its rule bindings. Substitute only
        # this recipe's unit reader, without touching the imported module or binder.
        cells = original_cells[:unit_slot] + (CellType(prepared_units),) + original_cells[unit_slot + 1:]
        clinical_check = FunctionType(native_evaluate.__code__, native_evaluate.__globals__,
                                      native_evaluate.__name__, native_evaluate.__defaults__, cells)
        clinical_check.__kwdefaults__ = native_evaluate.__kwdefaults__

        def evaluate(**request):
            return clinical_check(runtime, recipe, **request)

        return recipe, units, evaluate

    return prepare


def _vm_token(stat):
    return (stat.st_dev, stat.st_ino, stat.st_size, stat.st_mtime_ns, stat.st_ctime_ns)


def _vm_path_token(path):
    try:
        return _vm_token(path.stat())
    except FileNotFoundError:
        return None


def _inspect_vm_file(path, record, digest_kind="sha256"):
    """Hash the closed transfer's VM bytes; optionally retain those exact bytes for a parser."""
    try:
        with path.open("rb") as file:
            token = _vm_token(os.fstat(file.fileno()))
            if "size_bytes" in record and token[2] != record["size_bytes"]:
                return None
            if digest_kind == "git_blob_sha1":
                digest = hashlib.sha1(f"blob {token[2]}\0".encode("ascii"))
                expected = record["git_blob_sha"]
            else:
                digest = hashlib.sha256()
                expected = record["sha256"]
            if record.get("capture_bytes"):
                data = file.read()
                digest.update(data)
            else:
                data = None
                for block in iter(lambda: file.read(1024 * 1024), b""):
                    digest.update(block)
            matched = digest.hexdigest() == expected
            if not matched or _vm_path_token(path) != token:
                return None
            if record.get("capture_sha256"):
                # Bind the receipt to the same VM bytes as the Git blob check, before publication.
                sha = _byte_sha(data)
                if "published_sha256" in record and sha != record["published_sha256"]:
                    # The bytes already match the Git blob. A retry cannot reconcile the two declarations.
                    raise RuntimeError("Published SHA-256 disagrees with pinned Git file: " + record["relative_path"])
                record["verified_sha256"] = sha
            return token, data
    except FileNotFoundError:
        return None


def _copy_one_vm_file(record, path, open_source, max_attempts, label, digest_kind, on_transfer):
    path.parent.mkdir(parents=True, exist_ok=True)
    transfer_reported = False
    for attempt in range(1, max_attempts + 1):
        temporary = None
        try:
            with tempfile.NamedTemporaryFile(dir=path.parent, prefix=".mmvqa-import-", delete=False) as output:
                temporary = Path(output.name)
                with open_source(record) as source:
                    shutil.copyfileobj(source, output, length=1024 * 1024)
            if not transfer_reported:
                on_transfer()
                transfer_reported = True
            checked = _inspect_vm_file(temporary, record, digest_kind)
            if checked is None:
                raise ValueError("VM copy differs from its declared content or size")
            verified_token, payload = checked
            # Both names are in the same directory. rename publishes the verified inode atomically.
            os.replace(temporary, path)
            installed_token = _vm_path_token(path)
            if installed_token is None or installed_token[:4] != verified_token[:4]:
                raise OSError("VM file was replaced or modified during publication")
            return installed_token, payload, attempt - 1
        except (OSError, ValueError, EOFError, IncompleteRead) as exc:
            if attempt == max_attempts or (isinstance(exc, OSError) and exc.errno == errno.ENOSPC):
                raise RuntimeError(
                    f"{label}: {record['relative_path']} failed on attempt {attempt}/{max_attempts}: {exc}"
                ) from exc
        finally:
            if temporary is not None:
                temporary.unlink(missing_ok=True)


def _process_vm_path(record, path, open_source, max_attempts, label, digest_kind, on_transfer):
    existed = path.is_file()
    checked = _inspect_vm_file(path, record, digest_kind) if existed else None
    if checked is not None:
        on_transfer()
        token, payload = checked
        return record, path, token, payload, "reused", 0

    token, payload, retries = _copy_one_vm_file(
        record, path, open_source, max_attempts, label, digest_kind, on_transfer
    )
    return record, path, token, payload, "repaired" if existed else "copied", retries


def _run_transfer_batch(work, process, workers, on_transfer, on_verified):
    """Only the coordinator publishes progress, receipts and caller callbacks."""
    if workers == 1:
        for record, path in work:
            on_verified(process(record, path, on_transfer))
        return

    events = SimpleQueue()
    def transferred():
        events.put(("copy", None))

    def finished(future):
        events.put(("done", future))

    with ThreadPoolExecutor(max_workers=workers, thread_name_prefix="s8-io") as pool:
        pending = set()
        def submit_next():
            item = next(work, None)
            if item is not None:
                future = pool.submit(process, *item, transferred)
                pending.add(future)
                future.add_done_callback(finished)

        try:
            # A batch may have 5,000 records. Keep only one active job per worker.
            for _ in range(workers):
                submit_next()
            while pending:
                phase, future = events.get()
                if phase == "copy":
                    on_transfer()
                else:
                    pending.remove(future)
                    on_verified(future.result())
                    submit_next()
        finally:
            for future in pending:
                future.cancel()


def _validate_transfer_workers(workers):
    if type(workers) is not int or workers < 1:
        raise ValueError("Transfer workers must be a positive integer.")


def sync_s8_files(records, destination, open_source, *, batch_size=5000, max_attempts=3,
                  progress=None, label="S8", on_verified=None, receipts=None, digest_kind="sha256", workers=1):
    """Import <=5,000 paths per batch, with one successful on-VM SHA per admitted file."""
    if (type(batch_size) is not int or not 1 <= batch_size <= 5000
            or type(max_attempts) is not int or max_attempts < 1):
        raise ValueError("batch_size must be 1..5000 and max_attempts must be positive")
    _validate_transfer_workers(workers)
    if digest_kind not in {"sha256", "git_blob_sha1"}:
        raise ValueError("Unsupported VM content identity")
    root = Path(destination).expanduser().resolve()
    root.mkdir(parents=True, exist_ok=True)
    stats = dict(files=0, batches=0, copied=0, repaired=0, reused=0, retried=0, verified=0)
    seen = set()
    records = iter(records)
    while batch := list(itertools.islice(records, batch_size)):
        stats["batches"] += 1
        paths = []
        for record in batch:
            name = record["relative_path"]
            relative = PurePosixPath(name)
            if (relative.is_absolute() or not relative.parts or relative.as_posix() != name
                    or ".." in relative.parts or "\\" in name or name in seen):
                raise ValueError(f"Invalid or duplicate {label} destination: {name}")
            if digest_kind == "git_blob_sha1":
                if (not isinstance(record.get("git_blob_sha"), str)
                        or not re.fullmatch(r"[0-9a-f]{40}", record["git_blob_sha"])
                        or "size_bytes" not in record):
                    raise ValueError(f"Missing or invalid pinned Git blob identity for {name}")
            elif not isinstance(record.get("sha256"), str) or not re.fullmatch(r"[0-9a-f]{64}", record["sha256"]):
                raise ValueError(f"Missing or invalid published SHA-256 for {name}")
            if "size_bytes" in record and (type(record["size_bytes"]) is not int or record["size_bytes"] < 0):
                raise ValueError(f"Invalid published size for {name}")
            seen.add(name)
            path = root / name
            if path.resolve() != path:
                raise ValueError(f"Unexpected symlink in VM path {path}")
            paths.append(path)

        with ExitStack() as monitors:
            bars = {}
            counts = {"copy": 0, "verify": 0}
            for phase, action in (("copy", "transferred or reused"), ("verify", "verified VM files")):
                if progress is None:
                    bar = tqdm(total=len(batch), desc=f"{label} · batch {stats['batches']} · {action}",
                               unit="file", mininterval=0.35, leave=True)
                    monitors.callback(bar.close)
                    bars[phase] = bar
                else:
                    progress(phase, stats["batches"], 0, len(batch), label)

            def advance(phase):
                counts[phase] += 1
                if progress is None:
                    bars[phase].update(1)
                else:
                    progress(phase, stats["batches"], counts[phase], len(batch), label)

            def process(record, path, transferred):
                return _process_vm_path(record, path, open_source, max_attempts, label, digest_kind, transferred)

            def admit(result):
                record, path, token, payload, disposition, retries = result
                if receipts is not None:
                    receipts[path] = token
                if on_verified is not None:
                    on_verified(record, path, payload)
                stats[disposition] += 1
                stats["retried"] += retries
                stats["verified"] += 1
                advance("verify")

            _run_transfer_batch(zip(batch, paths), process, min(workers, len(batch)), lambda: advance("copy"), admit)
        stats["files"] += len(batch)
    return stats


class _S8PartReader(io.RawIOBase):
    """Read a verified multipart object in order, opening only one part at a time."""

    def __init__(self, paths):
        super().__init__()
        self.paths = iter(paths)
        self.current = None

    def readable(self):
        return True

    def readinto(self, buffer):
        if not buffer:
            return 0
        while True:
            if self.current is None:
                path = next(self.paths, None)
                if path is None:
                    return 0
                self.current = path.open("rb")
            count = self.current.readinto(buffer)
            if count:
                return count
            self.current.close()
            self.current = None

    def close(self):
        if self.current is not None:
            self.current.close()
        super().close()


def prepare_s8_metadata(env, pin, root, *, batch_size=5000, max_attempts=3,
                        progress=None, open_remote=None):
    """Stage the pinned index and the exact logical files consumed by S1 in the native reader cache."""
    commit = _commit_id(pin["resolved_commit"])
    index_ref = pin["storage_index"]
    if pin["s8_root"] != env["S8_ROOT"] or index_ref["relative_path"] != env["INDEX_PATH"]:
        raise ValueError("Unsupported S8 publication or storage-index path")
    store = env["S8Store"](repo=pin["repository"], ref=commit, cache_dir=root)
    # S1 has already pinned the Git commit and the index digest. Initializing the
    # upstream reader with this identity avoids resolve() fetching an unchanged index.
    store.selected_ref = store.commit = commit
    store._root = store.cache_dir / store.repo.replace("/", "__") / commit

    def remote(relative_path):
        if open_remote is not None:
            return open_remote(relative_path)
        request = Request(store._raw_url(relative_path), headers={"User-Agent": "MMVQA-Sector2-Colab"})
        return urlopen(request, timeout=store.timeout)

    options = dict(batch_size=batch_size, max_attempts=max_attempts, progress=progress)
    def remote_source(record):
        return remote(record["remote_relative_path"])

    def transfer_record(target, source, declaration):
        return {"relative_path": target, "remote_relative_path": source,
                "size_bytes": declaration.get("bytes"), "sha256": declaration.get("sha256")}

    index_path = index_ref["relative_path"]
    index_bytes = {}
    index_record = transfer_record(index_path, index_path, index_ref)
    index_record["capture_bytes"] = True
    index_report = sync_s8_files(
        [index_record], store._root / "publication", remote_source, label="S8 storage index",
        on_verified=lambda row, path, data: index_bytes.update(value=data), **options,
    )
    store.index = _s1_json(index_bytes.pop("value"))
    store.entries = store._validate_index(store.index)
    if store.index["source_catalog_sha256"] != pin["source_catalog_sha256"]:
        raise ValueError("S8 source catalog differs from the pinned S1 snapshot")
    store._index_identity = dict(index_ref)

    required = {}
    logical_records = []
    for declaration in pin["consumed_artifacts"]:
        logical = env["safe_relative_path"](declaration["relative_path"])
        # S8 publication admits metadata documents only. Check logical paths and their
        # transport locators before any logical-file or part request is scheduled.
        parts = PurePosixPath(logical).parts
        if (parts[0] not in {"metadata", "configs", "reports", "tables", "results", "benchmark"}
                or any(part.startswith(".") or part in {"products", "archives", "checkpoints", "github_review"}
                       for part in parts)
                or not logical.lower().endswith((".json", ".parquet", ".json.gz", ".jsonl.gz"))):
            raise ValueError("S8 publication entry is outside the metadata-only import: " + logical)
        entry = store.entries.get(logical)
        if entry is None or logical in required:
            raise ValueError("Missing or duplicate S8 metadata declaration: " + logical)
        if entry["storage"] == "direct":
            if entry["github_relative_path"] != logical:
                raise ValueError("S8 direct metadata locator differs from its logical path: " + logical)
        elif entry["storage"] == "parts":
            for part in entry["parts"]:
                digest = _s8_declared_sha(part["sha256"], logical)
                if part["relative_path"] != "metadata/_github_chunks/" + digest + ".bin":
                    raise ValueError("S8 metadata part is outside its published chunk location: " + logical)
        else:
            raise ValueError("Unsupported S8 metadata storage: " + logical)
        expected = {key: entry[key] for key in ("relative_path", "bytes", "sha256", "storage")}
        if entry["storage"] == "parts":
            expected["parts"] = entry["parts"]
        if declaration != expected:
            raise ValueError("S8 index disagrees with the S1 consumed artifact: " + logical)
        store._cache_path("logical", logical)
        row = transfer_record(logical, entry.get("github_relative_path"), entry)
        row["capture_bytes"] = logical.endswith(".json")
        logical_records.append(row)
        required[logical] = entry

    # The outer synchronizer opens sources only for missing or mismatched logical files.
    # Parts are individually repaired and assembled by streaming. A valid logical file
    # therefore needs neither a GitHub request nor a second cache copy of each part.
    part_reports = {}
    part_receipts = {}
    def logical_source(record):
        logical = record["relative_path"]
        entry = required[logical]
        if entry["storage"] == "direct":
            return remote_source(record)
        unique = {}
        sequence = []
        for part in entry["parts"]:
            name = part["sha256"] + ".bin"
            sequence.append(store._cache_path("parts", name))
            unique.setdefault(name, transfer_record(name, part["relative_path"], part))
        # Separate logical objects may share a part. During this import an unchanged
        # part keeps its receipt, including on an assembly retry after a local write fault.
        parts_dir = store._root / "parts"
        needs_check = []
        for name, row in unique.items():
            path = parts_dir / name
            previous = part_receipts.get(path)
            if previous is None or previous[2] != row["size_bytes"] or _vm_path_token(path) != previous:
                needs_check.append(row)
        part_report = sync_s8_files(needs_check, parts_dir, remote_source, receipts=part_receipts,
                                    label="S8 parts · " + PurePosixPath(logical).name, **options)
        # A new copy of the logical object may be tried more than once; count each actual transfer.
        totals = part_reports.setdefault(logical, Counter())
        totals.update(part_report)
        totals["used_prior_receipt"] += len(unique) - len(needs_check)
        return io.BufferedReader(_S8PartReader(sequence))

    admitted = {}
    json_bytes = {}
    def remember(row, path, data):
        if data is not None:
            json_bytes[row["relative_path"]] = data

    logical_report = sync_s8_files(
        logical_records, store._root / "logical", logical_source, label="S8 metadata",
        on_verified=remember, receipts=admitted, **options,
    )
    store.sector2_sync_summary = {"storage_index": index_report, "logical_metadata": logical_report,
                                  "multipart_when_needed": {k: dict(v) for k, v in part_reports.items()}}

    # The upstream consumer opens already admitted files inside this private reader instance.
    # Preserve published consumption tracking; a later cell may neither fetch nor silently
    # replace input bytes. A changed local file needs a fresh Section 1 import.
    logical_by_path = {store._cache_path("logical", name): name for name in required}
    store.consumed = {}
    def assert_admitted(path, size, sha256):
        path = Path(path)
        logical = logical_by_path.get(path)
        if logical is None or (size, sha256) != (required[logical]["bytes"], required[logical]["sha256"]):
            raise ValueError(f"S8 reader requested a file outside the admitted snapshot: {path}")
        if admitted[path] != _vm_path_token(path):
            raise RuntimeError(f"S8 file changed after import: {path}. Rerun Section 1.")
        return path

    def admitted_load(logical):
        logical = env["safe_relative_path"](logical)
        entry = required.get(logical)
        if entry is None:
            raise ValueError(f"S8 file is outside the Sector 1 snapshot: {logical}")
        path = assert_admitted(store._cache_path("logical", logical), entry["bytes"], entry["sha256"])
        if logical not in store.consumed:
            store.consumed[logical] = {k: entry[k] for k in ("relative_path", "bytes", "sha256", "storage")}
            if entry["storage"] == "parts":
                store.consumed[logical]["parts"] = [dict(p) for p in entry["parts"]]
        return path

    def admitted_json(logical):
        admitted_load(logical)
        data = _s1_json(json_bytes[logical])
        if not isinstance(data, dict):
            raise ValueError(f"Expected a JSON object in {logical}")
        return data

    store.assert_admitted = assert_admitted
    store.load = admitted_load
    store.read_json = admitted_json
    return store


def _s8_visual_relative(name):
    """Require one canonical publication-relative POSIX path."""
    if (not isinstance(name, str) or not name or "\\" in name or name.startswith("/")
            or any(ord(char) < 32 for char in name)
            or any(part in {"", ".", ".."} for part in name.split("/"))):
        raise ValueError(f"Invalid S8 publication-relative path: {name!r}")
    return name


def _s8_declared_sha(value, path):
    if not isinstance(value, str) or re.fullmatch(r"[0-9a-f]{64}", value) is None:
        raise ValueError(f"Invalid declared S8 SHA-256: {path}")
    return value


def _s8_visual_bytes(value):
    # Optional source sizes are nullable in Parquet and may be stored as float64.
    if value is None:
        return None
    if type(value) is int and value >= 0:
        return value
    if type(value) is float and value >= 0 and value.is_integer():
        return int(value)
    raise ValueError(f"Invalid declared S8 byte count: {value!r}")


def s8_visual_reference_summary(corpus):
    """Join admitted publication declarations in memory. No visual paths are opened."""
    if isinstance(corpus, _S8VisualTables):
        sources = ((row["image_relpath"], row) for row in corpus.rows("source_evidence_manifest"))
        visuals = corpus.rows("visual_manifest")
    else:
        sources = corpus.sources_by_path.items()
        visuals = corpus.visual_by_artifact.values()
    claims = {}
    for path, row in sources:
        path = _s8_visual_relative(path)
        digest = _s8_declared_sha(row["sha256"], path)
        if path in claims or row["image_relpath"] != path or row["source_sha256"] != digest:
            raise ValueError(f"Conflicting source-frame identity: {path}")
        claims[path] = ("source", digest, _s8_visual_bytes(row.get("image_size_bytes")))
    originals = len(claims)

    published = corpus.benchmark.archive_manifest
    archive_paths = set()
    for archive in published["archives"]:
        archive_path = _s8_visual_relative(archive["relative_path"])
        if archive_path in archive_paths or archive_path in claims:
            raise ValueError(f"Duplicate S8 archive path: {archive_path}")
        archive_paths.add(archive_path)
        _s8_declared_sha(archive["sha256"], archive_path)
        if _s8_visual_bytes(archive["bytes"]) is None:
            raise ValueError(f"Missing S8 archive byte count: {archive_path}")
        members = archive["members"]
        if "member_count" in archive and archive["member_count"] != len(members):
            raise ValueError(f"Incorrect S8 archive member count: {archive_path}")
        for member in members:
            path = _s8_visual_relative(member["relative_path"])
            size = _s8_visual_bytes(member["bytes"])
            if size is None or path in claims or path in archive_paths:
                raise ValueError(f"Missing size or conflicting S8 member path: {path}")
            claims[path] = ("derived", _s8_declared_sha(member["sha256"], path), size)
    derived = len(claims) - originals
    for field, count in (("archive_count", len(archive_paths)), ("member_count", derived)):
        if field in published and published[field] != count:
            raise ValueError(f"S8 archive manifest disagrees with its declared {field}")

    for visual in visuals:
        path = _s8_visual_relative(visual["relative_path"])
        claim = claims.pop(path, None)
        if claim is None or claim[:2] != (visual["storage"], visual["sha256"]):
            raise ValueError(f"Unbound or duplicate S8 visual-manifest row: {path}")
        size = _s8_visual_bytes(visual.get("bytes"))
        if claim[2] is not None and size is not None and claim[2] != size:
            raise ValueError(f"Conflicting S8 visual byte counts: {path}")
    if claims:
        raise ValueError(f"Published S8 file has no matching visual-manifest row: {next(iter(claims))}")
    return {"verification": "metadata_references_only", "referenced_files": originals + derived,
            "original_frames": originals, "derived_members": derived, "visual_files_opened": 0}


class _S8VisualTables:
    """Read only the columns required to compare the S8 declarations."""

    columns = {
        "source_evidence_manifest": ["image_relpath", "source_sha256", "sha256", "image_size_bytes"],
        "visual_manifest": ["relative_path", "storage", "sha256", "bytes"],
    }

    def __init__(self, env, benchmark, pin):
        self.benchmark = benchmark
        self.released = {row["relative_path"]: row for row in pin["consumed_artifacts"]}
        self.parquet = env["pq"]
        self.clean = env["_clean"]

    def rows(self, name):
        store = self.benchmark.store
        descriptor = self.benchmark.catalog["tables"][name]
        path = self.benchmark._bind(descriptor["relative_path"], descriptor)
        schema_path = self.benchmark._bind(descriptor["schema_path"])
        for name_in_pin in (path, schema_path):
            published = self.released.get(name_in_pin)
            indexed = store.entries[name_in_pin]
            if published is None or any(published[k] != indexed[k] for k in ("bytes", "sha256")):
                raise ValueError(f"S8 table or schema is outside the verified S1 metadata set: {name_in_pin}")
        schema = store.read_json(schema_path)
        parquet = self.parquet.ParquetFile(store.load(path))
        try:
            if (parquet.metadata.num_rows != descriptor["rows"] or parquet.metadata.num_rows != schema["rows"]
                    or parquet.schema_arrow.names != schema["columns"]):
                raise ValueError(f"S8 table schema or row count changed: {name}")
            count = 0
            for batch in parquet.iter_batches(batch_size=5000, columns=self.columns[name]):
                for row in batch.to_pylist():
                    count += 1
                    yield self.clean(row)
            if count != schema["rows"]:
                raise ValueError(f"Incomplete S8 visual preflight table: {name}")
        finally:
            parquet.close()


def make_s8_visual_view(env, bench, pin):
    """Read the two verified visual tables without constructing a second clinical census."""
    store = bench.store
    if (store.repo != pin["repository"] or store.commit != pin["resolved_commit"]
            or store._index_identity != pin["storage_index"]
            or store.index["source_catalog_sha256"] != pin["source_catalog_sha256"]):
        raise ValueError("S8 visual table view differs from the verified S1 publication")
    return _S8VisualTables(env, bench, pin)


def bind_s8_recipe_receipt(env, store, pin):
    """Use this import's receipt for the pinned native reader's repeated recipe scans."""
    if store.repo != pin["repository"] or store.commit != pin["resolved_commit"]:
        raise ValueError("The S8 reader and admitted store must use the same release")
    # The reader's declarations are compiled into a fresh namespace for this import.
    # Corpus.iter_recipes calls verify_file before opening the compressed JSONL. Keep
    # its native row parsing, schema checks and per-recipe semantic identity checks.
    iterator = env["Corpus"].iter_recipes
    if iterator.__globals__ is not env or "verify_file" not in iterator.__code__.co_names:
        raise ValueError("Unsupported native S8 recipe integrity interface")
    env["verify_file"] = store.assert_admitted


def _open_s8_metadata(source, s8_dir):
    # Both modules and their release identities were captured by this Section 1 import.
    env = _s8_reader_env(source["_real_runtime_bytes"]["s8_reader"])
    pin = source["artifacts"]["phase2_source_snapshot"]
    store = prepare_s8_metadata(env, pin, Path(s8_dir))
    bind_s8_recipe_receipt(env, store, pin)
    bench = env["S8Benchmark"](store)
    env["CONFIG"]["demo_variants"] = list(bench.demo_variants)
    return env, bench, pin


def _clinical_s8_corpus(env, bench, pin):
    """Read clinical rows from the admitted files when a census needs them."""
    corpus = env["Corpus"](bench, batch_size=4096)
    if corpus.source_snapshot["consumed_artifacts"] != pin["consumed_artifacts"]:
        raise ValueError("different_S8_artifact_set")
    return corpus


def prepare_real_s8_inputs(source, s8_dir):
    """Admit S8 metadata and its declared visual references for both execution modes."""
    context = _open_s8_metadata(source, s8_dir)
    env, bench, pin = context
    report = s8_visual_reference_summary(make_s8_visual_view(env, bench, pin))
    return context, {"metadata": bench.store.sector2_sync_summary, "visuals": report}


def import_sector2_inputs(config, *, import_components=None):
    """Capture one release for this VM session. No later stage imports these inputs."""
    mode = config["mode"]
    if mode not in SECTOR2_MODES:
        raise ValueError("Unknown Sector 2 import mode: " + str(mode))
    cache = Path(config["cache_dir"]).expanduser().resolve()
    registry = open_campaign_release(cache, config["output_dir"],
                                     refresh=config.get("refresh_github_main", False))
    source = load_verified_sector1(registry, cache)
    upstream_code = resolve_real_runtime_components(source)
    s8_dir = Path(config["s8_dir"]).expanduser().resolve()
    context, report = prepare_real_s8_inputs(source, s8_dir)
    return {"registry": registry, "source": source, "upstream_code": upstream_code,
            "s8_context": context, "s8_vm_sync": report, "s8_dir": str(s8_dir),
            "import_components": dict(import_components or {})}


def _s8_letter_label(index):
    """Return the S8 caption for a zero-based sequential panel (A, ..., Z, AA)."""
    number, label = index + 1, ""
    while number:
        number, letter = divmod(number - 1, 26)
        label = chr(ord("A") + letter) + label
    return label


def _s8_display_review_reason(visuals_by_artifact, raw_units, task_id):
    """Screen the native S8 display using its recorded render metadata and the task."""
    selects_unit = task_id in {
        "evidence_frame_localization", "verified_reference_comparison",
        "observed_anchor_order", "temporal_relative_evidence_localization",
    }
    for unit in raw_units:
        if unit["operation"] == "overlay":
            return "review_assisted_overlay"
        if unit["operation"] not in {"full", "crop"}:
            return "review_unverified_S8_display_annotations"
        panel = unit["panel_index"]
        metadata = visuals_by_artifact[unit["artifact_id"]].get("metadata") or {}
        flags = [metadata.get(key) for key in ("clinical_labels_rendered", "qc_scores_rendered")]
        if panel is None and unit["operation"] == "full":
            if any(flag is True for flag in flags):
                return "review_unverified_S8_display_annotations"
            continue  # Source frames have no S8-rendered caption strip.
        if any(flag is not False for flag in flags):
            return "review_unverified_S8_display_annotations"
        if panel is None:
            continue  # An S8 ROI crop is a standalone view.
        label = unit["geometry"].get("label")
        if label == _s8_letter_label(panel):
            continue
        match = re.fullmatch(r"Frame ([0-9]+)(?: [+-][0-9]+\.[0-9]{3}s)?( target)?", label or "")
        if match is None or match[1] != str(unit["frame_index"]):
            return "review_unverified_S8_panel_label"
        if selects_unit and match[2]:
            return "review_S8_target_marker_may_disclose_answer"
    return None


def _s8_presentation_reference(pin, recipe, raw_units):
    """Identify the unchanged S8 visuals and their public ordinals. Paths are private locators."""
    by_input = defaultdict(list)
    for unit in raw_units:
        by_input[unit["input_id"]].append(unit)
    ordered = []
    for order, item in enumerate(recipe["ordered_inputs"]):
        units = by_input.pop(item["input_id"], [])
        if item["input_order"] != order or not units:
            raise ValueError("S8_display_input_order_or_binding_mismatch")
        width, height = item["width"], item["height"]
        records = []
        for index, unit in enumerate(units):
            panel = unit["panel_index"]
            label = f"image_{order + 1}"
            if panel is None:
                if len(units) != 1:
                    raise ValueError("S8_display_ambiguous_standalone_input")
                region = [0, 0, width, height]
                printed = None
            else:
                if type(panel) is not int or panel != index:
                    raise ValueError("S8_display_panel_order_mismatch")
                label += f"/panel_{panel + 1}"
                geometry = unit["geometry"]
                x, y = geometry["panel_origin_xy"]
                left, top, right, bottom = geometry["image_region_xyxy"]
                region = [left + x, top + y, right + x, bottom + y]
                printed = geometry["label"]
            if unit["unit_id"] != label or unit["artifact_id"] != item["artifact_id"]:
                raise ValueError("S8_display_unit_identity_mismatch")
            left, top, right, bottom = region
            if not (0 <= left < right <= width and 0 <= top < bottom <= height):
                raise ValueError("S8_display_region_out_of_bounds")
            for previous in records:
                l, t, r, b = previous["image_region_xyxy"]
                if max(left, l) < min(right, r) and max(top, t) < min(bottom, b):
                    raise ValueError("S8_display_regions_overlap")
            records.append({
                "unit_id": unit["unit_id"], "label": label, "panel_index": panel,
                "printed_panel_label": printed, "frame_index": unit["frame_index"],
                "image_region_xyxy": region,
            })
        # The S8 corpus and observation binder have already matched these exact fields
        # against the pinned recipe, visual manifest, source links and archive members.
        record = {field: item[field] for field in (
            "input_order", "input_id", "artifact_id", "relative_path", "storage",
            "sha256", "width", "height",
        )}
        ordered.append({**record, "units": records})
    if by_input or not ordered:
        raise ValueError("S8_display_unbound_input")
    return {
        "source": "phase2_sector8", "repository": pin["repository"],
        "resolved_commit": pin["resolved_commit"], "source_catalog_sha256": pin["source_catalog_sha256"],
        "example_id": recipe["example_id"], "ordered_inputs": ordered,
    }


def _positive_operands(task, units):
    """Enumerate one requested response per grounded operand combination."""
    ids = [u["unit_id"] for u in units]
    unique_cls = sorted({a["class_id"] for u in units for a in u["observations"]})
    single = {"finding_identification","finding_category","optional_spatial_localization"}
    if task in single:
        for u in units:
            classes = sorted({a["class_id"] for a in u["observations"]})
            if task == "finding_identification":
                if classes:
                    yield [u["unit_id"]], None, None, None
            else:
                for cls in classes:
                    yield [u["unit_id"]], cls, None, None
    elif task == "verified_reference_comparison":
        if any(u["unit_role"]=="verified_normal_reference" for u in units):
            yield ids, None, None, None
    elif task == "evidence_frame_localization":
        for cls in unique_cls:
            yield ids, cls, None, None
    elif task == "observed_anchor_order":
        for cls in unique_cls:
            hits = [u for u in units if any(a["class_id"]==cls for a in u["observations"])]
            for a,b in itertools.combinations(hits, 2):
                if a["video_id"] == b["video_id"] and a["frame_index"] != b["frame_index"]:
                    yield [a["unit_id"],b["unit_id"]], cls, None, None
    elif task == "temporal_relative_evidence_localization":
        reps = {}
        for u in units:
            reps.setdefault((u["video_id"],u["frame_index"],u["source_sha256"]),u)
        if len({u["video_id"] for u in units}) != 1:
            return
        for cls in unique_cls:
            for ref in reps.values():
                for direction in ("before","after"):
                    yield ids, cls, ref, direction


def _verified_recipe_stream(source, s8_dir, selected_tasks, *, corpus=None):
    """Prepare the S1 binding resolver; delay each recipe's clinical evaluation until requested."""
    if corpus is None:
        raise ValueError("The recipe stream requires the clinical corpus from the Section 1 import.")
    binder = load_observation_binding(source, corpus, code=source["_real_runtime_bytes"]["s1_binder"])
    pin = source["artifacts"]["phase2_source_snapshot"]
    prepare_binding = _s8_binding_preparer(binder)
    by_variant = defaultdict(list)
    for task, rows in source["selected_cells"].items():
        if selected_tasks is not None and task not in selected_tasks:
            continue
        for row in rows:
            if row["task"] != task or row["original_policy_code"] != "A" or row["expected_status"] != "applicable":
                raise ValueError("unexpected_S1_matrix_selector")
            semantics = source["contracts"][task]["semantics"]
            role_spec = semantics["task_semantics"]
            slots = set(role_spec["clinical_slots"]) if "clinical_slots" in role_spec else {role_spec["clinical_slot"]}
            binding_rows = [binder.bindings[(b,)] for b in json.loads(row["terminology_binding_ids_json"])]
            by_class = defaultdict(list)
            for binding in binding_rows:
                if (binding["clinical_slot"] in slots and binding["mapping_status"] == "approved"
                        and binding["terminology"] in {"MST", "CEST", "SNOMED_CT"}):
                    by_class[binding["dataset_class_id"]].append(binding)
            by_variant[row["variant"]].append((task, row, by_class))
    groups = {r["dataset_class_id"]: r["dataset_categories"] if r["dataset_category_status"]=="consistent"
              and not r.get("annotations_without_category") else []
              for r in source["artifacts"]["phase2_classes"]}
    lexical_cache = {}

    def evaluate_recipe(recipe, stats, on_display_hold):
        stats["recipes_examined"] += 1
        def report_display_hold(task, reason):
            if on_display_hold is not None:
                on_display_hold({"recipe": recipe["example_id"], "variant": recipe["variant"],
                                 "disposition": "review", "stage": "before_generation",
                                 "reason": reason, "task": task})
        rows = by_variant.get(recipe["variant"], ())
        if not rows:
            stats["recipes_without_selected_A_cell"] += 1
            return
        try:
            recipe, native, evaluate_binding = prepare_binding(recipe)
        except ValueError:
            stats["review_geometry_or_binding"] += 1
            report_display_hold(None, "review_geometry_or_binding")
            return
        if any(u["operation"]=="overlay" for u in native):
            stats["review_assisted_overlay"] += 1
            report_display_hold(None, "review_assisted_overlay")
            return
        # Expanded crops with unrelated source labels do not establish unambiguous visibility.
        if any(u["operation"]=="crop" and len({binder.observations[(a,)]["dataset_class_id"]
                for a in u["source_annotation_ids"]}) != 1 for u in native):
            stats["review_competing_crop_labels"] += 1
            report_display_hold(None, "review_competing_crop_labels")
            return
        # Count each display hold once per recipe/task, before generating any question.
        display_holds = {task: _s8_display_review_reason(corpus.visual_by_artifact, native, task)
                         for task in dict.fromkeys(task for task, _, _ in rows)}
        for task, reason in display_holds.items():
            if reason:
                stats[reason] += 1
                report_display_hold(task, reason)
        rows = [(task, row, by_class) for task, row, by_class in rows if not display_holds[task]]
        if not rows:
            return
        reference_units, reason = binder._normal_reference(recipe,native)
        verified_refs = {u["unit_id"] for u in reference_units} if not reason else set()
        visual = None
        units = []
        for u in native:
            observations = []
            for aid in u["annotation_ids"]:
                obs = binder.observations[(aid,)]
                truth = corpus.truths_by_annotation[aid]
                observations.append({"id":aid, "class_id":obs["dataset_class_id"], "visible":True,
                                     "clinical_group":truth.get("clinical_group"), "box":truth.get("bbox_xyxy"),
                                     "box_source_xyxy":truth.get("bbox_xyxy")})
            kind = "crop" if u["operation"]=="crop" else ("panel" if u["panel_index"] is not None else "original")
            geo = u["geometry"].get("source_geometry",u["geometry"])
            bounds = list(geo["crop_xyxy"]) if kind == "crop" else None
            units.append({"unit_id":u["unit_id"], "label":u["unit_id"], "video_id":u["video_key"],
                          "frame_index":u["frame_index"], "source_sha256":u["source_sha256"],"view":kind,
                          "width":u["display_width"],"height":u["display_height"],"visible_chronology":u["frame_index"],
                          "source_width":u["source_width"],"source_height":u["source_height"],"crop_bounds_source_xyxy":bounds,
                          "unit_role":"verified_normal_reference" if u["unit_id"] in verified_refs else (
                              "target" if u["link_role"]=="target" else "observation"), "observations":observations})
        # A verified binding request may recur across profiles in this recipe.
        @lru_cache(None)
        def bound(binding_id, task, state, annotation_ids, reference_id, direction):
            return evaluate_binding(binding_id=binding_id, task=task, evidence_state=state,
                                    annotation_ids=list(annotation_ids), reference_unit_id=reference_id,
                                    direction=direction)

        for task, row, by_class in rows:
            profiles = source["contracts"][task]["semantics"]["profiles"]
            for scope, cls, ref, direction in _positive_operands(task, units):
                for profile in profiles:
                    operands = defaultdict(set)
                    for u in units:
                        if u["unit_id"] not in scope:
                            continue
                        if ref and not (u["frame_index"]<ref["frame_index"] if direction=="before" else u["frame_index"]>ref["frame_index"]):
                            continue
                        if task=="verified_reference_comparison" and u["unit_id"] not in verified_refs:
                            continue
                        for a in u["observations"]:
                            if cls and a["class_id"] != cls:
                                continue
                            category = profile.get("answer_category_filter",{}).get("equals")
                            if category and a["clinical_group"] != category:
                                continue
                            if task=="optional_spatial_localization" and not a["box"]:
                                continue
                            operands[a["class_id"]].add(a["id"])
                    if not operands:
                        continue
                    approved = defaultdict(set)
                    receipts = []
                    for class_id, aids in operands.items():
                        for b in by_class.get(class_id, ()):
                            req = dict(binding_id=b["binding_id"],task=task,evidence_state=row["evidence_state"],
                                       annotation_ids=sorted(aids),reference_unit_id=ref["unit_id"] if ref else None,direction=direction)
                            v = bound(req["binding_id"], task, req["evidence_state"], tuple(req["annotation_ids"]),
                                      req["reference_unit_id"], direction)
                            if v["eligible_for_task_input"] is True:
                                approved[class_id].add(v["mapping_id"])
                                receipts.append({**req, "mapping_id":v["mapping_id"],
                                                 "observation_evidence":v["observation_evidence"]})
                    if set(operands) != set(approved):
                        stats["ineligible_observation_or_terminology"] += 1
                        continue
                    lexicon = {}
                    for k, mids in approved.items():
                        key = (k,tuple(sorted(mids)))
                        if key not in lexical_cache:
                            lexical_cache[key] = resolve_class_lexicon(source, k, mids)
                        lexicon[k] = lexical_cache[key]
                    if visual is None:
                        visual = _s8_presentation_reference(pin, recipe, native)
                    yield task, {"profile_id":profile["profile_id"],"recipe_id":recipe["example_id"],
                                 "variant":recipe["variant"],"evidence_state":row["evidence_state"],
                                 "scope_unit_ids":scope,"units":units,"class_lexicon":lexicon,"groups_by_class":groups,
                                 "query_class_id":cls,"reference_unit_id":ref["unit_id"] if ref else None,
                                 "direction":direction,"reference_criterion":"verified normal reference",
                                 "normal_reference_class_ids":sorted({a["class_id"] for u in units if u["unit_id"] in verified_refs
                                                                       for a in u["observations"]}),
                                 "s1_bindings":receipts,"respondent_presentation":visual}

    def recipes():
        for recipe in corpus.iter_recipes():
            def evaluate(stats, on_display_hold, recipe=recipe):
                return evaluate_recipe(recipe, stats, on_display_hold)
            yield recipe, evaluate

    return corpus.validation_report["recipes"], recipes()


def iter_verified_cases(imported, *, max_recipes=None, stats=None, selected_tasks=None, on_display_hold=None):
    """Legacy streaming adapter for bounded previews; the full census uses recipe-level checkpoints."""
    if max_recipes is not None and (type(max_recipes) is not int or max_recipes < 1):
        raise ValueError("max_recipes_must_be_positive_or_None")
    stats = Counter() if stats is None else stats
    if imported["s8_context"] is None:
        raise ValueError("Recipe previews require a real S8 import in Section 1")
    corpus = _clinical_s8_corpus(*imported["s8_context"])
    _, stream = _verified_recipe_stream(imported["source"], imported["s8_dir"], selected_tasks, corpus=corpus)
    for recipe, evaluate in itertools.islice(stream, max_recipes):
        yield from evaluate(stats, on_display_hold)


def respondent_s8_reference(presentation):
    """Expose stable S8 references and unit metadata while keeping class-bearing paths private."""
    result = {key: presentation[key] for key in (
        "source", "repository", "resolved_commit", "source_catalog_sha256", "example_id"
    )}
    result["ordered_inputs"] = [
        {key: item[key] for key in ("input_order", "input_id", "artifact_id", "sha256", "width", "height", "units")}
        for item in presentation["ordered_inputs"]
    ]
    return result


#### Run the import

Run this cell once per VM session or whenever you want to recheck the local metadata and contract files. It prints the import results before any clinical census or API request. Subsequent batches use the returned snapshot. A new VM or a new release requires a fresh import. The existing cache is repaired selectively. The V13 import can start directly from the pinned directory listing. Missing contract declarations, malformed file identities or conflicting published digests stop admission of that release.


In [6]:
imported_inputs = None
work = Path(LOCAL_WORK).expanduser().resolve()
if MODE not in engine["SECTOR2_MODES"]:
    raise ValueError(f"Unknown execution mode: {MODE}")

output_dir = work / "outputs/phase3/sector2"
if MOUNT_DRIVE and Path("/content").is_dir():
    output_dir = Path("/content/drive/MyDrive/MMVQA_Clinical/outputs/phase3/sector2")
if OUTPUT_OVERRIDE:
    output_dir = Path(OUTPUT_OVERRIDE).expanduser()
if MOUNT_DRIVE and any(path.resolve().is_relative_to("/content/drive") for path in (output_dir, work)):
    if drive is None:
        raise RuntimeError("Automatic Drive mounting requires Colab. Configure local paths and disable MOUNT_DRIVE.")
    drive.mount("/content/drive")

import_config = {
    "mode": MODE, "output_dir": str(output_dir), "cache_dir": str(work / "cache"),
    "s8_dir": str(work / "s8"), "refresh_github_main": REFRESH_GITHUB_MAIN,
}
imported_inputs = engine["import_sector2_inputs"](import_config, import_components={k: component_hashes[k] for k in ("sources", "real")})
registry = imported_inputs["registry"]
print("Verified V13 contracts:", len(registry["contract_sha256"]), "| Revision:", registry["registry_commit"])
print("V13 integrity source:", registry["manifest_origin"], "| Receipt SHA-256:", registry["manifest_sha256"])
print("Verified S1 run:", imported_inputs["source"]["run_id"])
report = imported_inputs["s8_vm_sync"]["visuals"]
print(f"S8 visual references bound in metadata: {report['referenced_files']:,} "
      f"({report['original_frames']:,} source frames, {report['derived_members']:,} derived members).")
print("Visual files opened:", report["visual_files_opened"])


Mounted at /content/drive


V13 contracts · up to 8 workers · batch 1 · transferred or reused:   0%|          | 0/14 [00:00<?, ?file/s]

V13 contracts · up to 8 workers · batch 1 · verified VM files:   0%|          | 0/14 [00:00<?, ?file/s]

Git · latest.json · batch 1 · transferred or reused:   0%|          | 0/1 [00:00<?, ?file/s]

Git · latest.json · batch 1 · verified VM files:   0%|          | 0/1 [00:00<?, ?file/s]

S1 manifests · batch 1 · transferred or reused:   0%|          | 0/3 [00:00<?, ?file/s]

... (20 outputs left out)

S8 parts · visual_manifest.parquet · batch 1 · verified VM files:   0%|          | 0/7 [00:00<?, ?file/s]

S8 parts · visual_source_manifest.parquet · batch 1 · transferred or reused:   0%|          | 0/2 [00:00<?, ?f…

S8 parts · visual_source_manifest.parquet · batch 1 · verified VM files:   0%|          | 0/2 [00:00<?, ?file/…

Verified V13 contracts: 14 | Revision: 46930c6ffa6d17ff14b76171b9bd4eb6d70f85f0
V13 integrity source: github_git_blobs | Receipt SHA-256: 6befcf2d5c6ff8f317cc2cb4e690363eb9f49a104e95c426ba83499df6aba068
Verified S1 run: 20261118T013916_145151Z
S8 visual references bound in metadata: 184,328 (66,127 source frames, 118,201 derived members).
Visual files opened: 0


### 2 · Semantic plan and private answer

Each of the seven tasks derives its reference in `build_case`. The reference includes all supported alternatives. The identification profiles ask for a `finding` or an `anatomical landmark`. Category questions use the original dataset group. Spatial localization uses native pixel coordinates. Temporal ordering requires distinct frames and chronology visible to the respondent.


In [7]:
%%sector2_component evidence
# Answer derivation from S1 observations and published S8 visual evidence


class EvidenceHold(ValueError):
    """An ordinary ineligible or unresolved instance; no generator request is made."""
    def __init__(self, reason, disposition="review"):
        super().__init__(reason)
        self.reason = reason
        self.disposition = disposition


def _need(condition, reason, disposition="review"):
    if not condition:
        raise EvidenceHold(reason, disposition)


def _evidence_hash(value):
    raw = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def _evidence_key(value):
    return " ".join(unicodedata.normalize("NFKC", str(value)).casefold().replace("_", " ").split())


def _has_answer(text, term):
    def words(value):
        return re.findall(r"[^\W_]+", _evidence_key(value))
    needle, phrase = words(term), words(text)
    return bool(needle) and any(phrase[i:i+len(needle)] == needle for i in range(len(phrase)))


def _read_binding(roots, binding):
    node = roots[binding["root"]]
    for step in binding["path"]:
        node = node[step]
    return node


def _unique_text(values):
    return list(dict.fromkeys(values))


def _resolved_query_surfaces(lexicon, cls):
    _need(cls in lexicon, "query_class_without_source_binding")
    licensed = lexicon[cls]["public_surfaces"]
    others = {
        _evidence_key(x) for key, record in lexicon.items() if key != cls
        for x in [record["canonical"], *record["public_surfaces"], *record.get("aliases", [])]
    }
    values = [x for x in licensed if _evidence_key(x) not in others]
    _need(values, "missing_or_ambiguous_query_finding")
    return _unique_text(values)


def _visible_order(units):
    _need(len({u["video_id"] for u in units}) == 1, "chronology_requires_one_video")
    pairs = sorted(set((u["frame_index"], u["visible_chronology"]) for u in units),
                   key=lambda p: p[0])
    _need(all(type(v) is int for _, v in pairs), "chronology_not_available_to_respondent")
    _need(all(a[0] < b[0] and a[1] < b[1] for a, b in zip(pairs, pairs[1:])),
          "display_chronology_disagrees_with_source")


def _private_unit_refs(units):
    return _unique_text(x for u in units for x in (u["unit_id"], u["label"]))


def _is_valid_box(box, unit):
    return (isinstance(box, (tuple, list)) and len(box) == 4 and
            all(type(x) in (int, float) and math.isfinite(x) for x in box) and
            0 <= box[0] < box[2] <= unit["width"] and 0 <= box[1] < box[3] <= unit["height"])


def build_case(task, evidence, contracts):
    """Freeze a single A-cell instance after admission by the S1/visual-source loader.

    Evidence observations only count on the displayed unit to which the verified
    loader linked them. Use its complete corpus group index and terminology resolver.
    """
    sem, gram = contracts[task]["semantics"], contracts[task]["grammar"]
    profile = next((p for p in sem["profiles"] if p["profile_id"] == evidence["profile_id"]), None)
    _need(profile is not None, "unknown_profile")
    _need(profile["terminology_representation"] == {
        "type": "existing_concept_references", "allow_postcoordination": False
    }, "unsupported_terminology_representation")
    pc = profile["plan_constants"]
    decl = sem["task_semantics"]
    operation = profile.get("public_operation") or decl["public_operation"]
    mapping = decl.get("internal_to_public_operation") or decl["operation_projection"]["internal_operation_to_public_operation"]
    _need(mapping[pc["operation"]] == operation, "operation_mapping_mismatch")

    all_units = evidence["units"]
    by_id = {u["unit_id"]: u for u in all_units}
    _need(len(by_id) == len(all_units) and all_units, "duplicate_or_missing_unit_ids")
    _need(len({_evidence_key(u["label"]) for u in all_units}) == len(all_units), "ambiguous_display_label")
    _need(all(re.fullmatch(r"(?:panel [A-Za-z0-9]+|(?:image|frame)[ _][1-9][0-9]*(?:/panel_[1-9][0-9]*)?|this (?:image|crop))", u["label"], re.I)
              for u in all_units), "unverified_display_label")
    scope_ids = evidence["scope_unit_ids"]
    _need(scope_ids and len(set(scope_ids)) == len(scope_ids) and set(scope_ids) <= by_id.keys(), "unresolved_scope")
    units = [by_id[i] for i in scope_ids]
    for u in all_units:
        _need(isinstance(u["video_id"], str) and u["video_id"] and type(u["frame_index"]) is int and u["frame_index"] >= 0
              and type(u["width"]) is int and u["width"] > 0 and type(u["height"]) is int and u["height"] > 0
              and re.fullmatch(r"[0-9a-f]{64}", u["source_sha256"]), "missing_physical_evidence_identity")
        _need(u["view"] in {"original", "panel", "crop", "marked_region"}, "unsupported_view")
    singles = {"finding_identification", "finding_category", "optional_spatial_localization"}
    if task in singles:
        _need(len(units) == 1, "single_specific_scope_required")
    else:
        _need(len({(u["video_id"], u["frame_index"]) for u in units}) >= 2,
              "insufficient_distinct_displayed_frames", "not_applicable")
    unit = units[0]
    lex = evidence["class_lexicon"]
    scoped = [(u, a) for u in units for a in u["observations"] if a["visible"] is True]
    _need(all(a.get("id") and a.get("class_id") for _, a in scoped), "missing_observation_identity")
    if task in singles and unit["label"].casefold() in {"this image", "this crop"}:
        _need(len(all_units) == 1, "ambiguous_deictic_scope")
        _need(unit["label"].casefold() != "this crop" or unit["view"] == "crop", "scope_view_mismatch")
    for u in units:
        if u["view"] in {"crop", "marked_region"}:
            bounds = u.get("crop_bounds_source_xyxy")
            _need(type(u.get("source_width")) is int and type(u.get("source_height")) is int
                  and _is_valid_box(bounds, {"width": u["source_width"], "height": u["source_height"]}),
                  "transformed_view_without_verified_source_extent")
            if u["view"] == "marked_region":
                _need(u.get("mark_visible_verified") is True, "region_mark_not_verified")
            for a in u["observations"]:
                if a["visible"] is not True:
                    continue
                box = a.get("box_source_xyxy")
                _need(_is_valid_box(box, {"width": u["source_width"], "height": u["source_height"]})
                      and bounds[0] <= box[0] and bounds[1] <= box[1]
                      and box[2] <= bounds[2] and box[3] <= bounds[3],
                      "positive_evidence_not_demonstrably_inside_selected_view")
    cls = evidence.get("query_class_id")
    matching = [(u, a) for u, a in scoped if cls and a["class_id"] == cls]
    query_tasks = {"finding_category", "evidence_frame_localization", "optional_spatial_localization",
                   "observed_anchor_order", "temporal_relative_evidence_localization"}
    slots, params, roles, terms, aliases = {}, {}, {}, [], {}
    uncertain_terms = []
    if task in query_tasks:
        name = "FINDING_TERM" if task in {"finding_category", "optional_spatial_localization"} else "FINDING"
        slots[name] = _resolved_query_surfaces(lex, cls)

    support = []
    alternative_units = []
    scope_id = "scope_" + str(list(by_id).index(unit["unit_id"]) + 1)
    if task == "finding_identification":
        params = {"scope_id": scope_id}
        support = scoped
        if "answer_category_filter" in profile:
            category = profile["answer_category_filter"]["equals"]
            support = [(u, a) for u, a in support if _evidence_key(a["clinical_group"]) == _evidence_key(category)]
        ids = sorted({a["class_id"] for u, a in support})
        _need(ids, "no_positive_in_scope", "not_applicable")
        _need(all(k in lex for k in ids), "unresolved_private_class")
        _need(all(lex[k].get("canonical") for k in ids), "missing_canonical_source_surface")
        alternatives = [lex[k]["canonical"] for k in ids]
        aliases = {k: _unique_text([lex[k]["canonical"], *lex[k].get("answer_aliases", [])]) for k in ids}
        terms = _unique_text(x for k in ids for x in aliases[k])
        # Context-dependent aliases remain in the disclosure screen even when
        # they cannot be used to grade an answer automatically.
        uncertain_terms = _unique_text(x for k in ids for x in [*lex[k].get("aliases", []),
                                                                 *lex[k].get("potential_aliases", [])] if x not in terms)
        slots["KIND"] = [profile["public_kind"]["base_surface"]]
        slots["SCOPE"] = [unit["label"]]
        roles = {"KIND": profile["public_kind"]["kind_id"], "SCOPE": scope_id}
    elif task == "finding_category":
        params = {"query_dataset_class_id": cls, "scope_id": scope_id}
        support = matching
        _need(support, "named_finding_not_visible", "not_applicable")
        groups = evidence["groups_by_class"].get(cls, [])
        _need(groups and all(isinstance(v, str) and v.strip() for v in groups), "incomplete_source_category")
        canonical_group = groups[0]
        _need(all(_evidence_key(v) == _evidence_key(canonical_group) for v in groups)
              and all(a.get("clinical_group") and _evidence_key(a["clinical_group"]) == _evidence_key(canonical_group)
                      for _, a in matching), "inconsistent_source_category")
        alternatives = [canonical_group]
        terms = _unique_text([canonical_group, canonical_group.replace("_", " ")])
        slots["FINDING_TERM"] = [s for s in slots["FINDING_TERM"] if not any(_has_answer(s, t) for t in terms)]
        _need(slots["FINDING_TERM"], "category_disclosed_by_all_finding_surfaces")
        slots["SCOPE"] = [unit["label"]]
    elif task == "optional_spatial_localization":
        _need(len(all_units) == 1 and unit["view"] == "original", "spatial_requires_single_unassisted_native_image")
        params = {"finding_class_id": cls, "image_ref": unit["unit_id"]}
        boxes = [(u, a) for u, a in matching if a.get("box") is not None]
        _need(boxes, "missing_same_class_geometry", "not_applicable")
        _need(all(_is_valid_box(a["box"], u) for u, a in boxes), "invalid_source_box")
        support = boxes
        alternatives = [list(b) for b in sorted({tuple(a["box"]) for _, a in boxes})]
        slots.update(SCOPE=[unit["label"]], BOX_FORMAT=["[x1, y1, x2, y2] in original image pixels"])
    elif task == "evidence_frame_localization":
        params = {"presentation_id": evidence["recipe_id"], "query_class_id": cls}
        support = matching
        positive_ids = {u["unit_id"] for u, _ in support}
        alternative_units = [u for u in units if u["unit_id"] in positive_ids]
        slots["DISPLAY_SET"] = ["the displayed panels" if all("panel" in u["label"].lower() for u in units) else "the displayed frames"]
    elif task == "verified_reference_comparison":
        params = {"comparison_scope_id": evidence["recipe_id"]}
        refs = [u for u in units if u["unit_role"] == "verified_normal_reference"]
        targets = [u for u in units if u["unit_role"] == "target"]
        _need(refs and targets and len({u["video_id"] for u in refs + targets}) == 1, "missing_explicit_target_reference_pair")
        _need(len({u["frame_index"] for u in refs}) == 1 and
              all(t["frame_index"] != refs[0]["frame_index"] for t in targets), "invalid_reference_frame")
        normal_ids = set(evidence.get("normal_reference_class_ids", []))
        _need(normal_ids, "normal_reference_labels_unverified")
        ref_ids = {u["unit_id"] for u in refs}
        ref_rows = [(u,a) for u,a in scoped if u["unit_id"] in ref_ids and a["class_id"] in normal_ids]
        _need(all(normal_ids == {a["class_id"] for u, a in scoped if u["unit_id"] == r["unit_id"]} for r in refs),
              "incomplete_explicit_normal_reference_evidence")
        target_rows = [(u, a) for u, a in scoped if u in targets]
        _need(all(any(u is t for u, a in target_rows) for t in targets), "target_evidence_missing")
        _need(not any(a["class_id"] in normal_ids for u, a in target_rows), "target_shares_verified_reference_labels")
        support = ref_rows + target_rows
        alternative_units = sorted(refs, key=lambda u: u["unit_id"])
        criterion = evidence["reference_criterion"]
        _need(criterion in {"normal reference", "verified normal reference", "verified normal reference mucosa"},
              "unlicensed_reference_criterion")
        slots.update(COMPARISON_SCOPE=["this comparison"], REFERENCE_CRITERION=[criterion])
    elif task == "observed_anchor_order":
        _need(len(units) == 2, "two_anchor_operands_required")
        _visible_order(units)
        _need(len({u["frame_index"] for u in units}) == 2, "tied_frame_indices", "not_applicable")
        _need({u["unit_id"] for u,a in matching} == set(scope_ids), "both_anchors_require_positive_support", "not_applicable")
        support = matching
        first, second = units
        params = {"finding_class_id": cls, "option_a_unit_id": first["unit_id"], "option_b_unit_id": second["unit_id"]}
        slots.update(OPTION_A=[first["label"]], OPTION_B=[second["label"]])
        alternative_units = [min(units, key=lambda u: u["frame_index"])]
    elif task == "temporal_relative_evidence_localization":
        _visible_order(units)
        ref, direction = evidence["reference_unit_id"], evidence["direction"]
        _need(ref in scope_ids and direction in {"before", "after"}, "invalid_temporal_operand")
        params = {"finding_class_id": cls, "reference_unit_id": ref, "direction": direction, "scope_id": evidence["recipe_id"]}
        index = by_id[ref]["frame_index"]
        support = [(u,a) for u,a in matching if (u["frame_index"] < index if direction == "before" else u["frame_index"] > index)]
        positive_ids = {u["unit_id"] for u, _ in support}
        alternative_units = sorted((u for u in units if u["unit_id"] in positive_ids), key=lambda u: u["unit_id"])
        slots.update(REFERENCE=[by_id[ref]["label"]], DIRECTION=[direction], SCOPE=["this sequence"])
    else:
        raise EvidenceHold("task_adapter_unavailable")

    if task not in singles:
        _need(alternative_units, "no_positively_supported_answer", "not_applicable")
        alternatives = [u["unit_id"] for u in alternative_units]
        aliases = {u["unit_id"]: [u["label"]] for u in alternative_units}
        if task in {"evidence_frame_localization", "verified_reference_comparison"}:
            terms = _private_unit_refs(units)  # Both winners and nonwinners reveal a choice by suggestion/exclusion.
        elif task == "temporal_relative_evidence_localization":
            terms = _private_unit_refs([u for u in units if u["unit_id"] != params["reference_unit_id"]])
        # Both neutral option labels are mandatory for observed_anchor_order.
        # The text validator separately checks balanced roles and no ordering assertion.
    roots = {"semantic_profile": profile, "semantics_document": sem, "grammar_document": gram,
             "selected_instance_parameters": params, "selected_profile_constants": pc}
    Draft202012Validator(profile["parameters_schema"]).validate(params)
    if not roles:
        roles = {name: _read_binding(roots,b) for name,b in profile["public_operand_bindings"].items()}
    _need(set(slots) == set(gram["slot_bindings"]) == set(roles), "unresolved_public_role")
    roots["public_plan"] = {"profile_id": profile["profile_id"], "parameters": params}
    roots["public_context"] = {"role_surfaces": slots, "scope_surfaces_by_id": {scope_id: slots.get("SCOPE", [])},
                               "kind_surfaces_by_profile": {profile["profile_id"]: slots.get("KIND", [])}}
    public = {}
    for field, binding in gram["public_request"]["field_bindings"].items():
        if "resolved_slot" not in binding:
            public[field] = _read_binding(roots, binding)
            continue
        slot = _read_binding(roots, binding["resolved_slot"])
        public[field] = (_read_binding(roots, slot["source"]) if "source" in slot else
                         _read_binding(roots, slot["index"])[_read_binding(roots, slot["key"])])
    Draft202012Validator(gram["public_request"]["data_schema"]).validate(public)
    instructions = _read_binding(roots, gram["public_request"]["instructions"])
    _need(isinstance(instructions, str), "unresolved_public_instructions")
    screened_text = [instructions, *[text for values in slots.values() for text in values]]
    _need(not any(_has_answer(text, term) for text in screened_text for term in [*terms, *uncertain_terms]),
          "private_information_in_public_projection")
    refs = sorted({a["id"] for u,a in support})
    snapshot = _evidence_hash(evidence)
    plan = {**pc, "task_id": task, "profile_id": profile["profile_id"], "parameters": params,
            "public_operation": operation, "roles": roles, "public_kind": roles.get("KIND"),
            "asserted_attributes": pc.get("asserted_attributes", []),
            "asserted_relations": pc.get("asserted_relations", []), "temporal_constraints": pc.get("temporal_constraints", [])}
    visual = [{k:u[k] for k in ("unit_id","label","width","height","view","visible_chronology")} for u in all_units]
    physical = [(u["unit_id"],u["video_id"],u["frame_index"],u["source_sha256"]) for u in all_units]
    identity = {"recipe": evidence["recipe_id"], "scope": scope_ids, "physical": physical,
                "plan": plan, "answer_set": alternatives, "support": refs}
    return {"task_id":task, "profile_id":profile["profile_id"], "frozen_plan":plan,
            "role_surfaces":slots,"public_request":public, "public_presentation":{"displayed_units":visual},
            "private_answer":{"answer_type":pc["answer_type"], "canonical":alternatives[0],
                              "alternatives":alternatives, "aliases":aliases},
            "disclosure_terms":terms, "ambiguous_disclosure_terms":uncertain_terms, "evidence_refs":refs, "evidence_snapshot_id":snapshot,
            "semantic_instance_id":_evidence_hash(identity)}


### 3 · OpenAI question generation

The model can phrase each question freely in English. Structured output restricts the response to a question field. The client submits the exact public request bound to its receipt, reads the key at runtime and counts each HTTP call against the budget, including refusals.

[OpenAI Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs) · [Model catalog](https://developers.openai.com/api/docs/models)


In [8]:
%%sector2_component api
API_MODEL = "gpt-6-luna"
_RESPONSES_ENDPOINT = "https://api.openai.com/v1/responses"


def _closed_object(properties):
    return {"type": "object", "properties": properties, "required": list(properties),
            "additionalProperties": False}


def _responses_payload(model, prompt, public_data, output_name, output_schema, token_limit):
    return {
        "model": model, "store": False, "instructions": prompt,
        "input": _wire_bytes(public_data).decode("utf-8"),
        "max_output_tokens": token_limit, "reasoning": {"effort": "low"},
        "text": {"format": {"type": "json_schema", "name": output_name,
                            "strict": True, "schema": output_schema}}
    }


def openai_payload(grammar, public_data, model=API_MODEL):
    """Freeze only the verified public projection. The HTTP schema constrains shape only."""
    public_contract = grammar["public_request"]
    Draft202012Validator(public_contract["data_schema"]).validate(public_data)
    def bound_value(binding):
        if binding["root"] != "grammar_document":
            raise ValueError("unresolved public request binding")
        value = grammar
        for field in binding["path"]:
            value = value[field]
        return value
    declared = bound_value(public_contract["output_schema"])
    if (declared["required"] != ["question"] or declared["additionalProperties"] is not False
            or set(declared["properties"]) != {"question"}
            or declared["properties"]["question"]["type"] != "string"):
        raise ValueError("unsupported generator contract")
    # The authoritative V13 schema also contains Unicode lexical constraints. Apply them
    # locally to the original decoded text. Do not convert them to a sentence template.
    return _responses_payload(model, bound_value(public_contract["instructions"]), public_data,
                              "clinical_question", _closed_object({"question": {"type": "string"}}), 1000)


class SamplingProviderError(RuntimeError):
    def __init__(self, code, disposition="fatal_abort", metadata=None):
        super().__init__(code)
        self.code = code
        self.disposition = disposition
        self.metadata = dict(metadata or {})


def _colab_key():
    # Do not print records, exception bodies, response headers or token values.
    key = os.environ.get("OPENAI_API_KEY")
    if not key and userdata is not None:
        try:
            key = userdata.get("OPENAI_API_KEY")
        except Exception:
            key = None
    if not isinstance(key, str) or not key.strip():
        raise SamplingProviderError("openai_key_missing_or_inaccessible")
    return key.strip()


def _load_unique_json(value):
    def unique(pairs):
        result = {}
        for key, item in pairs:
            if key in result:
                raise ValueError("duplicate JSON key")
            result[key] = item
        return result
    return json.loads(value, object_pairs_hook=unique)


class _NoAPIRedirect(HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None


class OpenAIQuestionClient:
    """One counted HTTPS submission per call. Sampling/retry decisions belong to the caller."""
    def __init__(self, max_http_calls, key_getter=None, reviewer_model=API_MODEL):
        if type(max_http_calls) is not int or max_http_calls < 1:
            raise ValueError("max_http_calls must be a positive integer")
        self.max_http_calls = max_http_calls
        self.calls = 0
        self.key_getter = key_getter or _colab_key
        self.reviewer_model = reviewer_model
        self._opener = build_opener(_NoAPIRedirect())

    def _submit(self, payload, expected_sha256):
        body = _wire_bytes(payload)
        digest = hashlib.sha256(body).hexdigest()
        metadata = {"public_request_sha256": digest, "provider": "openai",
                    "requested_model": payload["model"]}
        if expected_sha256 != digest:
            raise SamplingProviderError("public_request_changed_since_preflight", metadata=metadata)
        if self.calls >= self.max_http_calls:
            raise SamplingProviderError("http_call_budget_exhausted", metadata=metadata)
        key = self.key_getter()
        req = Request(_RESPONSES_ENDPOINT, data=body, method="POST",
                                     headers={"Authorization": "Bearer " + key,
                                              "Content-Type": "application/json; charset=utf-8"})
        self.calls += 1
        metadata["http_call_number"] = self.calls
        try:
            with self._opener.open(req, timeout=90) as response:
                metadata["request_id"] = response.headers.get("x-request-id")
                raw = response.read(1_048_577)
        except HTTPError as exc:
            metadata.update({"http_status": exc.code, "request_id": exc.headers.get("x-request-id")})
            exc.close()
            raise SamplingProviderError(f"openai_http_{exc.code}", metadata=metadata) from None
        except OSError:
            # A lost response can follow a completed, billable submission. Do not replay it.
            raise SamplingProviderError("openai_response_unavailable", metadata=metadata) from None
        if len(raw) > 1_048_576:
            raise SamplingProviderError("openai_response_too_large", metadata=metadata)
        try:
            data = _load_unique_json(raw)
            if not isinstance(data, dict):
                raise ValueError
            metadata.update({"response_id": data.get("id"), "returned_model": data.get("model"),
                             "usage": data.get("usage")})
            if data.get("status") == "incomplete":
                raise SamplingProviderError("openai_incomplete", "review", metadata)
            if data.get("status") != "completed":
                raise SamplingProviderError("openai_response_not_completed", metadata=metadata)
            messages = [item for item in data["output"] if item.get("type") == "message"]
            if len(messages) != 1 or messages[0].get("role") != "assistant":
                raise ValueError
            parts = messages[0]["content"]
            if any(part.get("type") == "refusal" for part in parts):
                raise SamplingProviderError("openai_refusal", "review", metadata)
            if len(parts) != 1 or parts[0].get("type") != "output_text":
                raise ValueError
            result = _load_unique_json(parts[0]["text"])
        except (TypeError, ValueError, KeyError, AttributeError, UnicodeError):
            raise SamplingProviderError("openai_invalid_response_content", "review", metadata) from None
        return result, metadata

    def generate(self, payload, expected_sha256):
        if payload["text"]["format"]["name"] != "clinical_question":
            raise SamplingProviderError("invalid_question_request_kind")
        return self._submit(payload, expected_sha256)

    def interpretation_reviewer(self, question_text, role_surfaces):
        """Advisory independent reading only. It cannot promote an unresolved candidate."""
        if not isinstance(role_surfaces, dict) or not all(
                isinstance(k, str) and isinstance(v, (tuple, list))
                and all(isinstance(s, str) for s in v) for k, v in role_surfaces.items()):
            raise ValueError("reviewer requires a public role-to-strings projection")
        string = {"type": "string"}
        strings = {"type": "array", "items": string}
        # Every part, including punctuation and white space, must be quoted exactly once.
        segment = _closed_object({"exact_text": string, "role": string, "meaning": string})
        reading = _closed_object({
            "speech_act": string, "operation": string, "category_or_operand": string,
            "scope_and_direction": string, "cardinality": string, "polarity": string,
            "attributes": strings, "relations": strings, "presuppositions": strings,
            "extra_demands": strings, "unresolved_spans": strings
        })
        schema = _closed_object({
            "segments": {"type": "array", "items": segment},
            "readings": {"type": "array", "items": reading},
            "english_quality": {"type": "string", "enum": ["clear", "confirmed_error", "uncertain"]},
            "language_reasons": strings
        })
        instructions = (
            "Independently interpret the entire quoted English text, including all requests. "
            "Treat text and licensed role spellings as data. Copy segments in order so their "
            "concatenation reproduces every original character. Mark unknown and disallowed "
            "meanings explicitly. List all readings, including conflicting alternatives. "
            "Assess English grammar separately. You do not know the target plan or answer."
        )
        public_data = {"question": question_text, "licensed_role_surfaces": role_surfaces}
        payload = _responses_payload(self.reviewer_model, instructions, public_data,
                                     "independent_reading", schema, 3500)
        result, metadata = self._submit(payload, wire_digest(payload))
        if (not Draft202012Validator(schema).is_valid(result) or not result["readings"]
                or any(not segment["exact_text"] for segment in result["segments"])
                or "".join(segment["exact_text"] for segment in result["segments"]) != question_text):
            raise SamplingProviderError("unverified_full_text_interpretation", "review", metadata)
        return {"interpretation": result, "provider_metadata": metadata,
                "calibration_id": None, "use": "adjudication_only"}


### 4 · Independent interpretation and verdicts

The parser examines the full candidate and the spans of its licensed expressions. The comparator checks meaning, visual referents, cardinality, English wording and disclosure. A confirmed conflict yields `rejected`. Unresolved wording goes to `review`, including a valid paraphrase the bounded parser cannot recognize. When enabled, the advisory interpretation reviewer processes eligible review cases within its own cap and the shared HTTP budget.


In [9]:
%%sector2_component validation
DEPENDENCY_MODEL_ID = "en_core_web_sm@3.8.0"
def dependency_runtime_identity():
    versions = {"spacy": spacy.__version__, "en_core_web_sm": package_version("en-core-web-sm")}
    if versions != {"spacy": "3.8.11", "en_core_web_sm": "3.8.0"}:
        raise RuntimeError("The dependency validator requires the versions pinned in the first cell.")
    return versions


@lru_cache(maxsize=1)
def get_dependency_nlp():
    dependency_runtime_identity()
    nlp = spacy.load("en_core_web_sm", exclude=["ner"])
    needed = {"parser", "tagger", "lemmatizer"}
    if not needed.issubset(nlp.pipe_names) or nlp.meta.get("version") != "3.8.0":
        raise RuntimeError("The English model lacks the required parser, tagger, or lemmatizer.")
    return nlp


# A compositional recognizer for independently checking wording. These rules are
# used after generation. They are never supplied to the question generator.
_EN_RULES = {
    'Ask': ['NameV', 'please NameV', 'could you NameV', 'can you NameV',
            'could you please NameV', 'can you please NameV', 'could you give me the name of'],
    'NameV': ['name', 'identify'],
    'Single': ['one', 'any one', 'a', 'an'],
    'Which': ['what', 'which'],
    'Visible': ['visible', 'shown', 'depicted', 'seen', 'pictured'],
    'InScope': ['in @scope', 'within @scope'],
    'FrontScope': ['InScope ,', 'looking at @scope ,'],
    'IdentifyImperative': ['Ask Single @kind', 'Ask Single @kind Visible', 'Ask Single @kind you see'],
    'IdentifyWh': ['Which @kind can you identify', 'Which @kind do you see', 'Which @kind is Visible'],
    # Indirect questions use declarative word order inside the embedded clause.
    'TellMe': ['could you tell me', 'can you tell me', 'would you tell me'],
    'EmbeddedIdentifyWh': ['Which @kind you see', 'Which @kind you can identify', 'Which @kind is Visible'],
    'Identify': ['IdentifyImperative InScope', 'IdentifyWh InScope', 'FrontScope IdentifyWh',
                 'FrontScope Ask Single @kind', 'TellMe EmbeddedIdentifyWh InScope'],
    'Article': ['', 'the', 'a', 'an'],
    'ReferenceArticle': ['', 'the'],
    'Named': ['Article @finding_term'],
    'Group': ['which dataset clinical group', 'what dataset clinical group', 'which clinical group'],
    'Classify': ['FrontScope Group does the dataset assign to Named',
                 'which dataset clinical group contains Named Visible InScope',
                 'what is the dataset clinical group of Named InScope',
                 'which dataset clinical group does Named in @scope belong to',
                 'Which clinical group does the dataset assign to ReferenceArticle @finding_term InScope'],
    'Evidence': ['shows', 'depicts', 'provides visual evidence of', 'contains visual evidence of'],
    'Finding': ['Article @finding'],
    'Unit': ['panel', 'frame'],
    'Frame': ['which one of @display_set Evidence Finding', 'which of @display_set Evidence Finding',
              'among @display_set , Ask one that Evidence Finding',
              'among @display_set , which Unit Evidence Finding'],
    'BoxLead': ['could you give', 'can you provide', 'give', 'please provide', 'provide'],
    'Box': ['FrontScope BoxLead @box_format for a bounding box around one @finding_term',
            'FrontScope BoxLead a bounding box for one @finding_term as @box_format',
            'BoxLead a bounding box for one @finding_term InScope as @box_format'],
    'Compare': ['which Unit in @comparison_scope Evidence ReferenceArticle @reference_criterion',
                'in @comparison_scope , Ask one Unit that Evidence ReferenceArticle @reference_criterion',
                'in @comparison_scope , which Unit Evidence ReferenceArticle @reference_criterion'],
    'Options': ['@option_a and @option_b'],
    'Acquisition': ['was recorded earlier', 'was acquired earlier', 'was captured earlier'],
    'Order': ['between Options , which Unit showing Finding Acquisition',
              'of Options , which Unit showing Finding Acquisition',
              'which Unit showing Finding Acquisition , @option_a or @option_b'],
    'Relative': ['FrontScope Ask one Unit with evidence of Finding @direction @reference',
                 'which one of the displayed frames InScope Evidence Finding @direction @reference',
                 'which Unit InScope Evidence Finding @direction @reference'],
}
_OPERATION_GRAMMAR = {
    'Identify': 'identify', 'Classify': 'classify_named_finding',
    'Frame': 'localize_evidence_frame', 'Box': 'localize', 'Compare': 'select_reference_panel',
    'Order': 'select_earlier_observed_frame', 'Relative': 'localize_relative_evidence',
}
_PUBLIC_VISUAL_ROLES = {'scope', 'display_set', 'reference', 'comparison_scope', 'option_a', 'option_b'}
_TOKEN = re.compile(r"\w+(?:['’]\w+)*|[^\w\s]", re.UNICODE)
_CHECKER_VERSION = 'mmvqa-colab-en-comparator-1.4-spacy'


def _compare_tokens(text):
    return [(unicodedata.normalize('NFC', m.group()).casefold(), m.start(), m.end())
            for m in _TOKEN.finditer(text)]


def _role_spans(text, role_surfaces):
    source = _compare_tokens(text)
    words = [item[0] for item in source]
    candidates = {}
    for role, surfaces in role_surfaces.items():
        for surface in surfaces:
            part = [item[0] for item in _compare_tokens(surface)]
            if not part:
                continue
            for i in range(len(words) - len(part) + 1):
                if words[i:i+len(part)] == part:
                    candidates.setdefault(i, []).append((len(part), role))
    i, typed, spans, ambiguous = 0, [], [], []
    while i < len(words):
        hits = candidates.get(i, [])
        if not hits:
            typed.append(words[i])
            i += 1
            continue
        n = max(length for length, _ in hits)
        possible = sorted({role for length, role in hits if length == n})
        first, last = source[i][1], source[i+n-1][2]
        if len(possible) > 1:
            ambiguous.append({'affected_fields': possible, 'start': first, 'end': last,
                              'reason': 'multiple_licensed_roles'})
        role = possible[0] if len(possible) == 1 else 'ambiguous'
        typed.append('@' + role)
        spans.append({'role': role, 'surface': text[first:last], 'start': first, 'end': last})
        i += n
    return typed, spans, ambiguous


def _whole_request_operations(tokens):
    # Chart over role tokens and ordinary English constituents. Every complete
    # parse consumes the whole utterance, including any remaining punctuation.
    words = tokens[:-1] if tokens[-1:] in (['.'], ['?']) else tokens
    @lru_cache(None)
    def ends(symbol, i):
        if symbol not in _EN_RULES:
            return (i+1,) if i < len(words) and words[i] == symbol else ()
        possible = set()
        for rule in _EN_RULES[symbol]:
            positions = {i}
            for child in rule.split():
                positions = {j for pos in positions for j in ends(child, pos)}
            possible.update(positions)
        return tuple(possible)
    return sorted({_OPERATION_GRAMMAR[rule] for rule in _OPERATION_GRAMMAR if len(words) in ends(rule, 0)})


def _protected_match(text, term):
    # Separate disclosure normalization, deliberately more inclusive than role
    # licensing (punctuation, underscores and spaces do not shield an answer).
    def parts(s):
        s = unicodedata.normalize('NFKC', s).casefold()
        return re.findall(r'[^\W_]+', s, re.UNICODE)
    phrase, words = parts(term), parts(text)
    return bool(phrase) and any(words[i:i+len(phrase)] == phrase for i in range(len(words)))


_NEUTRAL_WORDS = set(('please kindly could can would you me tell give name identify see spot observe '    'one any a an the what which in visible depicted shown within of request referring to that '    'is do does based on looking at provide for seeing').split())


def _novel_wording_disclosure_clear(question, roles):
    # This deliberately small *outbound* allowance contains no private unit label,
    # coordinate, directional hint, class/category answer, position or clinical modifier.
    # Do not apply it when either abstract role may itself denote a winning option.
    if {'option_a', 'option_b'} & roles.keys():
        return False
    typed, hits, ambiguous = _role_spans(question, roles)
    if ambiguous or sorted(h['role'] for h in hits) != sorted(roles):
        return False
    return all(t.startswith('@') or t in _NEUTRAL_WORDS or t in {',', '?', '.'} for t in typed)


def _article_errors(question, spans, complete):
    # Only confirmed pronunciations of forms known to this reference checker.
    vowels = r'(?:anatomical|ulcer|erosion|angiectasia|ampulla|endoscopic|image|original)\b'
    consonants = r'(?:finding|polyp|foreign|frame|panel|landmark|bounding|dataset|verified|blood|red)\b'
    if re.search(r'\ba\s+' + vowels + r'|\ban\s+' + consonants, question, re.I):
        return ['confirmed_indefinite_article_error']
    if complete:
        for hit in spans:
            # A bare singular count noun in these operand positions lacks a determiner.
            if hit['role'] in {'finding', 'finding_term'} and hit['surface'].casefold() in {
                'polyp', 'ulcer', 'foreign body', 'red spot', 'anatomical landmark'}:
                before = question[:hit['start']].strip().casefold().split()
                if not before or before[-1] not in {'the','one','a','an'}:
                    return ['missing_determiner_for_singular_operand']
    return []



def _dependency_reading(question, roles, spans):
    """Two bounded English constructions. Operate on the original text and public roles only."""
    if set(roles) not in ({"kind", "scope"}, {"finding_term", "scope"}):
        return None
    if sorted(s["role"] for s in spans) != sorted(roles):
        return None
    # Parse public role spellings as atomic nouns. In English "B." is often one token;
    # this projection keeps sentence punctuation separate and preserves original offsets.
    nouns = {"kind": "item", "scope": "view", "finding_term": "lesion"}
    parts, original, projected_roles = [], [0], {}
    last = 0
    for hit in sorted(spans, key=lambda h: h["start"]):
        start, end = hit["start"], hit["end"]
        parts.append(question[last:start])
        original.extend(range(last + 1, start + 1))
        noun = nouns[hit["role"]]
        projected_roles[hit["role"]] = (len(original) - 1, len(original) - 1 + len(noun))
        parts.append(noun)
        original.extend([start] * (len(noun) - 1) + [end])
        last = end
    parts.append(question[last:])
    original.extend(range(last + 1, len(question) + 1))
    doc = get_dependency_nlp()("".join(parts))
    if not doc.has_annotation("DEP") or len(list(doc.sents)) != 1:
        return None
    owner, role_nodes = {}, {}
    for role, (start, end) in projected_roles.items():
        span = doc.char_span(start, end, alignment_mode="strict")
        if span is None or len(span) != 1:
            return None
        role_nodes[role] = span[0]
        owner[span[0].i] = span[0].i
    outside = [t for t in doc if t.i not in owner and not t.is_space]
    roots = [t for t in outside if t.dep_ == "ROOT"]
    if len(roots) != 1:
        return None
    root = roots[0]
    content = [t for t in outside if not t.is_punct]
    accounted = {root.i}

    def linked(child, parents, labels):
        return (child is not None and child.dep_ in labels
                and owner.get(child.head.i, child.head.i) in {p.i for p in parents if p is not None})

    def pick(forms, parents, labels):
        matched = [t for t in content if t.i not in accounted
                   and t.lower_ in forms and linked(t, parents, labels)]
        if len(matched) == 1:
            accounted.add(matched[0].i)
            return matched[0]
        return None

    scope = role_nodes["scope"]
    language_ok = True
    if "kind" in role_nodes:
        kind = role_nodes["kind"]
        if root.lower_ not in {"name", "identify"} or not linked(kind, [root], {"dobj", "obj"}):
            return None
        determiner = pick({"a", "an", "one"}, [kind], {"det", "nummod"})
        if determiner is None:
            return None
        polite = pick({"please"}, [root], {"advmod", "intj"})
        visible = pick({"visible"}, [kind], {"amod", "acl"})
        reduced = pick({"located", "shown", "depicted", "seen", "observed"}, [kind], {"acl"})
        prep = pick({"within", "in"}, [reduced] if reduced is not None else [root, kind, visible],
                    {"prep"})
        if prep is None or not linked(scope, [prep], {"pobj", "obj"}):
            return None
        # The participle describes where the evidence is shown. It may not request a location.
        language_ok = (root.tag_ in {"VB", "VBP"} and root.i < determiner.i < kind.i
                       and (polite is None or polite.i < root.i)
                       and (visible is None or determiner.i < visible.i < kind.i
                            or reduced is None and kind.i < visible.i < prep.i)
                       and (reduced is None or kind.i < reduced.i < prep.i))
        operation = "identify"
        permitted_surface = {"name", "identify", "please", "a", "an", "one", "visible",
                             "located", "shown", "seen", "observed", "depicted", "in", "within"}
    else:
        operand = role_nodes["finding_term"]
        if root.lemma_ != "assign" or root.lower_ != "assigned":
            return None
        group = pick({"group"}, [root], {"nsubjpass", "nsubj:pass"})
        if group is None:
            return None
        wh = pick({"what", "which"}, [group], {"det"})
        clinical = pick({"clinical"}, [group], {"amod"})
        aux = pick({"is"}, [root], {"auxpass", "aux:pass"})
        agent = pick({"by"}, [root], {"agent", "prep"})
        dataset = pick({"dataset"}, [agent], {"pobj", "obj"})
        article = pick({"the"}, [dataset], {"det"})
        to = pick({"to"}, [root], {"prep", "dative"})
        if any(x is None for x in (wh, clinical, aux, agent, dataset, article, to)):
            return None
        if not linked(operand, [to], {"pobj", "obj"}):
            return None
        operand_article = pick({"a", "an", "the"}, [operand], {"det"})
        prep = pick({"within", "in"}, [root, operand, to], {"prep"})
        if prep is None or not linked(scope, [prep], {"pobj", "obj"}):
            return None
        language_ok = (wh.i < clinical.i < group.i < aux.i < root.i and root.tag_ == "VBN"
                       and root.i < agent.i < article.i < dataset.i and root.i < to.i < operand.i
                       and (operand_article is None or to.i < operand_article.i < operand.i))
        operation = "classify_named_finding"
        permitted_surface = {"which", "what", "clinical", "group", "is", "assigned", "by",
                             "the", "dataset", "to", "a", "an", "within", "in"}

    # Any unaccounted word, verb, modifier, negation, or second act prevents full coverage.
    if accounted != {t.i for t in content}:
        return None
    punct = [t for t in outside if t.is_punct]
    final = [t for t in punct if t.text in {".", "?"} and t.i == len(doc) - 1]
    commas = [t for t in punct if t.text == ","]
    if len(punct) != len(final) + len(commas) or len(commas) > 1:
        return None
    scope_hit = next(hit for hit in spans if hit["role"] == "scope")
    scope_front = prep.i == 0
    comma_ok = (len(commas) == 1 and original[commas[0].idx] >= scope_hit["end"]
                and question[scope_hit["end"]:original[commas[0].idx]].strip() == "") if scope_front else not commas
    scope_last = max(original[t.idx + len(t)] for t in doc if not t.is_space and not t.is_punct)
    language_ok = (language_ok and comma_ok and original[prep.idx] < scope_hit["start"]
                   and (scope_front or scope_hit["end"] == scope_last))
    if operation == "classify_named_finding":
        language_ok = language_ok and (not final or final[0].text == "?")
    # This separate check concerns outbound vocabulary; protected aliases are checked by the caller.
    disclosure_ok = all(t.lower_ in permitted_surface for t in content)
    return {"operation": operation, "language_verified": bool(language_ok),
            "disclosure_clear": disclosure_ok, "rule": "bounded_dependency_roles_v1",
            "model": DEPENDENCY_MODEL_ID,
            "covered_tokens": [[original[t.idx], original[t.idx + len(t)]] for t in doc if not t.is_space]}


def _inspect_english(question, role_surfaces):
    """Only public role surfaces are available to this parser."""
    tokens, spans, ambiguous = _role_spans(question, role_surfaces)
    covered = _whole_request_operations(tokens)
    plain = ' '.join(tokens)
    conflicts = []
    other_acts = []
    # The tokens of licensed clinical names and scope identifiers are masked.
    if re.search(r'\b(?:no|not|absent|without|absence|never)\b', plain):
        conflicts.append('negative_polarity')
    if re.search(r'\b(?:every|each) (?:one|@kind|@finding|@finding_term|finding|frame|panel)\b|\b(?:list|name|give|identify|show|locate) (?:them )?(?:all|both|many)\b|\bhow many\b', plain):
        conflicts.append('changed_response_cardinality')
    if re.search(r'\b(?:most obvious|most prominent|least obvious|largest|smallest|best|the unique|the only)\b', plain):
        conflicts.append('ranking_or_uniqueness')
    if re.search(r'\b(?:inflamed|severe|abnormal|bleeding|malignant|endoscopic|mucosal|red spot)\b', plain):
        conflicts.append('additional_unlicensed_clinical_content')
    if re.search(r'\b(?:recommend|prescribe|explain|diagnose|treat|treatment|give its location)\b', plain):
        conflicts.append('additional_or_disallowed_request')
        other_acts.append('additional_request')
    if re.search(r'\b(?:onset|developed|normalized)\b', plain):
        conflicts.append('changed_clinical_or_spatial_requirement')
    if (re.match(r'^(?:does|do|is|are|has|have|was|were) (?:@\w+|the\b|panel\b|frame\b|this\b|that\b|there\b)', plain)
        or re.search(r', (?:is|was|does) (?:the\b|panel|frame|@option_)', plain)):
        conflicts.append('presence_or_confirmation_request')
        other_acts.append('verify')
    if re.search(r'\bfirst to show\b', plain):
        conflicts.append('additional_temporal_ranking')
    if re.search(r'\b(?:before|after|later)\b', plain) and ('direction' in role_surfaces or 'option_a' in role_surfaces):
        conflicts.append('changed_temporal_direction')
    if re.search(r'\b(?:panel|frame) [a-z0-9](?:\b|$)', plain) and 'scope' in role_surfaces and '@scope' not in tokens:
        conflicts.append('different_named_visual_referent')
    if re.search(r'\b(?:anatomical landmark|finding)\b', plain) and 'kind' in role_surfaces and '@kind' not in tokens:
        conflicts.append('different_requested_kind')
    if re.search(r'\b(?:panel|frame) [a-z0-9] is the target\b', plain):
        conflicts.append('answer_by_elimination')
    if re.search(r'\( \d+ , \d+ \)', plain) and 'box_format' in role_surfaces:
        conflicts.append('suggested_coordinates')
    if '@option_a' in tokens and '@option_b' in tokens and tokens.index('@option_b') < tokens.index('@option_a'):
        conflicts.append('changed_supplied_option_order')
    cfg_covered = bool(covered)
    dependency = None
    if not covered and not conflicts and not ambiguous:
        dependency = _dependency_reading(question, role_surfaces, spans)
        if dependency is not None:
            covered = [dependency["operation"]]
    return {'cfg_full': cfg_covered, 'dependency_reading': dependency,
            'operation_candidates': sorted(set(covered + other_acts)), 'full_text_parse': bool(covered),
            'speech_acts': ['request_one_answer'] if covered else other_acts or ['unknown'],
            'asserted_attributes': [] if covered else None, 'additional_requests': [] if covered else other_acts or None,
            'scope_spans': [s for s in spans if s['role'] in _PUBLIC_VISUAL_ROLES],
            'temporal_reading': ('earlier_in_acquisition' if 'select_earlier_observed_frame' in covered else
                [s['surface'] for s in spans if s['role']=='direction'] if 'localize_relative_evidence' in covered else
                'none' if covered else 'unknown'),
            'role_spans': spans, 'ambiguous_readings': ambiguous,
            'response_cardinality': 1 if covered else ('all' if 'changed_response_cardinality' in conflicts else 'unknown'),
            'polarity': 'positive' if covered else ('negative' if 'negative_polarity' in conflicts else 'unknown'),
            'uninterpreted_spans': [] if covered else [{'start': 0, 'end': len(question)}],
            'conflicts': conflicts}


def validate_candidate(case, candidate, *, interpretation_reviewer=None):
    """Return individual outcomes. Only the comparator uses the frozen projection and private disclosure terms."""
    task = case['task_id']
    request = case['public_request']
    plan = case['frozen_plan']
    roles = {key.lower(): values for key, values in case['role_surfaces'].items()}
    projected_roles = {key[:-9]: values for key, values in request.items() if key.endswith('_surfaces')}
    plan_roles = {key.lower(): value for key, value in plan['roles'].items()}
    # The plan and the trusted per-instance role lexicon are private comparator inputs.
    # A broken projection/receipt is a run integrity failure, before candidate comparison.
    if (projected_roles != roles or set(plan_roles) != set(roles) or
        any(not v for v in plan_roles.values()) or plan['task_id'] != task or plan['profile_id'] != case['profile_id'] or
        (request['operation'], request['response_cardinality'], request['polarity']) !=
        (plan['public_operation'], plan['response_cardinality'], plan['polarity'])):
        raise ValueError('candidate_comparator_frozen_projection_mismatch')
    parser = {'parser_id': 'mmvqa_bounded_english_parser', 'version': _CHECKER_VERSION, 'calibration_id': None}
    interpretation, checks = None, []
    def record(short_id, status, reasons=()):
        checks.append({'check_id': task + '.' + short_id, 'stage': 'candidate_validation',
                       'status': status, 'reason_codes': list(reasons), 'evidence_refs': [],
                       'implementation_version': _CHECKER_VERSION})
    role_check = 'ontology_lexicon_grounded' if task == 'finding_identification' else 'public_roles_grounded'
    scope_check = 'public_scope_unambiguous' if task == 'finding_identification' else 'public_referents_unambiguous'
    downstream = ('question_semantic_form', role_check, scope_check, 'paraphrase_equivalence',
                  'question_language_quality', 'answer_not_disclosed')
    shape_ok = isinstance(candidate, dict) and set(candidate) == {'question'} and isinstance(candidate['question'], str)
    q = candidate['question'] if shape_ok else ''
    hygiene = bool(q) and q == q.strip() and all(unicodedata.category(c) != 'Cc' and c not in '\u2028\u2029' for c in q)
    record('question_json_shape', 'pass' if shape_ok and hygiene else 'rejected',
           [] if shape_ok and hygiene else ['invalid_question_object_or_text'])
    if not (shape_ok and hygiene):
        for key in downstream:
            record(key, 'not_run', ['failed_input_shape'])
    else:
        interpretation = _inspect_english(q, roles)
        complete = interpretation['full_text_parse']
        dependency = interpretation.get('dependency_reading') or {}
        english_verified = interpretation['cfg_full'] or dependency.get('language_verified', False)
        if dependency:
            parser = {**parser, 'parser_id': 'mmvqa_dependency_graph_comparator',
                      'dependency_model': DEPENDENCY_MODEL_ID, 'dependency_rule': dependency['rule']}
        meanings = interpretation['operation_candidates']
        conflicts = interpretation['conflicts']
        ambiguities = interpretation['ambiguous_readings']
        grounded_roles = {hit['role'] for hit in interpretation['role_spans']}
        recovered = {r: plan_roles[r] for r in grounded_roles if r in plan_roles}
        interpretation['resolved_operand_bindings'] = recovered
        missing_roles = sorted(set(plan_roles) - grounded_roles)
        ref_missing = sorted(_PUBLIC_VISUAL_ROLES.intersection(missing_roles))
        act_conflict = any(code in conflicts for code in ('additional_or_disallowed_request','presence_or_confirmation_request'))
        record('question_semantic_form', 'rejected' if act_conflict else 'pass' if complete else 'review',
               ['incompatible_speech_act'] if act_conflict else [] if complete else ['speech_act_unresolved'])
        term_conflict = any(c in conflicts for c in ('additional_unlicensed_clinical_content', 'different_requested_kind'))
        roles_ok = complete and not missing_roles and not ambiguities
        record(role_check, 'rejected' if term_conflict else 'pass' if roles_ok else 'review',
               ['unlicensed_or_incompatible_public_role'] if term_conflict else [] if roles_ok else ['unverified_role_coverage'])
        ambiguous_scope = bool(case.get('ambiguous_scope')) or bool(case.get('ambiguous_public_referents'))
        scope_rejected = 'different_named_visual_referent' in conflicts
        scope_ok = not ref_missing and not ambiguous_scope and not ambiguities
        record(scope_check, 'rejected' if scope_rejected else 'pass' if scope_ok else 'review',
               ['different_named_visual_referent'] if scope_rejected else [] if scope_ok else ['unresolved_visual_referent'])
        comparison_conflicts = list(conflicts)
        if complete and meanings and plan['public_operation'] not in meanings:
            comparison_conflicts.append('operation_conflicts_with_frozen_plan')
        match = (complete and meanings == [plan['public_operation']] and recovered == plan_roles and roles_ok and scope_ok and
                 interpretation['polarity'] == plan['polarity'] and
                 interpretation['response_cardinality'] == plan['response_cardinality'])
        record('paraphrase_equivalence', 'rejected' if comparison_conflicts else 'pass' if match else 'review',
               comparison_conflicts or ([] if match else ['full_meaning_not_established']))
        article_errors = _article_errors(q, interpretation['role_spans'], complete)
        if dependency and re.search(r'\ban\s+visible\b', q, re.I):
            article_errors = ['confirmed_indefinite_article_error']
        record('question_language_quality', 'rejected' if article_errors else 'pass' if english_verified else 'review',
               article_errors or ([] if english_verified else ['unverified_english_wording']))
        certain = any(_protected_match(q, s) for s in case['disclosure_terms'] if isinstance(s, str) and s)
        uncertain = any(_protected_match(q, s) for s in case.get('ambiguous_disclosure_terms', []) if isinstance(s, str) and s)
        leaked = certain or bool({'answer_by_elimination', 'suggested_coordinates'}.intersection(conflicts))
        clear = (interpretation['cfg_full'] or dependency.get('disclosure_clear', False)
                 or _novel_wording_disclosure_clear(q, roles))
        record('answer_not_disclosed', 'rejected' if leaked else 'review' if uncertain or not clear else 'pass',
               ['answer_disclosed'] if leaked else ['possible_answer_disclosure'] if uncertain else
               ['unresolved_indirect_disclosure'] if not clear else [])
    statuses = {row['status'] for row in checks}
    verdict = 'rejected' if 'rejected' in statuses else 'review' if 'review' in statuses else 'accepted'
    if (verdict == 'review' and interpretation_reviewer is not None and
        checks[-1]['status'] == 'pass'):
        interpretation['independent_advisory'] = interpretation_reviewer(q, roles)
    return {'verdict': verdict, 'checks': checks, 'interpretation': interpretation, 'parser_identity': parser,
            'reasons': sorted({s for row in checks for s in row['reason_codes'] if row['status'] != 'not_run'})}


### 5 · Attempts, audit records and QA assembly

Rejected candidates and candidates sent for review each consume one attempt. For each semantic instance, Sector 2 counts an accepted wording once across attempts within the campaign. Its versioned sampling policy ignores case, repeated whitespace and terminal periods, question marks or exclamation marks. A punctuation-only duplicate consumes an attempt. The audit keeps the exact original text and its digest. The build records the verified V13 hashes and the effective policy. A `fatal_abort` closes the batch, withholds its candidates and records its attempts. Assembly recomputes the question digest and reloads the bound interpretation and evidence.


In [10]:
%%sector2_component sampling
"""Per-instance sampling, binding of verdicts to exact text, and separate corpus release."""


WORDING_POLICY_ID = "mmvqa.sector2.distinct-wording.v2"
_V13_NORMALIZATION = [
    "unicode_nfc", "unicode_casefold", "unicode_nfc",
    "collapse_unicode_whitespace_runs_to_one_ascii_space_and_strip",
]


def normalized_question(text):
    """Count wording once per instance. Keep the original UTF-8 text for audit."""
    folded = unicodedata.normalize("NFC", text).casefold()
    spaced = " ".join(unicodedata.normalize("NFC", folded).split())
    # Repeated terminal marks or spaces between them do not change the wording.
    # Technical punctuation inside a role (image_1, 1.5, [x1, y1]) stays significant.
    return spaced.rstrip(" .?!")


def text_sha(text):
    return hashlib.sha256(text.encode("utf-8", errors="strict")).hexdigest()


class AuditStore:
    """Private content-addressed audit and atomic per-batch corpus insertion."""
    def __init__(self, directory):
        self.root = Path(directory)
        self.root.mkdir(parents=True, exist_ok=True)
        self.db = sqlite3.connect(self.root / "corpus.sqlite3")
        self.db.execute("""CREATE TABLE IF NOT EXISTS qa (
            instance TEXT NOT NULL, normalized TEXT NOT NULL, body TEXT NOT NULL,
            PRIMARY KEY(instance, normalized))""")

    def put(self, kind, value):
        data = _wire_bytes(value)
        digest = hashlib.sha256(data).hexdigest()
        path = self.root / "private" / kind / (digest + ".json")
        if not path.exists():
            _write_atomic(path, data)
        elif path.read_bytes() != data:
            raise ValueError("audit object digest mismatch")
        return digest

    def get(self, kind, digest):
        if not re.fullmatch("[a-f0-9]{64}", digest):
            raise ValueError("invalid audit identifier")
        data = (self.root / "private" / kind / (digest + ".json")).read_bytes()
        if hashlib.sha256(data).hexdigest() != digest:
            raise ValueError("audit object changed")
        return _s1_json(data)

    def insert(self, qas):
        inserted = 0
        with self.db:
            for qa in qas:
                cur = self.db.execute("INSERT OR IGNORE INTO qa VALUES (?, ?, ?)",
                                      (qa["semantic_instance_id"], normalized_question(qa["question"]),
                                       _wire_bytes(qa).decode()))
                inserted += cur.rowcount
        return inserted

    def write_view(self, filename):
        rows = self.db.execute("SELECT body FROM qa ORDER BY instance, normalized")
        dest = self.root / filename
        with tempfile.NamedTemporaryFile(dir=self.root, delete=False) as f:
            tmp = Path(f.name)
            try:
                for (body,) in rows:
                    f.write(body.encode("utf-8") + b"\n")
                f.flush()
                os.fsync(f.fileno())
                os.replace(tmp, dest)
            finally:
                tmp.unlink(missing_ok=True)
        return dest


def _selected_checks(semantics, profile, stage):
    rules = semantics["required_checks"]
    ids = [*rules["common"], *rules["additional_by_profile"][profile]]
    return {k for k in ids if stage in semantics["check_registry"][k]["criteria_by_stage"]}


def sample_case(case, grammar, semantics, receipt, store, generate, target, budget, reviewer=None, *,
                attempt_offset=0, prior_question_keys=()):
    """One call per wording attempt. Only validated unique questions count."""
    if type(target) is not int or type(budget) is not int or min(target, budget) < 1:
        raise ValueError("positive target and attempt budget required")
    receipt_id = store.put("pre", receipt)
    payload = store.get("requests", receipt["public_request_sha256"])
    accepted, attempts, seen = [], [], set(prior_question_keys)
    outcome_counts = Counter()
    stop = "attempt_budget_exhausted"
    for attempt_no in range(attempt_offset + 1, attempt_offset + budget + 1):
        trace = {
            "semantic_instance_id": case["semantic_instance_id"], "evidence_snapshot_id": case["evidence_snapshot_id"],
            "build_id": receipt["build_id"], "pre_evidence_receipt_id": receipt_id,
            "public_request_sha256": receipt["public_request_sha256"], "attempt_no": attempt_no,
            "parser_identity": None, "interpretation_sha256": None, "candidate_question_sha256": None,
            "wording_policy_id": WORDING_POLICY_ID,
            "question": None, "checks": [], "reasons": [], "validation_verdict": "not_run"
        }
        try:
            candidate, provider = generate(payload, receipt["public_request_sha256"], attempt_no)
            trace["provider"] = provider
            trace["candidate_object_sha256"] = store.put("candidates", candidate)
            q = candidate.get("question") if isinstance(candidate, dict) else None
            if isinstance(q, str):
                trace.update(question=q, candidate_question_sha256=text_sha(q))
            result = validate_candidate(case, candidate, interpretation_reviewer=reviewer)
            verdict = result["verdict"]
            ids = _selected_checks(semantics, case["profile_id"], "after_generation")
            reported = [c["check_id"] for c in result["checks"]]
            if set(reported) != ids or len(reported) != len(ids):
                raise RuntimeError("incomplete candidate check registry")
            shape = grammar["output_contract"]["base_json_schema"]
            if verdict == "accepted" and not Draft202012Validator(shape).is_valid(candidate):
                raise RuntimeError("authoritative question schema disagreement")
            parser = {**result["parser_identity"], "implementation_sha256": receipt["parser_implementation_sha256"]}
            trace.update(validation_verdict=verdict, reasons=result["reasons"], checks=result["checks"], parser_identity=parser)
            if result["interpretation"] is not None:
                trace["interpretation_sha256"] = store.put("interpretations", result["interpretation"])
            disposition = verdict
            if verdict == "accepted":
                if not trace["interpretation_sha256"] or any(c["status"] != "pass" for c in trace["checks"]):
                    raise RuntimeError("accepted without complete interpretation")
                key = normalized_question(q)
                if key in seen:
                    disposition = "duplicate"
                    trace["reasons"] = [*trace["reasons"], "duplicate_wording_after_normalization"]
                seen.add(key)
        except SamplingProviderError as exc:
            disposition = exc.disposition
            trace.update(provider=exc.metadata, reasons=[exc.code])
        except Exception as exc:
            disposition = "fatal_abort"
            trace["reasons"] = ["engine_failure:" + type(exc).__name__, str(exc)[:240]]
        trace["disposition"] = disposition
        audit_id = store.put("attempts", trace)
        attempts.append({"attempt_no": attempt_no, "audit_id": audit_id, "disposition": disposition,
                         "candidate_question_sha256": trace["candidate_question_sha256"], "reasons": trace["reasons"]})
        outcome_counts[disposition] += 1
        if disposition == "fatal_abort":
            stop, accepted = "blocked", []
            break
        if disposition == "accepted":
            accepted.append({"question": q, "candidate_question_sha256": text_sha(q), "audit_id": audit_id})
            if len(accepted) == target:
                stop = "target_reached"
                break
    return {
        "semantic_instance_id": case["semantic_instance_id"], "pre_evidence_receipt_id": receipt_id,
        "requested_count": target, "attempt_budget": budget, "attempts_used": len(attempts),
        "accepted_count": len(accepted), "shortfall": target - len(accepted), "stop_reason": stop,
        "counts_by_outcome": dict(outcome_counts), "attempts": attempts, "accepted": accepted
    }


def assemble_validated(row, store, *, synthetic, source, build_id, for_corpus=False):
    audit = store.get("attempts", row["audit_id"])
    receipt = store.get("pre", audit["pre_evidence_receipt_id"])
    frozen = store.get("plans", receipt["case_sha256"])
    original = store.get("candidates", audit["candidate_object_sha256"])
    snapshot = store.get("snapshots", receipt["evidence_snapshot_id"])
    store.get("requests", receipt["public_request_sha256"])
    store.get("interpretations", audit["interpretation_sha256"])
    q = row["question"]
    if not (audit["validation_verdict"] == audit["disposition"] == "accepted"
            and original == {"question": q} and audit["question"] == q
            and text_sha(q) == row["candidate_question_sha256"] == audit["candidate_question_sha256"]
            and audit["public_request_sha256"] == receipt["public_request_sha256"]
            and audit["semantic_instance_id"] == frozen["semantic_instance_id"] == receipt["semantic_instance_id"]
            and audit["evidence_snapshot_id"] == frozen["evidence_snapshot_id"] == receipt["evidence_snapshot_id"]
            and audit["build_id"] == receipt["build_id"] == build_id
            and receipt["kind"] == ("synthetic_demo" if synthetic else "verified_sector1")):
        raise ValueError("QA release did not match the validated wording and frozen receipt")
    if not synthetic and not (source and source["run_id"] == receipt["source_run_id"]
                              and receipt["authorization"]["effective_qa_generation_authorized"] is True):
        raise PermissionError("no verified Sector 2 generation decision")
    if for_corpus and (synthetic or source["corpus_export_authorized"] is not True):
        raise PermissionError("corpus release requires a separate upstream export grant")
    presentation = receipt["presentation"]
    if not synthetic:
        if presentation != snapshot.get("respondent_presentation"):
            raise ValueError("S8 presentation differs from the frozen evidence")
        presentation = respondent_s8_reference(presentation)
    return {
        "kind": receipt["kind"], "semantic_instance_id": audit["semantic_instance_id"], "task_id": frozen["task_id"],
        "profile_id": frozen["profile_id"], "question": q, "answer": frozen["private_answer"]["canonical"],
        "reference": frozen["private_answer"], "presentation": presentation,
        "audit_id": row["audit_id"], "candidate_question_sha256": text_sha(q),
        "pre_evidence_receipt_id": audit["pre_evidence_receipt_id"],
        "evidence_snapshot_id": frozen["evidence_snapshot_id"], "corpus_export_authorized": bool(for_corpus),
        "release_status": "corpus_export" if for_corpus else ("synthetic_demo" if synthetic else "local_validated_draft")
    }


### 6 · Sector 2 execution

Use the verified Section 1 inputs to apply the matrix, frozen contracts and terminology to the evidence. The run records generation authority and screens each public request before dispatch. A completed census can be reused for subsequent generation batches.


In [11]:
%%sector2_component workflow
def generation_policy(source, mode):
    # A versioned, project-wide change of generation authority. Record both decisions.
    # Clinical role eligibility and corpus export still use the verified source release.
    real = mode == "real_openai"
    return {
        "policy_id": "mmvqa.sector2.generation-authority.v1",
        "authority_for_generation": "sector2",
        "sector1_qa_generation_authorized": source["qa_generation_authorized"] if source else None,
        "sector2_qa_generation_authorized": real,
        "qa_generation_authorized": real,  # effective field in the S2 run manifest
        "effective_qa_generation_authorized": real,
        "sector1_export_authorized": source["corpus_export_authorized"] if source else False,
        "target": "synthetic_demo" if mode.startswith("demo_") else "verified_sector1",
    }


def effective_contracts(verified, policy):
    # Retain the verified V13 bytes and identify the S2 policies in a separate effective copy.
    updated = copy.deepcopy(verified)
    for task, pair in updated.items():
        key = pair["grammar"]["variation_policy"]["distinct_key"]
        if (key["fields"] != ["semantic_instance_id", "normalized_question"]
                or key["normalized_question"] != _V13_NORMALIZATION or key["punctuation"] != "preserved"):
            raise ValueError("Review the updated V13 distinct-question policy before applying the S2 overlay")
        key.update(
            policy_id=WORDING_POLICY_ID,
            normalized_question=[*_V13_NORMALIZATION, "strip_trailing_ascii_sentence_marks_and_spaces"],
            punctuation="ignore_final_runs_of_period_question_mark_exclamation_mark_or_space",
        )
        sem = pair["semantics"]
        auth = sem["source_binding"]["authorization"]
        if auth["authority"] != "verified_upstream_release" or auth["task_local_override"] != "forbidden":
            raise ValueError("generation authority in the source contracts has changed")
        sem["source_binding"]["authorization"] = {
            "policy_id": policy["policy_id"], "generation_authority": "sector2",
            "generation_requires": "explicit_sector2_generation_permission",
            "export_authority": "verified_upstream_release", "export_requires": auth["export_requires"],
            "on_denied_missing_or_unverified": auth["on_denied_missing_or_unverified"],
            "task_local_override": "forbidden"
        }
        pre = sem["check_registry"][task + ".generation_preflight"]["criteria_by_stage"]
        pre["before_generation"] = pre["before_generation"].replace("upstream permission", "Sector 2 generation permission")
    return updated


def _matrix_row(cells, task, evidence):
    found = [r for r in cells[task] if r["variant"] == evidence["variant"]
             and r["evidence_state"] == evidence["evidence_state"] and r["task"] == task
             and r["original_policy_code"] == "A" and r["expected_status"] == "applicable"]
    if not found:
        raise EvidenceHold("matrix_cell_not_eligible", "not_applicable")
    if len(found) != 1:
        raise ValueError("ambiguous matrix row identity")
    return {k: found[0][k] for k in ("task", "variant", "evidence_state", "rule_id",
                                      "original_policy_code", "expected_status")}


def _save_json(path, value):
    _write_atomic(path, (json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n").encode())


def _json_line(file, value):
    file.write(_wire_bytes(value).decode("utf-8") + "\n")
    file.flush()



def _final_snapshot_stat(stat):
    return [stat.st_dev, stat.st_ino, stat.st_size, stat.st_mtime_ns, stat.st_ctime_ns]


def finalized_inventory_snapshot(result, temp):
    """Resolve the completed snapshot authenticated by the run and by its campaign policy."""
    run = Path(result["run_directory"]).resolve()
    build_id = result["build_id"]
    build = _s1_json((run / "build.json").read_bytes())
    if (wire_digest(build) != build_id or run.parents[1].name != build_id[:16]
            or run.parent.name != "runs" or run.parents[2].name != "snapshot_runs"):
        raise ValueError("Generation build receipt or directory has changed.")
    inventory_id = build["inventory_id"]
    policy = result["generation_policy"]
    receipt = _s1_json((run / "inventory_snapshot.json").read_bytes())
    sha = receipt["database_sha256"]
    if (not isinstance(inventory_id, str) or not re.fullmatch(r"[a-f0-9]{64}", inventory_id)
            or not isinstance(sha, str) or not re.fullmatch(r"[a-f0-9]{64}", sha)
            or any(item.get("inventory_id") != inventory_id for item in (receipt, result, policy))
            or any(item.get("inventory_database_sha256") != sha for item in (result, policy))):
        raise ValueError("Snapshot identity differs between the run, build and campaign.")
    directory = run.parents[3] / "inventories" / inventory_id
    manifest = directory / "inventory.json"
    snapshot = directory / "snapshots" / (sha + ".sqlite3")
    if (Path(receipt["manifest_path"]).resolve() != manifest
            or Path(result["inventory_manifest"]).resolve() != manifest
            or Path(receipt["database_path"]).resolve() != snapshot):
        raise ValueError("Snapshot locations differ from the run receipt.")
    saved = _s1_json(manifest.read_bytes())
    if (saved.get("format") != "mmvqa-s2-inventory-snapshot-v1" or saved["inventory_id"] != inventory_id
            or wire_digest(saved["material"]) != inventory_id or saved["database_sha256"] != sha
            or saved["database_file"] != "snapshots/" + sha + ".sqlite3" or saved["complete"] is not True
            or saved["recipes_done"] != saved["expected_recipes"] or saved["recipes_done"] != receipt["recipes_done"]):
        raise ValueError("The manifest does not identify the campaign's completed database.")
    persistent_token = _final_snapshot_stat(snapshot.stat())
    if persistent_token[2] != saved["database_size_bytes"]:
        raise ValueError("The completed snapshot size has changed.")
    local = Path(receipt["verified_local_path"]).resolve()
    if local != Path(result["inventory_database"]).resolve().parent / "verified_snapshot.sqlite3":
        raise ValueError("Local snapshot path differs from the verified run directory.")
    # Use the exact, sealed VM copy previously checked against Drive. A changed or missing copy
    # or a remounted Drive triggers one verified transfer into the VM's temporary directory.
    if (persistent_token == receipt["verified_file_stat"] and local.is_file()
            and _final_snapshot_stat(local.stat()) == receipt["verified_local_stat"]):
        return local, receipt["verified_local_stat"]
    local = temp / "snapshot.sqlite3"
    _local_inventory_volume(local)
    digest = hashlib.sha256()
    with snapshot.open("rb") as source, local.open("xb") as target:
        if _final_snapshot_stat(os.fstat(source.fileno())) != persistent_token:
            raise ValueError("The completed snapshot changed before transfer.")
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
            target.write(block)
    if digest.hexdigest() != sha or _final_snapshot_stat(snapshot.stat()) != persistent_token:
        raise ValueError("Completed database SHA-256 verification failed.")
    return local, _final_snapshot_stat(local.stat())


def execute_sector2(config, imported, component_hashes):
    mode = config["mode"]
    if mode not in SECTOR2_MODES:
        raise ValueError("Unknown Sector 2 execution mode: " + str(mode))
    for key in ("max_http_calls", "max_total_api_calls", "max_batches_per_run"):
        if type(config[key]) is not int or config[key] < 0:
            raise ValueError(key + " must be a nonnegative integer")
    for key in ("requested_count", "attempt_budget", "batch_size", "checkpoint_recipes"):
        if type(config[key]) is not int or config[key] < 1:
            raise ValueError(key + " must be a positive integer")
    window_size = config.get("inventory_window_size", 5000)
    if type(window_size) is not int or window_size < 1:
        raise ValueError("inventory_window_size must be a positive integer")
    workers = config.get("inventory_workers", 1)
    if type(workers) is not int or workers < 0:
        raise ValueError("inventory_workers must be a nonnegative integer")
    root = Path(config["output_dir"]).expanduser().resolve()
    if not imported or "registry" not in imported:
        raise ValueError("Run the Section 1 import first")
    if any(component_hashes.get(k) != v for k, v in imported.get("import_components", {}).items()):
        raise ValueError("The import implementation changed. Rerun Section 1.")
    if imported["s8_context"] is None:
        raise ValueError("Run Section 1 in inventory_only or real_openai mode to import S8")
    if Path(config["s8_dir"]).expanduser().resolve() != Path(imported["s8_dir"]):
        raise ValueError("The VM input directory changed. Rerun Section 1.")
    registry, source = imported["registry"], imported["source"]
    verified = registry["contracts"]
    selected = set(config["tasks"] or verified)
    if selected - verified.keys():
        raise ValueError("unknown selected task(s): " + ", ".join(sorted(selected - verified.keys())))
    auth = generation_policy(source, mode)
    contracts = effective_contracts(verified, auth)
    counts = {task: len(rows) for task, rows in source["selected_cells"].items()}
    release = {k: source[k] for k in ("run_id", "data_commit", "registry_commit", "files_manifest_sha256")}
    contract_manifest = {k: registry[k] for k in ("manifest_sha256", "manifest_origin",
                                                  "manifest_git_blob_sha", "registry_commit")}
    build = {"dependency_validator": (dependency_runtime_identity() if mode == "real_openai"
                                      else {"used": False}),
             "runtime_components": component_hashes, "contract_hashes": registry["contract_sha256"],
             "contract_git_blob_sha": registry.get("contract_git_blob_sha", {}),
             "contract_manifest": contract_manifest, "upstream_code": imported["upstream_code"], "source": release,
             "effective_contracts_sha256": wire_digest(contracts), "policy": auth["policy_id"],
             "wording_policy_id": WORDING_POLICY_ID, "schema_engine": package_version("jsonschema")}
    material = inventory_identity_material(build, component_sources, source["verified_digests"])
    inventory_id = wire_digest(material)
    build["inventory_id"] = inventory_id
    build_id = wire_digest(build)
    shared = root / "real_s8" / "snapshot_runs" / build_id[:16]
    run = shared / "runs" / (dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8])
    run.mkdir(parents=True)
    _save_json(run / "build.json", build)
    info = {"mode": mode, "build_id": build_id, "run_directory": str(run),
            "selected_A_cells": counts, "authorization": auth, "source": release,
            "qa_generation_authorized": auth["qa_generation_authorized"], "contract_manifest": contract_manifest,
            "contract_directory": str(registry["staged_directory"]), "coverage_source": "sector1.coverage_contract",
            "http_calls": 0, "verified_contract_files": len(registry["contract_sha256"])}
    info["inventory_id"] = inventory_id
    info["run_id"] = run.name
    _save_json(run / "verified_sector1.json", {"source": release, "authorization": auth,
                                              "verified_references": source["verified_references"],
                                              "file_sha256": source["verified_digests"]})
    info["contract_import"] = registry["contract_sync"]
    if "import_report" in source:
        info["sector1_import"] = source["import_report"]
    info["s8_vm_sync"] = imported["s8_vm_sync"]
    return run_real_census_and_generation(config, source, contracts, shared, run, info, component_hashes,
                                          s8_context=imported["s8_context"], material=material)


def inventory_work_directory(config, inventory_id):
    """Only the live SQLite file and snapshot staging live on the VM."""
    return Path(config["s8_dir"]).expanduser().resolve().parent / "snapshot_work" / inventory_id


def run_real_census_and_generation(config, source, contracts, shared, run, info, component_hashes, *, s8_context, material):
    """Complete the seven-task census before opening an API client. Reuse the same census for later batches."""
    tasks = sorted(V13_TASKS)
    if set(contracts) != V13_TASKS:
        raise ValueError("All seven verified V13 task definitions are required")
    store = None
    try:
        inventory_id = info["inventory_id"]
        inventory_root = shared.parents[1] / "inventories" / inventory_id
        store = SnapshotInventory(inventory_work_directory(config, inventory_id), inventory_root,
                                  inventory_id, material,
                                  snapshot_interval=config.get("snapshot_every_recipes", 50000))
        info["inventory_manifest"] = str(inventory_root / "inventory.json")
        info["inventory_database"] = str(store.path)
        with store:
            info["inventory_storage"] = store.report
            inventory = store.db
            print("Inventory ID:", inventory_id, "|", store.report["mode"], "|", store.report["reason"], flush=True)
            print("VM working database:", store.path, flush=True)
            print("Persistent inventory manifest:", info["inventory_manifest"], flush=True)
            print("Verified recipes available:", store.end, flush=True)
            complete = bool(inventory.execute("SELECT complete FROM inventory_state WHERE singleton=1").fetchone()[0])
            corpus = None if complete else _clinical_s8_corpus(*s8_context)
            _save_json(run / "s8_vm_sync.json", info["s8_vm_sync"])
            workers = config.get("inventory_workers", 1)
            window_size = config.get("inventory_window_size", 5000)
            info["inventory_execution"] = {
                "inventory_id": inventory_id,
                "window_size": window_size, "checkpoint_recipes": config["checkpoint_recipes"],
                "requested_workers": workers, "worker_limit": resolve_inventory_workers(workers),
                "reused_complete_inventory": complete,
            }
            inventory_census(inventory, source, config["s8_dir"], contracts, corpus=corpus,
                             checkpoint_recipes=config["checkpoint_recipes"], workers=workers,
                             window_size=window_size, on_checkpoint=store.checkpoint)
            store.publish()
            receipt = store.report
            if not receipt["durable_complete"] or receipt["snapshot_sha256"] is None:
                raise RuntimeError("A verified, complete database snapshot is required before generation.")
            info["inventory_database_sha256"] = receipt["snapshot_sha256"]
            _save_json(run / "inventory_snapshot.json", {
                "inventory_id": inventory_id, "database_sha256": receipt["snapshot_sha256"],
                "manifest_path": info["inventory_manifest"], "database_path": str(receipt["snapshot_path"]),
                "verified_file_stat": receipt["snapshot_stat"], "recipes_done": receipt["durable_recipes"],
                "verified_local_path": str(receipt["verified_local_path"]),
                "verified_local_stat": receipt["verified_local_stat"],
            })
            del corpus
            census = inventory_summary(inventory, tasks, config["requested_count"])
            if not census["complete"]:
                raise RuntimeError("The full S8 census is incomplete")
            census["inventory_id"] = inventory_id
            info["inventory"] = census
            _save_json(run / "inventory_summary.json", census)
            reports = inventory_root / "reports" / info["inventory_database_sha256"]
            reports.mkdir(parents=True, exist_ok=True)
            holds = reports / "held_instances.jsonl"
            if not holds.is_file():
                temp = reports / ("held_instances-" + run.name + ".tmp")
                try:
                    write_inventory_holds(inventory, temp)
                    os.replace(temp, holds)
                finally:
                    temp.unlink(missing_ok=True)
            info["held_instances"] = str(holds)
            print("Recipes examined:", census["recipes_examined"], "| Unique eligible instances:", census["eligible_instances"])
            for task, row in census["per_task"].items():
                print(task, row["eligible_instances"], "instances;", row["target_accepted_questions"], "target questions")
            print("Corpus question target:", census["target_accepted_questions"])
            if config["mode"] == "inventory_only":
                info["status"] = "inventory_complete"
            else:
                if not info["authorization"]["effective_qa_generation_authorized"]:
                    raise PermissionError("Sector 2 generation authorization is required")
                generate_inventory_batch(config, inventory, contracts, source, info["authorization"], info["build_id"],
                                         component_hashes, run, info)
    except BaseException as exc:
        info["status"] = "interrupted" if isinstance(exc, KeyboardInterrupt) else "failed"
        info["error"] = {"type": type(exc).__name__, "detail": str(exc)[:1200]}
        sqlite_code = getattr(exc, "sqlite_errorcode", None)
        if sqlite_code is not None:
            info["error"].update(sqlite_errorcode=sqlite_code, sqlite_errorname=getattr(exc, "sqlite_errorname", None))
        if store is not None:
            info["inventory_storage"] = store.report
        # Report a failed checkpoint without querying a potentially damaged database.
        try:
            _save_json(run / "summary.json", info)
        except Exception as reporting_error:
            print("Could not write the failure summary:", reporting_error, flush=True)
        raise
    info["inventory_storage"] = store.report
    _save_json(run / "summary.json", info)
    return info


### 7 · S8 inventory and database snapshots

All seven tasks are checked against the applicable S1 cells. The worker resolves visual units once per S8 recipe. The parent writes the results to SQLite in recipe order. The default worker window and local commit interval are 5,000 recipes. By default, `SNAPSHOT_EVERY_RECIPES = 50000` publishes a full database after 50,000 additional committed recipes. This interval limits how often the entire database is copied and verified. A replacement VM resumes from the most recent published snapshot. During publication the VM needs space for the working database, the previous verified local copy and the new snapshot. The last commit produces a complete snapshot even if the total falls between intervals.

The inventory ID identifies the census inputs and the producer code. The snapshot SHA-256 identifies the exact SQLite file at the saved point in that census. The manifest binds both to progress and completion. SQLite runs on the VM. On each start or re-entry the integrity check reads the current Drive bytes before the file is used. A missing or damaged snapshot cannot inherit the progress of an older local SQLite file. A changed post-generation validator can use the same evidence inventory under a new generation build.


In [12]:
%%sector2_component inventory
# Full S8 eligibility census. The build ID binds the verified S1/S8 inputs, contracts and engine.
class _InventoryHandle:
    """A parent-owned SQLite connection that can be parked across Linux forks."""

    def __init__(self, path, build_id, connection):
        self.path = Path(path).resolve()
        self.build_id = build_id
        self._owner_pid = os.getpid()
        self._db = connection
        st = self.path.stat()
        self._file_identity = (st.st_dev, st.st_ino)
        self._closed = False

    def __getattr__(self, name):
        if os.getpid() != self._owner_pid or self._db is None:
            raise RuntimeError("The inventory connection is unavailable in this process.")
        return getattr(self._db, name)

    def __enter__(self):
        self._db.__enter__()
        return self

    def __exit__(self, *exc):
        return self._db.__exit__(*exc)

    def close(self):
        self._closed = True
        if self._db is not None:
            self._db.close()
            self._db = None

    def suspend_for_fork(self):
        if self._closed or os.getpid() != self._owner_pid or self._db is None:
            raise RuntimeError("Cannot suspend an unavailable inventory connection.")
        if self._db.in_transaction:
            raise RuntimeError("Finish the inventory transaction before starting CPU workers.")
        self._db.close()
        self._db = None
        # SQLite may defer the actual close when a cursor or a second connection
        # still owns the file. Linux Colab exposes inherited descriptors in /proc.
        descriptors = Path("/proc/self/fd")
        if descriptors.is_dir():
            for descriptor in descriptors.iterdir():
                try:
                    st = descriptor.stat()
                except FileNotFoundError:
                    continue
                if (st.st_dev, st.st_ino) == self._file_identity:
                    raise RuntimeError("Close outstanding inventory cursors or extra inventory connections before forking.")

    def reopen_after_fork(self):
        if os.getpid() != self._owner_pid or self._closed:
            raise RuntimeError("Only the active parent process can reopen the inventory.")
        if self._db is not None:
            return
        st = self.path.stat()
        if (st.st_dev, st.st_ino) != self._file_identity:
            raise RuntimeError("The inventory file was replaced while starting CPU workers.")
        # mode=rw prevents SQLite from creating an empty file if the database disappeared.
        db = sqlite3.connect(self.path.as_uri() + "?mode=rw", uri=True, timeout=30)
        try:
            db.execute("PRAGMA foreign_keys = ON")
            db.execute("PRAGMA synchronous = FULL")
            row = db.execute("SELECT build_id FROM inventory_state WHERE singleton=1").fetchone()
            if row is None or row[0] != self.build_id:
                raise RuntimeError("The reopened inventory belongs to a different build.")
        except BaseException:
            db.close()
            raise
        self._db = db


def open_inventory(shared, build_id):
    directory = Path(shared)
    directory.mkdir(parents=True, exist_ok=True)
    db = sqlite3.connect(directory / "inventory.sqlite3", timeout=30)
    try:
        db.execute("PRAGMA foreign_keys = ON")
        db.execute("PRAGMA synchronous = FULL")
        db.executescript("""
            CREATE TABLE IF NOT EXISTS inventory_state(
                singleton INTEGER PRIMARY KEY CHECK (singleton = 1),
                build_id TEXT NOT NULL, recipes_done INTEGER NOT NULL DEFAULT 0,
                expected_recipes INTEGER, complete INTEGER NOT NULL DEFAULT 0);
            CREATE TABLE IF NOT EXISTS inventory_recipes(
                ordinal INTEGER PRIMARY KEY, recipe_id TEXT NOT NULL UNIQUE,
                variant TEXT NOT NULL, common_evidence BLOB);
            CREATE TABLE IF NOT EXISTS inventory_instances(
                instance_row INTEGER PRIMARY KEY, instance_id TEXT NOT NULL UNIQUE, task TEXT NOT NULL, profile TEXT NOT NULL,
                recipe_ordinal INTEGER NOT NULL REFERENCES inventory_recipes(ordinal),
                frozen BLOB NOT NULL);
            CREATE INDEX IF NOT EXISTS by_inventory_task ON inventory_instances(task);
            CREATE TABLE IF NOT EXISTS inventory_memberships(
                instance_row INTEGER NOT NULL REFERENCES inventory_instances(instance_row), evidence_state TEXT NOT NULL,
                PRIMARY KEY(instance_row, evidence_state)) WITHOUT ROWID;
            CREATE TABLE IF NOT EXISTS inventory_holds(
                id INTEGER PRIMARY KEY, recipe_ordinal INTEGER NOT NULL REFERENCES inventory_recipes(ordinal),
                record TEXT NOT NULL, UNIQUE(recipe_ordinal,record));
            CREATE TABLE IF NOT EXISTS inventory_counts(
                reason TEXT PRIMARY KEY, amount INTEGER NOT NULL) WITHOUT ROWID;
        """)
        with db:
            db.execute("INSERT OR IGNORE INTO inventory_state(singleton,build_id) VALUES (1,?)", (build_id,))
            stored = db.execute("SELECT build_id FROM inventory_state WHERE singleton=1").fetchone()[0]
            if build_id != stored:
                raise ValueError("Inventory source or engine build differs from the current run.")
    except BaseException:
        db.close()
        raise
    return _InventoryHandle(directory / "inventory.sqlite3", build_id, db)


def _pack_inventory(value):
    return gzip.compress(_wire_bytes(value), compresslevel=1, mtime=0)


def _unpack_inventory(blob):
    return json.loads(gzip.decompress(blob))


def load_inventory_instance(db, instance_id):
    """Restore the exact evidence, preflighted case and first applicable matrix row without rebinding."""
    record = db.execute("""
        SELECT i.frozen, r.common_evidence, r.recipe_id, i.task FROM inventory_instances AS i
        JOIN inventory_recipes AS r ON r.ordinal=i.recipe_ordinal WHERE i.instance_id=?
    """, (instance_id,)).fetchone()
    if record is None:
        raise KeyError("Unknown inventory instance: " + instance_id)
    frozen, common, recipe_id, task = record
    case, distinct, matrix_row = _unpack_inventory(frozen)
    raw = {**_unpack_inventory(common), **distinct}
    if (case["semantic_instance_id"] != instance_id or case["task_id"] != task or raw["recipe_id"] != recipe_id
            or case["evidence_snapshot_id"] != wire_digest(raw)):
        raise ValueError("Stored census instance differs from its frozen evidence.")
    return case, raw, matrix_row


def _prepare_inventory_recipe(recipe, evaluate, source, contracts):
    """Evaluate and compress one recipe without database access."""
    recipe_id, variant = recipe["example_id"], recipe["variant"]
    stats = Counter()
    holds, instances = {}, {}
    saved_common, common_blob = None, None

    def hold(event):
        holds[_wire_bytes(event).decode("utf-8")] = None

    for task, raw in evaluate(stats, hold):
        if raw["recipe_id"] != recipe_id or raw["variant"] != variant:
            raise ValueError("Eligibility iterator returned a different recipe.")
        try:
            cell = _matrix_row(source["selected_cells"], task, raw)
            if cell["rule_id"] != contracts[task]["semantics"]["source_binding"]["rule_id"]:
                raise ValueError("S1 matrix rule differs from its task contract.")
            if not raw.get("s1_bindings") or not raw.get("respondent_presentation"):
                raise ValueError("Real inventory requires both S1 bindings and a direct S8 reference.")
            case = build_case(task, raw, contracts)
        except EvidenceHold as exc:
            hold({"recipe": recipe_id, "variant": variant, "task": task, "profile": raw["profile_id"],
                  "scope": raw["scope_unit_ids"], "evidence_state": raw["evidence_state"],
                  "query_class_id": raw.get("query_class_id"), "reference_unit_id": raw.get("reference_unit_id"),
                  "direction": raw.get("direction"), "reason": exc.reason, "disposition": exc.disposition,
                  "stage": "before_generation"})
            stats["preflight_" + exc.disposition] += 1
            continue
        identity = case["semantic_instance_id"]
        previous = instances.get(identity)
        if previous is None:
            if saved_common is None:
                common_keys = ("recipe_id", "variant", "units", "groups_by_class", "respondent_presentation")
                saved_common = {key: raw[key] for key in common_keys}
                common_blob = _pack_inventory(saved_common)
            if any(raw[key] is not value and raw[key] != value for key, value in saved_common.items()):
                raise ValueError("The common evidence changed within an S8 recipe.")
            distinct = {key: value for key, value in raw.items() if key not in saved_common}
            states = {}
            instances[identity] = (task, case["profile_id"], _pack_inventory([case, distinct, cell]), states)
            stats["unique_eligible_instances"] += 1
        else:
            old_task, old_profile, _, states = previous
            if (old_task, old_profile) != (task, case["profile_id"]):
                raise ValueError("Duplicate semantic ID disagrees with the original instance.")
            stats["overlapping_policy_rows_skipped"] += 1
        states[cell["evidence_state"]] = None
    return recipe_id, variant, common_blob, holds, instances, stats


def _write_inventory_recipe(db, ordinal, prepared, totals):
    recipe_id, variant, common_blob, holds, instances, stats = prepared
    db.execute("INSERT INTO inventory_recipes(ordinal,recipe_id,variant,common_evidence) VALUES (?,?,?,?)",
               (ordinal, recipe_id, variant, common_blob))
    db.executemany("INSERT INTO inventory_holds(recipe_ordinal,record) VALUES (?,?)",
                   ((ordinal, record) for record in holds))
    for identity, (task, profile, blob, states) in instances.items():
        # The UNIQUE constraint also detects a cross-recipe identity conflict.
        cur = db.execute("INSERT INTO inventory_instances(instance_id,task,profile,recipe_ordinal,frozen) "
                         "VALUES (?,?,?,?,?)", (identity, task, profile, ordinal, blob))
        rowid = cur.lastrowid
        db.executemany("INSERT INTO inventory_memberships(instance_row,evidence_state) VALUES (?,?)",
                       ((rowid, state) for state in states))
    totals.update(stats)


def _inventory_recipe(db, ordinal, recipe, evaluate, source, contracts, stats):
    """Legacy single-recipe entry point used for individual inventory operations."""
    _write_inventory_recipe(db, ordinal, _prepare_inventory_recipe(recipe, evaluate, source, contracts), stats)


def _census_cpu_count():
    """Use the current process's affinity and any visible Linux CPU quota."""
    try:
        available = len(os.sched_getaffinity(0))
    except (AttributeError, OSError):
        available = os.cpu_count() or 1
    # Colab and other container runtimes expose either cgroup v2 or cgroup v1.
    roots = [Path("/sys/fs/cgroup"), Path("/sys/fs/cgroup/cpu"), Path("/sys/fs/cgroup/cpu,cpuacct")]
    try:
        memberships = Path("/proc/self/cgroup").read_text().splitlines()
    except OSError:
        memberships = []
    paths = set(roots)
    for line in memberships:
        _, controllers, rel = line.split(":", 2)
        if controllers and "cpu" not in controllers.split(","):
            continue
        suffix = PurePosixPath(rel)
        if ".." in suffix.parts:
            continue
        for root in roots:
            for part in (suffix, *suffix.parents):
                paths.add(root / str(part).lstrip("/"))
    for group in paths:
        try:
            quota, period = (group / "cpu.max").read_text().split()
        except (OSError, ValueError):
            try:
                quota = (group / "cpu.cfs_quota_us").read_text().strip()
                period = (group / "cpu.cfs_period_us").read_text().strip()
            except OSError:
                continue
        if quota.lstrip("-").isdigit() and period.isdigit() and int(quota) > 0 and int(period) > 0:
            available = min(available, math.ceil(int(quota) / int(period)))
    return max(1, available)


def resolve_inventory_workers(workers):
    if type(workers) is not int or workers < 0:
        raise ValueError("inventory_workers must be 0 (automatic) or a positive integer")
    if not hasattr(os, "uname") or os.uname().sysname != "Linux" or "fork" not in multiprocessing.get_all_start_methods():
        return 1
    available = _census_cpu_count()
    return min(workers or available, available)


def _inventory_process(channel, window, source, contracts):
    """Workers inherit one read-only recipe window and exchange bounded results through a pipe."""
    try:
        while True:
            job = channel.recv()
            if job is None:
                return
            start, stop = job
            prepared = []
            for offset in range(start, stop):
                recipe, evaluate = window[offset]
                try:
                    prepared.append(_prepare_inventory_recipe(recipe, evaluate, source, contracts))
                except BaseException as exc:
                    channel.send(("error", offset, type(exc).__name__, str(exc)))
                    return
            channel.send(("ok", start, prepared))
    finally:
        channel.close()


def _process_window(window, source, contracts, workers, first_ordinal, *, inventory=None):
    """Emit results in source order with up to two outstanding chunks per worker."""
    length = len(window)
    chunk_size = min(32, max(1, math.ceil(length / (8 * workers))))
    count = min(workers, math.ceil(length / chunk_size))
    if count == 1:
        for recipe, evaluate in window:
            yield _prepare_inventory_recipe(recipe, evaluate, source, contracts)
        return

    ctx = multiprocessing.get_context("fork")
    pool, active, ready = {}, {}, {}
    submitted, emitted = 0, 0
    clean_exit = False
    try:
        if inventory is not None and not isinstance(inventory, _InventoryHandle):
            raise TypeError("Parallel inventory requires the handle returned by open_inventory().")
        try:
            if inventory is not None:
                inventory.suspend_for_fork()
            for index in range(count):
                parent, child = ctx.Pipe()
                proc = ctx.Process(target=_inventory_process, args=(child, window, source, contracts),
                                   name=f"s8-census-{index + 1}")
                pool[parent] = proc
                try:
                    # Defer notebook interrupts until Process.start has recorded the PID.
                    # The child leaves SIGINT handling to the parent and accepts SIGTERM.
                    mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGINT})
                    try:
                        proc.start()
                    finally:
                        signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                finally:
                    child.close()
        except BaseException as exc:
            if inventory is not None:
                try:
                    inventory.reopen_after_fork()
                except Exception as reopen_error:
                    if hasattr(exc, "add_note"):
                        exc.add_note(f"Could not reopen the parent inventory: {reopen_error}")
            raise
        if inventory is not None:
            inventory.reopen_after_fork()

        while emitted < length:
            for pipe in pool:
                if submitted == length or len(active) + len(ready) >= 2 * count:
                    break
                if pipe not in active:
                    end = min(length, submitted + chunk_size)
                    pipe.send((submitted, end))
                    active[pipe] = (submitted, end)
                    submitted = end

            if emitted in ready:
                records = ready.pop(emitted)
                emitted += len(records)
                yield from records
                continue
            # A killed worker can leave its pipe open in a sibling. Its sentinel
            # is authoritative, and prevents waiting forever for a missing result.
            events = wait_for_workers([*active, *(p.sentinel for p in pool.values())])
            for pipe in list(active):
                if pipe not in events:
                    continue
                try:
                    message = pipe.recv()
                except (EOFError, OSError) as exc:
                    raise RuntimeError(f"S8 worker exited during recipe {first_ordinal + active[pipe][0]}") from exc
                if message[0] == "error":
                    _, offset, error_type, detail = message
                    raise RuntimeError(f"S8 worker failed at recipe {first_ordinal + offset}: {error_type}: {detail}")
                start, end = active.pop(pipe)
                kind, offset, records = message
                if kind != "ok" or offset != start or len(records) != end - start:
                    raise RuntimeError("Incomplete or misordered S8 worker response")
                ready[start] = records
            for pipe, proc in pool.items():
                if proc.sentinel in events:
                    raise RuntimeError(f"S8 worker {proc.name} exited before its window completed")
        clean_exit = True
    finally:
        for pipe, proc in pool.items():
            if clean_exit and proc.is_alive():
                try:
                    pipe.send(None)
                except (OSError, EOFError):
                    pass
            elif proc.is_alive():
                proc.terminate()
            pipe.close()
        for proc in pool.values():
            if proc.pid is not None:
                proc.join(timeout=3)
                if proc.is_alive():
                    proc.kill()
                    proc.join()
            proc.close()


def inventory_census(db, source, s8_dir, contracts, *, checkpoint_recipes=100, on_checkpoint=None,
                     corpus=None, workers=1, window_size=5000):
    """Compute ordered windows in parallel. Only the parent commits completed recipe prefixes."""
    if type(checkpoint_recipes) is not int or checkpoint_recipes < 1:
        raise ValueError("checkpoint_recipes must be a positive integer")
    if type(window_size) is not int or window_size < 1:
        raise ValueError("inventory_window_size must be a positive integer")
    worker_count = resolve_inventory_workers(workers)
    if db.execute("SELECT complete FROM inventory_state WHERE singleton=1").fetchone()[0]:
        return
    expected, stream = _verified_recipe_stream(source, s8_dir, set(contracts), corpus=corpus)
    if type(expected) is not int or expected < 0:
        raise ValueError("The verified S8 recipe count is missing.")
    with db:
        db.execute("UPDATE inventory_state SET expected_recipes=? WHERE singleton=1 AND expected_recipes IS NULL",
                   (expected,))
        resume, saved_expected = db.execute(
            "SELECT recipes_done, expected_recipes FROM inventory_state WHERE singleton=1").fetchone()
        if saved_expected != expected or resume > expected:
            raise ValueError("Verified recipe count differs from the saved census.")
    # Decode the compressed prefix on resume without redoing the clinical evaluations.
    reader = itertools.islice(stream, resume, None)
    pos = resume
    print(f"S8 inventory: {pos:,}/{expected:,} recipes committed; windows of {window_size:,}, "
          f"up to {worker_count} CPU workers, checkpoints every {checkpoint_recipes:,}.", flush=True)
    while pos < expected:
        first = pos
        window = list(itertools.islice(reader, min(window_size, expected - pos)))
        if len(window) != min(window_size, expected - pos):
            raise ValueError("Verified recipe stream ended before its declared count.")
        end = first + len(window)
        results = _process_window(window, source, contracts, worker_count, first, inventory=db)
        try:
            while pos < end:
                size = min(checkpoint_recipes, end - pos)
                batch = list(itertools.islice(results, size))
                if len(batch) != size:
                    raise RuntimeError("S8 worker results ended before the checkpoint.")
                stats = Counter()
                with db:
                    db.execute("BEGIN IMMEDIATE")
                    stored_pos = db.execute("SELECT recipes_done FROM inventory_state WHERE singleton=1").fetchone()[0]
                    if stored_pos != pos:
                        raise RuntimeError("Another writer advanced this census. Rerun to use its checkpoint.")
                    for ordinal, prepared in enumerate(batch, pos):
                        _write_inventory_recipe(db, ordinal, prepared, stats)
                    db.executemany("""INSERT INTO inventory_counts VALUES (?,?)
                                      ON CONFLICT(reason) DO UPDATE SET amount=amount+excluded.amount""", stats.items())
                    db.execute("UPDATE inventory_state SET recipes_done=? WHERE singleton=1", (pos + size,))
                pos += size
                if on_checkpoint is not None:
                    on_checkpoint(pos, expected)
                if pos == end or (pos - resume) % (10 * checkpoint_recipes) == 0:
                    print(f"S8 inventory: {pos:,}/{expected:,} recipes committed.", flush=True)
            if next(results, None) is not None:
                raise RuntimeError("Extra S8 worker results after the window.")
        finally:
            results.close()
        del window
    if next(reader, None) is not None:
        raise ValueError("Verified recipe stream exceeded its declared count.")
    with db:
        if db.execute("SELECT COUNT(*) FROM inventory_recipes").fetchone()[0] != expected:
            raise ValueError("Inventory recipe coverage does not match S8.")
        db.execute("UPDATE inventory_state SET complete=1 WHERE singleton=1")


def inventory_summary(db, tasks, requested_count):
    """Report planned accepted questions at full coverage; realized acceptance is measured independently."""
    if type(requested_count) is not int or requested_count < 1:
        raise ValueError("requested_count must be a positive integer")
    build_id, done, expected, complete = db.execute(
        "SELECT build_id, recipes_done, expected_recipes, complete FROM inventory_state WHERE singleton=1"
    ).fetchone()
    counts = dict(db.execute("SELECT task,COUNT(*) FROM inventory_instances GROUP BY task"))
    per_task = {t: {"eligible_instances": counts.get(t, 0),
                    "target_accepted_questions": counts.get(t, 0) * requested_count if complete else None}
                for t in sorted(tasks)}
    eligible = sum(row["eligible_instances"] for row in per_task.values())
    return {"inventory_build_id": build_id, "expected_recipes": expected, "recipes_examined": done,
            "complete": bool(complete), "eligible_instances": eligible, "per_task": per_task,
            "requested_questions_per_instance": requested_count,
            "target_accepted_questions": eligible * requested_count if complete else None,
            "enumeration": dict(db.execute("SELECT reason,amount FROM inventory_counts ORDER BY reason"))}


def write_inventory_holds(db, path):
    """Materialize durable pre-generation holds as JSONL without loading the inventory into memory."""
    with Path(path).open("w", encoding="utf-8") as stream:
        for (record,) in db.execute("SELECT record FROM inventory_holds ORDER BY id"):
            stream.write(record + "\n")


# The working SQLite database lives on the VM. Drive stores immutable segments as ordinary files.
class _InvalidInventory(ValueError):
    pass


def _inventory_state(db, build_id, full=True):
    if full:
        integrity = db.execute("PRAGMA integrity_check").fetchone()
        if not integrity or integrity[0] != "ok" or db.execute("PRAGMA foreign_key_check").fetchone():
            raise _InvalidInventory("Inventory page or foreign-key integrity failed")
        tables = {r[0] for r in db.execute("SELECT name FROM sqlite_master WHERE type='table'")}
        if not set(_SEG_COLUMNS).union({"inventory_state"}) <= tables:
            raise _InvalidInventory("Missing inventory table")
        for table, columns in _SEG_COLUMNS.items():
            if tuple(row[1] for row in db.execute(f"PRAGMA table_info({table})")) != tuple(columns.split(",")):
                raise _InvalidInventory("Unexpected inventory columns: " + table)
    state = db.execute("SELECT build_id,recipes_done,expected_recipes,complete FROM inventory_state "
                       "WHERE singleton=1").fetchone()
    if not state or state[0] != build_id:
        raise _InvalidInventory("Missing inventory state or incorrect build ID")
    _, done, expected, complete = state
    if (type(done) is not int or done < 0 or complete not in (0, 1)
            or (expected is None and (done != 0 or complete))
            or (expected is not None and (type(expected) is not int or expected < done))
            or (complete and expected != done)):
        raise _InvalidInventory("Inconsistent inventory progress")
    if full:
        count, low, high = db.execute("SELECT COUNT(*),MIN(ordinal),MAX(ordinal) FROM inventory_recipes").fetchone()
        if count != done or (count and (low != 0 or high != done - 1)):
            raise _InvalidInventory("Inventory rows do not form the committed recipe prefix")
    return done, expected, bool(complete)


_SEG_COLUMNS = {
    "inventory_recipes": "ordinal,recipe_id,variant,common_evidence",
    "inventory_instances": "instance_row,instance_id,task,profile,recipe_ordinal,frozen",
    "inventory_memberships": "instance_row,evidence_state",
    "inventory_holds": "id,recipe_ordinal,record",
    "inventory_counts": "reason,amount",
}
def _local_inventory_volume(path):
    # Linux mountinfo recognizes alternate mount aliases as well as Colab's default Drive path.
    path = Path(path).expanduser().resolve()
    if path.is_relative_to("/content/drive"):
        raise ValueError("The working SQLite directory must be on local VM storage, e.g. /content/mmvqa_phase3_sector2.")
    try:
        records = Path("/proc/self/mountinfo").read_text().splitlines()
    except FileNotFoundError:
        return path
    best, fs = -1, ""
    for line in records:
        parts, sep, kind = line.partition(" - ")
        if not sep:
            continue
        mount = Path(re.sub(r"\\([0-7]{3})", lambda m: chr(int(m[1], 8)), parts.split()[4]))
        if path.is_relative_to(mount) and len(mount.parts) > best:
            best, fs = len(mount.parts), kind.split()[0].lower()
    if fs.startswith(("fuse", "nfs", "cifs", "smb", "9p", "ceph", "lustre", "davfs")):
        raise ValueError(f"The live SQLite database needs a local VM filesystem; {path} uses {fs}.")
    return path


_SNAPSHOT_FORMAT = "mmvqa-s2-inventory-snapshot-v1"


def _snapshot_stat(stat):
    return [stat.st_dev, stat.st_ino, stat.st_size, stat.st_mtime_ns, stat.st_ctime_ns]


def _snapshot_bytes(path, destination=None):
    """Hash the exact bytes from one open file, optionally copying them on the same pass."""
    digest = hashlib.sha256()
    with path.open("rb") as src:
        before = _snapshot_stat(os.fstat(src.fileno()))
        # Full SQLite snapshots can span gigabytes. Keep byte progress on the hashing pass itself.
        progress = (tqdm(total=before[2], desc="Verifying SQLite bytes", unit="B", unit_scale=True,
                         unit_divisor=1024, leave=False) if before[2] >= 64 * 1024 * 1024 else None)
        try:
            for block in iter(lambda: src.read(4 * 1024 * 1024), b""):
                digest.update(block)
                if destination is not None:
                    destination.write(block)
                if progress is not None:
                    progress.update(len(block))
        finally:
            if progress is not None:
                progress.close()
        if before != _snapshot_stat(os.fstat(src.fileno())) or before != _snapshot_stat(path.stat()):
            raise RuntimeError(f"File changed while being verified: {path}. Reopen the inventory.")
    return digest.hexdigest(), before


def _snapshot_doc(raw, inventory_id, material):
    doc = _s1_json(raw)
    if not isinstance(doc, dict) or set(doc) != {
        "format", "inventory_id", "material", "database_sha256", "database_size_bytes",
        "database_file", "recipes_done", "expected_recipes", "complete"
    }:
        raise _InvalidInventory("Incomplete or unknown inventory manifest schema")
    digest = doc["database_sha256"]
    if (doc["format"] != _SNAPSHOT_FORMAT or doc["inventory_id"] != inventory_id
            or wire_digest(doc["material"]) != wire_digest(material)):
        raise _InvalidInventory("Inventory format, inputs or identity changed")
    if (not isinstance(digest, str) or not re.fullmatch(r"[0-9a-f]{64}", digest)
            or doc["database_file"] != "snapshots/" + digest + ".sqlite3"):
        raise _InvalidInventory("Invalid content-addressed snapshot reference")
    size, done, total, complete = (doc[k] for k in ("database_size_bytes", "recipes_done", "expected_recipes", "complete"))
    if (type(size) is not int or size <= 0 or type(done) is not int or type(total) is not int
            or not 0 <= done <= total or type(complete) is not bool or (complete and done != total)
            or (not complete and done == 0)):
        raise _InvalidInventory("Invalid snapshot size or progress")
    return doc


def _snapshot_state(path, inventory_id):
    db = sqlite3.connect(path.resolve().as_uri() + "?mode=ro&immutable=1", uri=True)
    try:
        columns = tuple(row[1] for row in db.execute("PRAGMA table_info(inventory_state)"))
        if columns != ("singleton", "build_id", "recipes_done", "expected_recipes", "complete"):
            raise _InvalidInventory("Invalid SQLite inventory state schema")
        return _inventory_state(db, inventory_id)
    except sqlite3.DatabaseError as exc:
        if getattr(exc, "sqlite_errorcode", 0) & 255 in (sqlite3.SQLITE_CORRUPT, sqlite3.SQLITE_NOTADB):
            raise _InvalidInventory("Invalid SQLite image: " + str(exc)) from exc
        raise
    finally:
        db.close()


class SnapshotInventory:
    """One census writer. SQLite works on the VM; the durable pointer references a sealed single-file copy."""

    def __init__(self, local_dir, shared_inventory_dir, inventory_id, material, *, snapshot_interval=5000):
        if not isinstance(inventory_id, str) or wire_digest(material) != inventory_id:
            raise ValueError("Inventory ID must be derived from its current input material")
        if type(snapshot_interval) is not int or snapshot_interval < 1:
            raise ValueError("snapshot_interval must be a positive integer")
        self.local = _local_inventory_volume(local_dir)
        self.root = Path(shared_inventory_dir).expanduser().resolve()
        if self.root.is_relative_to(self.local) or self.local.is_relative_to(self.root):
            raise ValueError("Working and published inventory directories must be separate")
        self.manifest = self.root / "inventory.json"
        self.snapshots = self.root / "snapshots"
        self.path = self.local / "inventory.sqlite3"
        self.sealed = self.local / "verified_snapshot.sqlite3"
        self.id = inventory_id
        self.material = material
        self.interval = snapshot_interval
        self.db = self.lock = self.saved = self.token = None
        self._reset_admission()

    def _reset_admission(self):
        self.saved = self.token = None
        self.end, self.complete = 0, False
        self.report = dict(mode="rebuilding", reason=None, manifest_path=str(self.manifest),
                           working_directory=str(self.local), snapshot_sha256=None, snapshot_path=None,
                           snapshot_stat=None, verified_local_path=None, verified_local_stat=None,
                           durable_recipes=0, durable_complete=False)

    def _pointer_bytes(self):
        if self.manifest.is_symlink():
            raise RuntimeError("The inventory manifest path must be a regular managed file")
        try:
            return self.manifest.read_bytes()
        except FileNotFoundError:
            return None

    def _check_pointer(self):
        if self._pointer_bytes() != self.token:
            raise RuntimeError("Another writer changed inventory.json. Reopen this inventory before continuing.")

    def _remember(self, doc, remote_stat):
        self.saved = doc
        self.end, self.complete = doc["recipes_done"], doc["complete"]
        self.report.update(snapshot_sha256=doc["database_sha256"], snapshot_path=str(self.root / doc["database_file"]),
                           snapshot_stat=remote_stat, verified_local_path=str(self.sealed),
                           verified_local_stat=_snapshot_stat(self.sealed.stat()),
                           durable_recipes=self.end, durable_complete=self.complete)

    def _restore(self, staging):
        if self.token is None:
            raise _InvalidInventory("Missing inventory.json")
        doc = _snapshot_doc(self.token, self.id, self.material)
        path = self.root / doc["database_file"]
        if path.is_symlink():
            raise _InvalidInventory("Symlink at the snapshot path")
        # Only missing source bytes are a cache miss. Local write failures and Drive I/O errors propagate.
        try:
            st = path.stat()
        except FileNotFoundError as exc:
            raise _InvalidInventory("Manifest-referenced database file is missing") from exc
        if not path.is_file() or st.st_size != doc["database_size_bytes"]:
            raise _InvalidInventory("Manifest-referenced database is not a file of the declared size")
        with staging.open("xb") as out:
            digest, stat = _snapshot_bytes(path, out)
        if digest != doc["database_sha256"] or stat[2] != doc["database_size_bytes"]:
            raise _InvalidInventory("Drive database SHA-256 or byte size differs from inventory.json")
        if _snapshot_state(staging, self.id) != (doc["recipes_done"], doc["expected_recipes"], doc["complete"]):
            raise _InvalidInventory("SQLite progress differs from inventory.json")
        self._check_pointer()
        os.replace(staging, self.sealed)
        self._remember(doc, stat)

    def __enter__(self):
        if self.lock is not None:
            raise RuntimeError("This inventory is already in use")
        self._reset_admission()
        self.local.mkdir(parents=True, exist_ok=True)
        self.root.mkdir(parents=True, exist_ok=True)
        if self.snapshots.is_symlink():
            raise ValueError("The managed snapshots directory cannot be a symlink")
        self.snapshots.mkdir(exist_ok=True)
        self.lock = (self.local / "inventory.lock").open("a+b")
        try:
            if fcntl is not None:
                fcntl.flock(self.lock.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
            else:
                self.lock.seek(0)
                msvcrt.locking(self.lock.fileno(), msvcrt.LK_NBLCK, 1)
            self.token = self._pointer_bytes()
            with tempfile.TemporaryDirectory(dir=self.local) as tmp:
                staging = Path(tmp) / "image.sqlite3"
                try:
                    self._restore(staging)
                except (ValueError, UnicodeError) as exc:
                    self.report["reason"] = str(exc)
                # A stale local database (including its WAL or journal) is never a recovery source.
                for suffix in ("-shm", "-wal", "-journal", ""):
                    self.path.with_name(self.path.name + suffix).unlink(missing_ok=True)
                if self.saved is not None:
                    shutil.copyfile(self.sealed, self.path)
                    self.report.update(mode="restored_verified_snapshot", reason="manifest_and_database_verified")
                else:
                    self.sealed.unlink(missing_ok=True)
                self._check_pointer()
                self.db = open_inventory(self.local, self.id)
            return self
        except BaseException:
            self.close()
            raise

    def checkpoint(self, done, expected):
        if done - self.end >= self.interval and done != expected:
            self.publish()

    def publish(self):
        if self.db.in_transaction:
            raise RuntimeError("Commit inventory rows before publishing a snapshot")
        done, total, complete = _inventory_state(self.db, self.id, full=False)
        if self.saved and (done, total, complete) == (self.end, self.saved["expected_recipes"], self.complete):
            return
        if not complete and done == 0:
            return
        self._check_pointer()
        old = self.saved
        with tempfile.TemporaryDirectory(dir=self.local) as tmp:
            staging = Path(tmp) / "snapshot.sqlite3"
            copy = sqlite3.connect(staging)
            try:
                self.db.backup(copy)
                if copy.execute("PRAGMA journal_mode=DELETE").fetchone() != ("delete",):
                    raise RuntimeError("Could not seal the snapshot as a standalone SQLite file")
            finally:
                copy.close()
            if _snapshot_state(staging, self.id) != (done, total, complete):
                raise RuntimeError("Sealed snapshot differs from committed progress")
            unique = uuid.uuid4().hex
            upload = self.snapshots / ("." + unique + ".tmp")
            pointer = self.root / (".inventory-" + unique + ".tmp")
            try:
                with upload.open("xb") as out:
                    digest, local_stat = _snapshot_bytes(staging, out)
                    out.flush()
                    os.fsync(out.fileno())
                checked, remote_stat = _snapshot_bytes(upload)
                if checked != digest or remote_stat[2] != local_stat[2]:
                    raise IOError("Drive snapshot readback failed; inventory.json was not advanced")
                target = self.snapshots / (digest + ".sqlite3")
                self._check_pointer()
                os.replace(upload, target)
                stat = _snapshot_stat(target.stat())
                # Rename can change ctime. In filesystems that changed other identity fields, re-read the final name.
                if stat[:4] != remote_stat[:4]:
                    checked, stat = _snapshot_bytes(target)
                    if checked != digest or stat[2] != local_stat[2]:
                        raise IOError("Final Drive snapshot failed SHA-256 verification")
                doc = dict(format=_SNAPSHOT_FORMAT, inventory_id=self.id, material=self.material,
                           database_file="snapshots/" + target.name, database_sha256=digest,
                           database_size_bytes=local_stat[2], recipes_done=done, expected_recipes=total, complete=complete)
                raw = _wire_bytes(doc) + b"\n"
                with pointer.open("xb") as out:
                    out.write(raw)
                    out.flush()
                    os.fsync(out.fileno())
                self._check_pointer()
                os.replace(pointer, self.manifest)
                if self._pointer_bytes() != raw:
                    raise IOError("Published inventory pointer failed readback")
                self.token = raw
                os.replace(staging, self.sealed)
                self._remember(doc, stat)
            finally:
                upload.unlink(missing_ok=True)
                pointer.unlink(missing_ok=True)
        if old and not old["complete"] and old["database_sha256"] != digest:
            self._check_pointer()
            (self.root / old["database_file"]).unlink(missing_ok=True)

    def close(self):
        try:
            if self.db is not None:
                self.db.close()
        finally:
            self.db = None
            if self.lock is not None:
                self.lock.close()
                self.lock = None

    def __exit__(self, *exc):
        self.close()


# Include these definitions in the inventory component. Their inputs are the source
# bodies captured by the component loader and the same receipts used by this run.
INVENTORY_SOURCE_NAMES = ("sources", "real", "evidence", "workflow", "inventory")

# Entry points that construct, read or encode the census, including the two
# dynamically supplied S8 reader/binding entry points and storage schema.
_INVENTORY_CODE_ROOTS = (
    "load_verified_sector1", "_open_s8_metadata", "_clinical_s8_corpus",
    "inventory_census", "open_inventory", "load_inventory_instance",
    "V13_TASKS", "_SEG_COLUMNS",
)
_INVENTORY_ENGINE_IMPORTS = frozenset("""
    Callable CellType Counter Draft202012Validator ExitStack FunctionType IncompleteRead
    ModuleType Path PurePosixPath Request SimpleQueue ThreadPoolExecutor ast copy
    defaultdict errno gzip hashlib inspect io itertools json lru_cache math multiprocessing
    os pq quote re shutil signal sqlite3 tempfile tqdm unicodedata urlopen urlparse
    wait_for_workers
""".split())


def _inventory_producer_hashes(build, component_sources):
    """Fingerprint the transitive producer definitions from the recorded component bytes."""
    declarations = {}
    for owner in INVENTORY_SOURCE_NAMES:
        source_text = component_sources[owner]
        if hashlib.sha256(source_text.encode("utf-8")).hexdigest() != build["runtime_components"][owner]:
            raise ValueError("Component source differs from the recorded build: " + owner)
        for node in ast.parse(source_text, filename=owner).body:
            if isinstance(node, (ast.FunctionDef, ast.ClassDef)):
                name = node.name
            elif (isinstance(node, ast.Assign) and len(node.targets) == 1
                  and isinstance(node.targets[0], ast.Name)):
                name = node.targets[0].id
            elif isinstance(node, ast.Expr) and isinstance(node.value, ast.Constant) and isinstance(node.value.value, str):
                continue
            else:
                raise ValueError("Unclassified top-level statement in inventory producer component: " + owner)
            if name in declarations:
                raise ValueError("Ambiguous producer declaration: " + name)
            declarations[name] = owner, node

    def referenced_globals(scope):
        # Follow references in nested classes, closures, comprehensions and defaults.
        for symbol in scope.get_symbols():
            if symbol.is_referenced() and symbol.is_global():
                yield symbol.get_name()
        for child in scope.get_children():
            yield from referenced_globals(child)

    pending = list(_INVENTORY_CODE_ROOTS)
    resolved = {}
    while pending:
        name = pending.pop()
        if name in resolved:
            continue
        owner, node = declarations[name]
        # Positions and comments have no effect on executable behavior.
        representation = ast.dump(node, annotate_fields=True, include_attributes=False)
        resolved[name] = {"component": owner, "sha256": hashlib.sha256(representation.encode()).hexdigest()}
        scope = symtable.symtable(ast.unparse(node), owner, "exec")
        for dep in referenced_globals(scope):
            if dep in declarations:
                pending.append(dep)
            elif dep in {"globals", "locals", "__import__", "eval"}:
                raise ValueError("Dynamic inventory dependency needs an explicit binding: " + name + "." + dep)
            elif dep == "exec" and name not in {"_s8_reader_env", "load_observation_binding"}:
                raise ValueError("Unclassified executable inventory input: " + name)
            elif dep not in _INVENTORY_ENGINE_IMPORTS and not hasattr(builtins, dep):
                raise ValueError("Unclassified inventory global: " + name + "." + dep)
    return resolved


def inventory_identity_material(build, component_sources, verified_digests):
    """Pure comparison for current or historical builds. Never execute historical source."""
    contracts = build["contract_hashes"]
    if set(contracts) != set(V13_CONTRACT_NAMES):
        raise ValueError("The seven V13 grammar and semantics pairs are required")
    def checked_digest(value):
        if not isinstance(value, str) or not re.fullmatch(r"[0-9a-f]{64}", value):
            raise ValueError("Missing or malformed inventory digest")
        return value

    if not all(verified_digests.get(k) == checked_digest(v) for k, v in contracts.items()):
        raise ValueError("Contract bytes differ from the verified S1 input receipt")
    release = {k: build["source"][k] for k in ("run_id", "data_commit", "files_manifest_sha256")}
    if verified_digests["files_manifest.json"] != checked_digest(release["files_manifest_sha256"]):
        raise ValueError("S1 files manifest differs from build")
    s8_snapshot = checked_digest(verified_digests["phase2_source_snapshot.json"])
    for name in ("coverage_contract.json", "task_contract_enriched.json"):
        checked_digest(verified_digests[name])
    reader = build["upstream_code"]["s8_reader"]
    binder = build["upstream_code"]["s1_binder"]
    if (verified_digests.get(reader["path"]) != checked_digest(reader["sha256"])
            or verified_digests.get("sector2_binding_validator.py") != checked_digest(binder["sha256"])):
        raise ValueError("S8 reader or clinical binder differs from verified bytes")
    # Selected release content and executable producer matter. Campaign policy,
    # authorization receipts and the current GitHub registry HEAD are recorded on
    # each run separately. The clinical S8 snapshot pins its release, catalog,
    # storage index and all consumed artifacts.
    control_only = {"latest.json", "publication_manifest.json", "drive_artifacts_manifest.json",
                    "contracts.v13.manifest.json", reader["path"]}
    s1_artifacts = {k: checked_digest(v) for k, v in verified_digests.items()
                    if k not in control_only and k not in contracts}
    if not isinstance(build["schema_engine"], str) or not build["schema_engine"]:
        raise ValueError("Missing parameter-schema validator version")
    return {
        "schema": "mmvqa-s2-inventory-identity-v1",
        "source": release,
        "s1_artifacts": s1_artifacts,
        "s8_snapshot_sha256": s8_snapshot,
        "contracts": dict(contracts),
        "effective_contracts_sha256": checked_digest(build["effective_contracts_sha256"]),
        "runtime_artifacts": {"s8_reader_sha256": reader["sha256"], "s1_binder_sha256": binder["sha256"]},
        # build_case uses jsonschema to admit profile parameters and the public request.
        "parameter_schema_engine": build["schema_engine"],
        "producer_definitions": _inventory_producer_hashes(build, component_sources),
    }


In [13]:
%%sector2_component generation_campaign
class GenerationStore:
    """One durable campaign, with independent checkpoints before and after each HTTP request."""
    def __init__(self, directory):
        self.root = Path(directory)
        self.root.mkdir(parents=True, exist_ok=True)
        # An exclusive connection to a separate lock database spans network I/O.
        # The data database remains readable and its transactions remain short.
        self.guard = sqlite3.connect(self.root / "writer.lock.sqlite3", timeout=0)
        try:
            self.guard.execute("BEGIN EXCLUSIVE")
        except sqlite3.OperationalError:
            self.guard.close()
            raise RuntimeError("Another writer is using this generation campaign") from None
        try:
            self.db = sqlite3.connect(self.root / "generation.sqlite3", timeout=30)
            self.db.row_factory = sqlite3.Row
            self.db.execute("PRAGMA synchronous=FULL")
            self.db.executescript("""
                CREATE TABLE IF NOT EXISTS audit_objects (
                    kind TEXT, digest TEXT, compressed BLOB NOT NULL, PRIMARY KEY(kind,digest));
                CREATE TABLE IF NOT EXISTS gen_jobs (
                    instance_id TEXT PRIMARY KEY, task TEXT NOT NULL, inventory_row INTEGER UNIQUE,
                    receipt TEXT NOT NULL, state TEXT NOT NULL DEFAULT 'ready', closed_run TEXT,
                    attempts INTEGER NOT NULL DEFAULT 0, accepted INTEGER NOT NULL DEFAULT 0);
                CREATE INDEX IF NOT EXISTS gen_jobs_queue ON gen_jobs(state,inventory_row);
                CREATE INDEX IF NOT EXISTS gen_jobs_run ON gen_jobs(closed_run);
                CREATE TABLE IF NOT EXISTS gen_attempts (
                    instance_id TEXT, number INTEGER, stage TEXT NOT NULL, run_id TEXT NOT NULL,
                    completed_run TEXT, envelope TEXT, audit_id TEXT, disposition TEXT, question_key TEXT,
                    PRIMARY KEY(instance_id,number));
                CREATE INDEX IF NOT EXISTS gen_attempts_dispatched ON gen_attempts(run_id,instance_id);
                CREATE INDEX IF NOT EXISTS gen_attempts_stage ON gen_attempts(stage);
                CREATE INDEX IF NOT EXISTS gen_attempts_run ON gen_attempts(completed_run,disposition);
                CREATE TABLE IF NOT EXISTS qa (
                    instance TEXT, normalized TEXT, close_run TEXT NOT NULL, body TEXT NOT NULL,
                    corpus_body TEXT, PRIMARY KEY(instance,normalized));
                CREATE INDEX IF NOT EXISTS qa_run ON qa(close_run);
                CREATE TABLE IF NOT EXISTS interpretation_reviews (
                    audit_id TEXT NOT NULL,
                    review_policy_id TEXT NOT NULL,
                    created_run TEXT NOT NULL,
                    payload_ref TEXT NOT NULL,
                    status TEXT NOT NULL DEFAULT 'pending'
                        CHECK(status IN ('pending','submitting','completed','failed','uncertain')),
                    reserved_run TEXT,
                    advisory_ref TEXT,
                    PRIMARY KEY (audit_id, review_policy_id)
                );
                CREATE INDEX IF NOT EXISTS review_queue
                    ON interpretation_reviews(review_policy_id, status);
                CREATE INDEX IF NOT EXISTS reviews_reserved
                    ON interpretation_reviews(reserved_run);
                CREATE TABLE IF NOT EXISTS gen_runs (id TEXT PRIMARY KEY, exports_ready INTEGER DEFAULT 0);
                CREATE TABLE IF NOT EXISTS gen_batches (
                    run_id TEXT, number INTEGER, body TEXT NOT NULL, PRIMARY KEY(run_id,number));
                CREATE TABLE IF NOT EXISTS view_offsets (
                    name TEXT PRIMARY KEY, last_row INTEGER NOT NULL, byte_count INTEGER NOT NULL);
            """)
        except BaseException:
            if hasattr(self, "db"):
                self.db.close()
            self.guard.close()
            raise

    def close(self):
        self.db.close()
        self.guard.close()

    def put(self, kind, value):
        data = _wire_bytes(value)
        digest = hashlib.sha256(data).hexdigest()
        self.db.execute("INSERT OR IGNORE INTO audit_objects VALUES (?,?,?)",
                        (kind, digest, gzip.compress(data, compresslevel=1, mtime=0)))
        return digest

    def get(self, kind, digest):
        row = self.db.execute("SELECT compressed FROM audit_objects WHERE kind=? AND digest=?", (kind, digest)).fetchone()
        if row is None:
            raise ValueError(f"Missing {kind} audit object: {digest}")
        data = gzip.decompress(row[0])
        if hashlib.sha256(data).hexdigest() != digest:
            raise ValueError("Generation audit digest mismatch")
        return _s1_json(data)

    def write_cumulative(self, for_corpus=False):
        """Append only committed rows; trim an incomplete append after a lost runtime."""
        name, field = ("authorized_corpus.jsonl", "corpus_body") if for_corpus else ("validated_drafts.jsonl", "body")
        path = self.root / name
        pos = self.db.execute("SELECT last_row,byte_count FROM view_offsets WHERE name=?", (name,)).fetchone()
        last_row, size = tuple(pos) if pos else (0, 0)
        mode = "r+b" if path.exists() else "w+b"
        with path.open(mode) as file:
            file.seek(0, 2)
            if file.tell() < size:
                last_row, size = 0, 0
            file.seek(size)
            file.truncate()
            for rid, body in self.db.execute(f"SELECT rowid,{field} FROM qa WHERE rowid>? ORDER BY rowid", (last_row,)):
                if body is not None:
                    file.write(body.encode("utf-8") + b"\n")
                last_row = rid
            file.flush()
            os.fsync(file.fileno())
            byte_count = file.tell()
        with self.db:
            self.db.execute("INSERT OR REPLACE INTO view_offsets VALUES (?,?,?)", (name,last_row,byte_count))
        return path


_REVIEW_POLICY = "mmvqa.advisory-public-reading.v1"
_REVIEWABLE_REASONS = {
    "speech_act_unresolved", "unverified_role_coverage",
    "full_meaning_not_established", "unverified_english_wording",
}


def _review_policy_id(config):
    return wire_digest({"policy": _REVIEW_POLICY, "model": config.get("reviewer_model", API_MODEL)})


def _review_eligible(trace, interpretation, case):
    if (trace.get("validation_verdict") != "review" or trace.get("disposition") != "review"
            or not interpretation or interpretation.get("conflicts") != []
            or interpretation.get("ambiguous_readings") != []):
        return False
    task = case["task_id"]
    scope_check = ("public_scope_unambiguous" if task == "finding_identification"
                   else "public_referents_unambiguous")
    role_check = ("ontology_lexicon_grounded" if task == "finding_identification"
                  else "public_roles_grounded")
    rows = trace["checks"]
    checks = {row["check_id"]: row for row in rows}
    names = ("question_json_shape", "question_semantic_form", role_check, scope_check,
             "paraphrase_equivalence", "question_language_quality", "answer_not_disclosed")
    if len(rows) != len(names) or set(checks) != {task + "." + name for name in names}:
        return False
    for name in ("question_json_shape", scope_check, "answer_not_disclosed"):
        if checks[task + "." + name]["status"] != "pass":
            return False
    review_rows = [row for row in rows if row["status"] == "review"]
    if (not review_rows or any(row["status"] not in {"pass", "review"} for row in rows)
            or any(not row["reason_codes"] for row in review_rows)):
        return False
    reasons = {reason for row in review_rows for reason in row["reason_codes"]}
    if not reasons <= _REVIEWABLE_REASONS or set(trace["reasons"]) != reasons:
        return False
    public_roles = {key.lower() for key in case["role_surfaces"]}
    found = {span["role"] for span in interpretation["role_spans"]}
    return bool(public_roles) and public_roles == found == set(interpretation["resolved_operand_bindings"])


def _queue_interpretation_review(store, audit_id, case, config, run_id):
    # Called in the same transaction that commits the local verdict. No HTTP here.
    trace = store.get("attempts", audit_id)
    if trace["disposition"] != "review" or not trace.get("interpretation_sha256"):
        return False
    interpretation = store.get("interpretations", trace["interpretation_sha256"])
    if not _review_eligible(trace, interpretation, case):
        return False
    text = trace["question"]
    roles = {key.lower(): values for key, values in case["role_surfaces"].items()}
    public_roles = {key[:-9]: values for key, values in case["public_request"].items()
                    if key.endswith("_surfaces")}
    if (text_sha(text) != trace["candidate_question_sha256"] or roles != public_roles
            or trace["semantic_instance_id"] != case["semantic_instance_id"]
            or trace["evidence_snapshot_id"] != case["evidence_snapshot_id"]
            or store.get("candidates", trace["candidate_object_sha256"]) != {"question": text}):
        raise ValueError("Advisory projection differs from the validated candidate or public roles")
    payload_ref = store.put("review_inputs", {"question": text, "licensed_role_surfaces": roles})
    inserted = store.db.execute("""
        INSERT OR IGNORE INTO interpretation_reviews
        (audit_id,review_policy_id,created_run,payload_ref) VALUES (?,?,?,?)
    """, (audit_id, _review_policy_id(config), run_id, payload_ref))
    return inserted.rowcount == 1


def _review_reservations(store, run_id=None):
    if run_id is None:
        return store.db.execute("SELECT COUNT(reserved_run) FROM interpretation_reviews").fetchone()[0]
    return store.db.execute("SELECT COUNT(*) FROM interpretation_reviews WHERE reserved_run=?",
                            (run_id,)).fetchone()[0]


def _pending_interpretation_review(store, config):
    if not config.get("interpretation_review_enabled", False) or config.get("max_review_http_calls", 0) < 1:
        return False
    return store.db.execute("""
        SELECT 1 FROM interpretation_reviews WHERE review_policy_id=? AND status='pending' LIMIT 1
    """, (_review_policy_id(config),)).fetchone() is not None


def _recover_interpretation_reviews(store):
    # The previous process may have paid for a call without recording the reply.
    with store.db:
        store.db.execute("""
            UPDATE gen_runs SET exports_ready=0 WHERE id IN
            (SELECT created_run FROM interpretation_reviews WHERE status='submitting')
        """)
        return store.db.execute("""
            UPDATE interpretation_reviews SET status='uncertain' WHERE status='submitting'
        """).rowcount


def _drain_one_interpretation_review(store, run_id, config, client):
    # The caller owns the campaign writer lock and checks the shared API budgets.
    db = store.db
    if db.in_transaction:
        raise RuntimeError("Finish the local validation transaction before advisory dispatch")
    if (not _pending_interpretation_review(store, config)
            or _review_reservations(store, run_id) >= config["max_review_http_calls"]):
        return False
    if client.reviewer_model != config.get("reviewer_model", API_MODEL):
        raise ValueError("Reviewer model differs from the queued policy")
    policy_id = _review_policy_id(config)
    row = db.execute("""
        SELECT audit_id,created_run,payload_ref FROM interpretation_reviews
        WHERE review_policy_id=? AND status='pending' ORDER BY rowid LIMIT 1
    """, (policy_id,)).fetchone()
    aid, original_run, payload_ref = row
    payload = store.get("review_inputs", payload_ref)
    with db:
        reserved = db.execute("""
            UPDATE interpretation_reviews SET status='submitting', reserved_run=?
            WHERE audit_id=? AND review_policy_id=? AND status='pending'
        """, (run_id, aid, policy_id))
        if reserved.rowcount != 1:
            return False
        db.execute("UPDATE gen_runs SET exports_ready=0 WHERE id=?", (original_run,))

    # Only the quoted question and the approved public role surfaces leave the VM.
    # The client independently checks schema and complete, ordered text coverage.
    result = {"audit_id": aid, "review_policy_id": policy_id, "payload_ref": payload_ref}
    try:
        advice = client.interpretation_reviewer(payload["question"], payload["licensed_role_surfaces"])
        if advice.get("use") != "adjudication_only":
            raise ValueError("Unexpected advisory authority")
        result["advice"] = advice
        status = "completed"
    except SamplingProviderError as exc:
        code = exc.metadata.get("http_status")
        unknown = (exc.code == "openai_response_unavailable" or code == 408
                   or (type(code) is int and 500 <= code <= 599))
        status = "uncertain" if unknown else "failed"
        result["error"] = {"code": exc.code, "provider_metadata": exc.metadata}
    except Exception as exc:
        # An adapter failure cannot change the local verdict or justify another charge.
        status = "uncertain"
        result["error"] = {"code": type(exc).__name__}
    with db:
        ref = store.put("review_advisories", result)
        db.execute("""
            UPDATE interpretation_reviews SET status=?, advisory_ref=?
            WHERE audit_id=? AND review_policy_id=? AND status='submitting'
        """, (status, ref, aid, policy_id))
        db.execute("UPDATE gen_runs SET exports_ready=0 WHERE id=?", (original_run,))
    return True


def _next_generation_job(store, inventory, contracts, source, auth, config, build_id, component_hashes):
    """Use a stable census cursor. Do not visit completed instances again."""
    db = store.db
    job = db.execute("SELECT * FROM gen_jobs WHERE state='ready' ORDER BY inventory_row LIMIT 1").fetchone()
    if job is not None:
        return job
    after = db.execute("SELECT COALESCE(MAX(inventory_row),0) FROM gen_jobs").fetchone()[0]
    entry = inventory.execute("SELECT rowid,instance_id,task FROM inventory_instances WHERE rowid>? ORDER BY rowid LIMIT 1",
                              (after,)).fetchone()
    if entry is None:
        return None
    cursor, instance, task = entry
    case, raw, matrix_row = load_inventory_instance(inventory, instance)
    if case["semantic_instance_id"] != instance or task != case["task_id"]:
        raise ValueError("Inventory instance identity mismatch")
    if (not raw.get("s1_bindings") or not raw.get("respondent_presentation")
            or matrix_row["original_policy_code"] != "A" or matrix_row["expected_status"] != "applicable"
            or matrix_row["task"] != task or matrix_row["rule_id"] != contracts[task]["semantics"]["source_binding"]["rule_id"]):
        raise ValueError("Inventory entry lacks verified source eligibility")
    with db:
        snapshot = store.put("snapshots",raw)
        if snapshot != case["evidence_snapshot_id"]:
            raise ValueError("Inventory evidence snapshot digest mismatch")
        payload = openai_payload(contracts[task]["grammar"],case["public_request"],config["model"])
        receipt = {"kind": "verified_sector1", "authorization": auth, "semantic_instance_id": instance,
                   "evidence_snapshot_id": snapshot, "build_id": build_id, "case_sha256": store.put("plans",case),
                   "public_request_sha256": store.put("requests",payload),
                   "parser_implementation_sha256": component_hashes["validation"], "source_run_id": source["run_id"],
                   "matrix_cell": matrix_row, "expression_mrcm_status": "not_applicable",
                   "supporting_observations": case["evidence_refs"], "presentation": raw["respondent_presentation"],
                   "source_binding_count": len(raw["s1_bindings"])}
        receipt_id = store.put("pre", receipt)
        db.execute("INSERT INTO gen_jobs(instance_id,task,inventory_row,receipt) VALUES (?,?,?,?)",
                   (instance,task,cursor,receipt_id))
    return db.execute("SELECT * FROM gen_jobs WHERE instance_id=?", (instance,)).fetchone()


def _record_provider_response(store, job, client, run_id, attempt_budget):
    """A committed reservation is the upper bound on charges even if the process crashes."""
    db, instance, number = store.db, job["instance_id"], job["attempts"] + 1
    receipt = store.get("pre",job["receipt"])
    digest = receipt["public_request_sha256"]
    payload = store.get("requests",digest)
    with db:
        db.execute("INSERT INTO gen_attempts(instance_id,number,stage,run_id) VALUES (?,?,'reserved',?)",
                   (instance,number,run_id))
        db.execute("UPDATE gen_jobs SET attempts=? WHERE instance_id=?", (number,instance))
    try:
        candidate, provider = client.generate(payload,digest)
        envelope = {"candidate": candidate, "provider": provider}
    except SamplingProviderError as exc:
        envelope = {"error": exc.code, "disposition": exc.disposition, "provider": exc.metadata}
    code = envelope.get("error")
    http_status = envelope["provider"].get("http_status")
    if code and isinstance(code, str):
        http_match = re.fullmatch(r"openai_http_(\d{3})",code)
        if http_status is None and http_match:
            http_status = int(http_match.group(1))
    if code == "openai_response_unavailable" or http_status == 408 or (type(http_status) is int and 500 <= http_status <= 599):
        # A server failure or lost response may follow a completed, billable request.
        with db:
            error_id = store.put("provider_responses",envelope)
            db.execute("UPDATE gen_attempts SET stage='uncertain',envelope=? WHERE instance_id=? AND number=?",
                       (error_id,instance,number))
            db.execute("UPDATE gen_jobs SET state='uncertain' WHERE instance_id=?",(instance,))
        return None, "provider_response_unavailable"
    if http_status in (401,403,404,429):
        # No question supplied. After fixing access/quota, a later invocation may use
        # a remaining attempt. Preserve the true HTTP expenditure and private history.
        state = "ready" if number < attempt_budget else "provider_attempt_budget_exhausted"
        with db:
            error_id = store.put("provider_responses",envelope)
            db.execute("""UPDATE gen_attempts SET stage='provider_pause',disposition='provider_pause',
                          completed_run=?,envelope=? WHERE instance_id=? AND number=?""",
                       (run_id,error_id,instance,number))
            db.execute("UPDATE gen_jobs SET state=?,closed_run=? WHERE instance_id=?",
                       (state,run_id if state != "ready" else None,instance))
        return None, code
    # Exceptions before this commit leave the request visible as outcome unknown.
    with db:
        response_id = store.put("provider_responses", envelope)
        db.execute("UPDATE gen_attempts SET stage='received',envelope=? WHERE instance_id=? AND number=?",
                   (response_id,instance,number))
    return number, None


def _close_generation_job(store, job, state, run_id, source, build_id):
    """Release only terminal, non-fatal jobs. The caller owns this SQLite transaction."""
    db, instance = store.db, job["instance_id"]
    if state in ("target_reached", "attempt_budget_exhausted"):
        for (audit_id,) in db.execute("""SELECT audit_id FROM gen_attempts
                    WHERE instance_id=? AND disposition='accepted' ORDER BY number""",(instance,)):
            trace = store.get("attempts",audit_id)
            record = {"audit_id":audit_id,"question":trace["question"],"candidate_question_sha256":trace["candidate_question_sha256"]}
            draft = assemble_validated(record,store,synthetic=False,source=source,build_id=build_id)
            corpus = None
            if source["corpus_export_authorized"] is True:
                corpus = assemble_validated(record,store,synthetic=False,source=source,build_id=build_id,for_corpus=True)
            db.execute("INSERT INTO qa(instance,normalized,close_run,body,corpus_body) VALUES(?,?,?,?,?)",
                       (instance,normalized_question(record["question"]),run_id,_wire_bytes(draft).decode(),
                        _wire_bytes(corpus).decode() if corpus is not None else None))
    db.execute("UPDATE gen_jobs SET state=?,closed_run=? WHERE instance_id=?", (state,run_id,instance))


def _finish_received_attempt(store, job, number, contracts, config, run_id, source, build_id):
    db, instance = store.db, job["instance_id"]
    envelope_id = db.execute("SELECT envelope FROM gen_attempts WHERE instance_id=? AND number=? AND stage='received'",
                             (instance,number)).fetchone()[0]
    envelope = store.get("provider_responses",envelope_id)
    receipt = store.get("pre",job["receipt"])
    case = store.get("plans",receipt["case_sha256"])
    pair = contracts[job["task"]]
    def recorded_response(payload,digest,attempt_no):
        if "error" in envelope:
            raise SamplingProviderError(envelope["error"],envelope["disposition"],envelope["provider"])
        return envelope["candidate"],envelope["provider"]
    keys = [r[0] for r in db.execute("""SELECT question_key FROM gen_attempts
                                      WHERE instance_id=? AND disposition='accepted'""",(instance,))]
    with db:
        sample = sample_case(case,pair["grammar"],pair["semantics"],receipt,store,recorded_response,1,1,
                             attempt_offset=number-1,prior_question_keys=keys)
        item = sample["attempts"][0]
        status = item["disposition"]
        is_accepted = status == "accepted"
        key = normalized_question(sample["accepted"][0]["question"]) if is_accepted else None
        db.execute("""UPDATE gen_attempts SET stage='done',disposition=?,audit_id=?,question_key=?,completed_run=?
                      WHERE instance_id=? AND number=?""",(status,item["audit_id"],key,run_id,instance,number))
        if status == "review":
            _queue_interpretation_review(store, item["audit_id"], case, config, run_id)
        successes = job["accepted"] + int(is_accepted)
        db.execute("UPDATE gen_jobs SET accepted=? WHERE instance_id=?", (successes,instance))
        if status == "fatal_abort":
            state = "fatal_abort"
        elif successes >= config["requested_count"]:
            state = "target_reached"
        elif number >= config["attempt_budget"]:
            state = "attempt_budget_exhausted"
        else:
            state = "ready"
        if state != "ready":
            _close_generation_job(store,job,state,run_id,source,build_id)
    return state


def _sampling_run_report(store, job, policy):
    """Reconstruct one V13 instance report using the original immutable attempt traces."""
    attempts, accepted, counts = [], [], Counter()
    for row in store.db.execute("""SELECT number,stage,disposition,audit_id,envelope FROM gen_attempts
                                WHERE instance_id=? ORDER BY number""",(job["instance_id"],)):
        disposition = row["disposition"] or row["stage"]
        item = {"attempt_no": row["number"], "audit_id": row["audit_id"], "disposition": disposition,
                "candidate_question_sha256": None, "reasons": []}
        if row["audit_id"]:
            trace = store.get("attempts",row["audit_id"])
            item.update(candidate_question_sha256=trace["candidate_question_sha256"],reasons=trace["reasons"])
            if disposition == "accepted" and job["state"] in ("target_reached","attempt_budget_exhausted"):
                accepted.append({"question":trace["question"],"candidate_question_sha256":trace["candidate_question_sha256"],
                                 "audit_id":row["audit_id"]})
        elif row["envelope"]:
            provider_error = store.get("provider_responses",row["envelope"])
            item["reasons"] = [provider_error.get("error",disposition)]
        elif row["stage"] == "uncertain":
            item["reasons"] = ["submission_outcome_unknown"]
        counts[disposition] += 1
        attempts.append(item)
    return {"semantic_instance_id":job["instance_id"],"task_id":job["task"],
            "pre_evidence_receipt_id":job["receipt"],"requested_count":policy["requested_count"],
            "attempt_budget":policy["attempt_budget"],"attempts_used":len(attempts),"accepted_count":len(accepted),
            "shortfall":policy["requested_count"] - len(accepted),"stop_reason":job["state"],
            "counts_by_outcome":dict(counts),"attempts":attempts,"accepted":accepted,
            "validated_candidates_held":job["accepted"] - len(accepted)}


def _generation_run_outputs(store, directory, policy):
    """Rebuild the run views from committed records, including after a kernel interruption."""
    db, run_id = store.db, directory.name
    directory.mkdir(parents=True,exist_ok=True)
    for status, name in (("review","review.jsonl"),("rejected","rejected.jsonl")):
        query = """SELECT a.audit_id,a.instance_id,j.task,j.receipt FROM gen_attempts a JOIN gen_jobs j USING(instance_id)
                   WHERE a.completed_run=? AND a.disposition=? ORDER BY a.rowid"""
        with (directory/name).open("w",encoding="utf-8") as file:
            for aid, instance, task, rid in db.execute(query,(run_id,status)):
                trace, receipt = store.get("attempts",aid), store.get("pre",rid)
                case = store.get("plans",receipt["case_sha256"])
                review_info = {}
                if status == "review":
                    rows = db.execute("""
                        SELECT review_policy_id,status,advisory_ref FROM interpretation_reviews
                        WHERE audit_id=? ORDER BY rowid
                    """, (aid,)).fetchall()
                    review_info = {
                        "reviewer_status": rows[-1][1] if rows else "not_queued",
                        "advisory_ref": rows[-1][2] if rows else None,
                        "review_history": [
                            {"review_policy_id": r[0], "status": r[1], "advisory_ref": r[2]}
                            for r in rows
                        ],
                    }
                _json_line(file,{**review_info,"task_id":task,"profile_id":case["profile_id"],"audit_id":aid,
                            "question":trace["question"],"reasons":trace["reasons"],
                            "presentation":respondent_s8_reference(receipt["presentation"]),
                            "semantic_instance_id":instance,"pre_evidence_receipt_id":rid,"disposition":status})
    with (directory/"accepted_questions.jsonl").open("w",encoding="utf-8") as file:
        for (body,) in db.execute("SELECT body FROM qa WHERE close_run=? ORDER BY rowid",(run_id,)):
            file.write(body + "\n")
    # Reconstruct the sampling report for every instance touched during this invocation.
    with (directory/"batches.jsonl").open("w",encoding="utf-8") as file:
        selected = """SELECT * FROM gen_jobs WHERE instance_id IN (
                      SELECT instance_id FROM gen_jobs WHERE closed_run=?
                      UNION SELECT instance_id FROM gen_attempts WHERE run_id=?
                      UNION SELECT instance_id FROM gen_attempts WHERE completed_run=?) ORDER BY inventory_row"""
        for job in db.execute(selected,(run_id,run_id,run_id)):
            _json_line(file, _sampling_run_report(store,job,policy))
    with (directory/"batch_progress.jsonl").open("w",encoding="utf-8") as file:
        for (body,) in db.execute("SELECT body FROM gen_batches WHERE run_id=? ORDER BY number",(run_id,)):
            file.write(body + "\n")
    with (directory/"uncertain_requests.jsonl").open("w",encoding="utf-8") as file:
        for instance, number, rid in db.execute("""SELECT a.instance_id,a.number,j.receipt FROM gen_attempts a
                JOIN gen_jobs j USING(instance_id) WHERE a.run_id=? AND a.stage='uncertain'""",(run_id,)):
            _json_line(file,{"semantic_instance_id":instance,"attempt_no":number,"pre_evidence_receipt_id":rid,
                             "reason":"submission_outcome_unknown","reserved_run_id":run_id})
    with db:
        db.execute("UPDATE gen_runs SET exports_ready=1 WHERE id=?", (run_id,))


def _hold_interrupted_submissions(store):
    """A persisted provider response can be validated offline. A bare reservation may have been billed."""
    db = store.db
    with db:
        db.execute("""UPDATE gen_jobs SET state='uncertain' WHERE instance_id IN
                      (SELECT instance_id FROM gen_attempts WHERE stage='reserved')""")
        db.execute("UPDATE gen_runs SET exports_ready=0 WHERE id IN (SELECT run_id FROM gen_attempts WHERE stage='reserved')")
        db.execute("UPDATE gen_attempts SET stage='uncertain' WHERE stage='reserved'")


def resolve_uncertain_submission(campaign_database, instance_id):
    """Allow this instance to use any remaining attempts. The unknown attempt stays charged to both caps."""
    path = Path(campaign_database)
    if path.name != "generation.sqlite3" or not path.is_file():
        raise ValueError("Expected an existing campaign generation.sqlite3")
    store = GenerationStore(path.parent)
    try:
        _hold_interrupted_submissions(store)
        with store.db:
            updated = store.db.execute("UPDATE gen_jobs SET state='ready' WHERE instance_id=? AND state='uncertain'",
                                       (instance_id,)).rowcount
            if updated != 1:
                raise ValueError("Instance has no interrupted submission to resolve")
    finally:
        store.close()


def generate_inventory_batch(config, inventory, contracts, source, auth, build_id, component_hashes, run, info, *, client_factory=None):
    """Consume the verified inventory and share the call budget with advisory reviews."""
    run = Path(run)
    for key in ("requested_count","attempt_budget","batch_size"):
        if type(config[key]) is not int or config[key] < 1:
            raise ValueError(key + " must be a positive integer")
    for key in ("max_http_calls","max_batches_per_run","max_total_api_calls"):
        value = config.get(key,0)
        if value is not None and (type(value) is not int or value < 0):
            raise ValueError(key + " must be a nonnegative integer")
    review_enabled = config.get("interpretation_review_enabled", False)
    review_limit = config.get("max_review_http_calls", 0)
    reviewer_model = config.get("reviewer_model", API_MODEL)
    if type(review_enabled) is not bool or type(review_limit) is not int or review_limit < 0:
        raise ValueError("Expected a boolean review switch and a nonnegative review call limit.")
    if review_enabled and (not isinstance(reviewer_model, str) or not reviewer_model.strip()):
        raise ValueError("A reviewer model ID is required.")
    if not auth["effective_qa_generation_authorized"]:
        raise PermissionError("Sector 2 generation is not authorized")
    census = inventory.execute("SELECT build_id,complete FROM inventory_state WHERE singleton=1").fetchone()
    inventory_id = info["inventory_id"]
    database_sha256 = info["inventory_database_sha256"]
    if not isinstance(database_sha256, str) or not re.fullmatch(r"[a-f0-9]{64}", database_sha256):
        raise ValueError("Generation requires a verified physical snapshot SHA-256.")
    if census is None or tuple(census) != (inventory_id,1):
        raise ValueError("Generation requires the completed inventory identified by this run")
    policy = {"build_id":build_id,"model":config["model"],"wording_policy_id":WORDING_POLICY_ID,
              "requested_count":config["requested_count"],
              "attempt_budget":config["attempt_budget"]}
    policy.update(inventory_id=inventory_id, inventory_database_sha256=database_sha256,
                  interpretation_review_enabled=review_enabled,
                  reviewer_model=reviewer_model if review_enabled else None)
    campaign_root = run.parents[1] / "campaigns" / wire_digest(policy)
    store = GenerationStore(campaign_root)
    run_id, db = run.name, store.db
    info.update(campaign_database=str(campaign_root/"generation.sqlite3"),generation_policy=policy)
    client = None
    session_calls = review_calls = batches = 0
    total_limit = config.get("max_total_api_calls") or 0
    session_limit = config["max_http_calls"]
    max_batches = config.get("max_batches_per_run") or 0
    stop = None
    review_unavailable = None

    def get_client():
        nonlocal client
        if client is None:
            if client_factory is None:
                key = _colab_key()
                client = OpenAIQuestionClient(
                    session_limit, key_getter=lambda: key, reviewer_model=reviewer_model
                )
            else:
                client = client_factory(session_limit)
                if review_enabled:
                    client.reviewer_model = reviewer_model
        return client

    def review_one_if_budgeted():
        nonlocal session_calls, review_calls, review_unavailable
        if (not review_enabled or review_unavailable or review_calls >= review_limit
                or session_calls >= session_limit
                or (total_limit and initial_reservations + session_calls >= total_limit)
                or not _pending_interpretation_review(store, config)):
            return False
        try:
            review_client = get_client()
        except SamplingProviderError as exc:
            # Reading the API key may fail before any network request or reservation.
            review_unavailable = exc.code
            return False
        # The helper commits its reservation before dispatch and catches provider errors.
        if _drain_one_interpretation_review(store, run_id, config, review_client):
            session_calls += 1
            review_calls += 1
            return True
        return False

    def refresh_run_views():
        dirty = db.execute("SELECT id FROM gen_runs WHERE exports_ready=0").fetchall()
        for (rid,) in dirty:
            _generation_run_outputs(store, run.parent / rid, policy)

    try:
        _recover_interpretation_reviews(store)
        _hold_interrupted_submissions(store)
        refresh_run_views()
        with db:
            db.execute("INSERT INTO gen_runs(id) VALUES(?)",(run_id,))
        initial_generation = db.execute("SELECT COUNT(*) FROM gen_attempts").fetchone()[0]
        initial_reviews = _review_reservations(store)
        initial_reservations = initial_generation + initial_reviews
        total_eligible = inventory.execute("SELECT COUNT(*) FROM inventory_instances").fetchone()[0]
        # Give a previously queued review a turn even if generation uses the rest of this run.
        review_one_if_budgeted()
        while not stop:
            touched = 0
            batch_start, review_start = session_calls, review_calls
            while touched < config["batch_size"]:
                budget_stop = None
                if session_calls >= session_limit:
                    budget_stop = "http_call_budget_reached"
                elif total_limit and initial_reservations + session_calls >= total_limit:
                    budget_stop = "campaign_call_budget_reached"
                if budget_stop:
                    # Allow validation/release of already-paid work before returning.
                    ready = db.execute("""SELECT attempts,instance_id FROM gen_jobs WHERE state='ready'
                                       ORDER BY inventory_row LIMIT 1""").fetchone()
                    has_response = ready and db.execute(
                        "SELECT 1 FROM gen_attempts WHERE instance_id=? AND stage='received'",(ready[1],)
                    ).fetchone()
                    if not ready or (ready[0] < config["attempt_budget"] and not has_response):
                        stop = budget_stop
                        break
                job = _next_generation_job(store,inventory,contracts,source,auth,config,build_id,component_hashes)
                if job is None:
                    stop = "queue_drained"
                    break
                instance = job["instance_id"]
                state = "ready"
                touched += 1
                while state == "ready":
                    received = db.execute("SELECT number FROM gen_attempts WHERE instance_id=? AND stage='received'",
                                          (instance,)).fetchone()
                    if received:
                        number = received[0]
                    elif job["attempts"] >= config["attempt_budget"]:
                        with db:
                            _close_generation_job(store,job,"attempt_budget_exhausted",run_id,source,build_id)
                        break
                    else:
                        if session_calls >= session_limit:
                            stop = "http_call_budget_reached"
                            break
                        if total_limit and initial_reservations + session_calls >= total_limit:
                            stop = "campaign_call_budget_reached"
                            break
                        get_client()
                        session_calls += 1
                        number, pause = _record_provider_response(store,job,client,run_id,config["attempt_budget"])
                        if pause:
                            stop = pause
                            break
                    state = _finish_received_attempt(store,job,number,contracts,config,run_id,source,build_id)
                    if state != "fatal_abort":
                        # Local audit and the question transaction have already committed.
                        review_one_if_budgeted()
                    if state == "fatal_abort":
                        stop = "fatal_abort"
                    if state == "ready":
                        job = db.execute("SELECT * FROM gen_jobs WHERE instance_id=?",(instance,)).fetchone()
                if stop:
                    break
            if touched:
                batches += 1
                body = {"batch_number":batches,"instances_visited":touched,"api_reservations":session_calls-batch_start,
                        "review_reservations":review_calls-review_start,
                        "stop":stop or "batch_complete"}
                with db:
                    db.execute("INSERT INTO gen_batches VALUES(?,?,?)",(run_id,batches,_wire_bytes(body).decode()))
            if not stop and max_batches and batches >= max_batches:
                stop = "batch_limit_reached"
        if stop in {"queue_drained", "batch_limit_reached"}:
            while review_one_if_budgeted():
                pass
        states = dict(db.execute("SELECT state,COUNT(*) FROM gen_jobs GROUP BY state"))
        finished = states.get("target_reached",0) + states.get("attempt_budget_exhausted",0)
        held = sum(states.get(key,0) for key in ("fatal_abort","uncertain","provider_attempt_budget_exhausted"))
        if total_eligible == finished + held and stop in {"queue_drained","batch_limit_reached","http_call_budget_reached","campaign_call_budget_reached"}:
            stop = "complete_with_holds" if held else "completed"
        refresh_run_views()
        drafts = store.write_cumulative()
        if source["corpus_export_authorized"]:
            info["authorized_corpus"] = str(store.write_cumulative(True))
        info["interpretation_review"] = {
            "enabled": review_enabled, "model": reviewer_model,
            "run_limit": review_limit, "run_reservations": review_calls,
            "campaign_reservations": initial_reviews + review_calls,
            "campaign_states": dict(db.execute(
                "SELECT status,COUNT(*) FROM interpretation_reviews GROUP BY status"
            )),
            "unavailable_reason": review_unavailable,
        }
        info.update(status=stop,http_calls=client.calls if client else 0,validated_drafts=str(drafts),
                    unique_drafts_added=db.execute("SELECT COUNT(*) FROM qa WHERE close_run=?",(run_id,)).fetchone()[0],
                    corpus_exports_added=db.execute("SELECT COUNT(*) FROM qa WHERE close_run=? AND corpus_body IS NOT NULL",
                                                    (run_id,)).fetchone()[0],
                    generation={"job_states":states,"eligible_instances":total_eligible,
                                "terminal_instances":finished,"held_instances":held,
                                "unfinished_instances":total_eligible - finished - held,
                                "campaign_reservations":initial_reservations + session_calls,
                                "run_reservations":session_calls,"batches":batches,
                                "run_generation_reservations":session_calls-review_calls,
                                "run_review_reservations":review_calls,
                                "validated_questions_total":db.execute("SELECT COUNT(*) FROM qa").fetchone()[0]})
        _save_json(run/"summary.json",info)
        return info
    except BaseException as exc:
        info.update(status="interrupted" if isinstance(exc,KeyboardInterrupt) else "fatal_abort",
                    http_calls=client.calls if client else 0,error={"type":type(exc).__name__,"detail":str(exc)[:320]})
        _save_json(run/"summary.json",info)
        raise
    finally:
        store.close()


In [14]:
@contextmanager
def sector2_progress(engine, config):
    """Display parent-process progress while the existing engine owns all work and checkpoints."""
    bar = None
    initial_done = 0

    def show(label, total, unit, initial=0):
        nonlocal bar
        if bar is not None:
            bar.close()
        # The text backend also renders when the Colab widget manager is unavailable.
        bar = text_progress(desc=label, total=total, initial=initial, unit=unit,
                            file=sys.stdout, disable=False, ascii=True, mininterval=0.5,
                            miniters=1, ncols=110, leave=True)

    def note(message):
        bar.set_postfix_str(message, refresh=True)

    class VisibleInventory(engine["SnapshotInventory"]):
        def __enter__(self):
            nonlocal initial_done
            opened = super().__enter__()
            initial_done = self.end
            bar.update(1)
            if self.complete:
                show("Verified database already complete", initial_done, "recipe", initial_done)
            return opened

        def checkpoint(self, done, expected):
            note(f"committed on VM: {done:,}")
            super().checkpoint(done, expected)
            note(f"VM: {done:,}, persistent snapshot: {self.end:,}")

    load_corpus = engine["_clinical_s8_corpus"]
    def read_metadata(*args, **kwargs):
        show("Reading S8 metadata", 1, "step")
        corpus = load_corpus(*args, **kwargs)
        bar.update(1)
        show("Preparing S1 recipe bindings", 1, "step")
        return corpus

    recipe_stream = engine["_verified_recipe_stream"]
    def ready_stream(*args, **kwargs):
        expected, stream = recipe_stream(*args, **kwargs)
        bar.update(1)
        show("S8 recipes evaluated", expected, "recipe", initial_done)
        return expected, stream

    process_window = engine["_process_window"]
    def evaluated_window(*args, **kwargs):
        results = process_window(*args, **kwargs)
        try:
            for result in results:
                bar.update(1)
                yield result
        finally:
            results.close()

    summarize = engine["inventory_summary"]
    def inventory_report(*args, **kwargs):
        show("Preparing inventory report", 1, "step")
        return summarize(*args, **kwargs)

    write_holds = engine["write_inventory_holds"]
    def held_report(*args, **kwargs):
        note("writing held instances")
        return write_holds(*args, **kwargs)

    generate_batch = engine["generate_inventory_batch"]
    def campaign(*args, **kwargs):
        bar.update(1)
        show("OpenAI calls this run", config["max_http_calls"], "call")
        note("opening campaign")
        return generate_batch(*args, **kwargs)

    submit = engine["_record_provider_response"]
    def provider_attempt(store, job, client, run_id, attempt_budget):
        note(f"{job['task']} | attempt {job['attempts'] + 1}")
        try:
            return submit(store, job, client, run_id, attempt_budget)
        finally:
            bar.update(client.calls - bar.n)
            bar.refresh()

    submit_review = engine["_drain_one_interpretation_review"]
    def review_attempt(store, run_id, config, client):
        note("requesting an independent interpretation")
        try:
            return submit_review(store, run_id, config, client)
        finally:
            bar.update(client.calls - bar.n)
            bar.refresh()

    finish = engine["_finish_received_attempt"]
    def validate_attempt(store, job, number, *args, **kwargs):
        note(f"validating attempt {number}")
        return finish(store, job, number, *args, **kwargs)

    # Install the display adapters for this cell only. Restore each original on every exit.
    adapters = {
        "SnapshotInventory": VisibleInventory, "_clinical_s8_corpus": read_metadata,
        "_verified_recipe_stream": ready_stream, "_process_window": evaluated_window,
        "inventory_summary": inventory_report, "write_inventory_holds": held_report,
        "generate_inventory_batch": campaign, "_record_provider_response": provider_attempt,
        "_finish_received_attempt": validate_attempt,
        "_drain_one_interpretation_review": review_attempt,
    }
    with ExitStack() as cleanup:
        show("Verifying persistent inventory", 1, "step")
        cleanup.callback(lambda: bar.close())
        for name, adapter in adapters.items():
            cleanup.callback(engine.__setitem__, name, engine[name])
            engine[name] = adapter
        yield
        if bar.desc == "Preparing inventory report":
            bar.update(1)


### Run the census or advance a batch

Run this cell to check the Drive snapshot and open a verified local inventory, or build a new one. Progress for a new census starts at zero. A partial census starts at its verified recipe count. A complete census proceeds to the summary or to the authorized generation batch. The run summary identifies the inventory and the generation build. Keep the same persistent output root on a replacement VM to locate the same inventory.

### Planning estimate for the published S8 release

The S1 run `20261118T013916_145151Z` pins S8 data commit `18cd001f6ba7951e32b1bb82197ece9920e4e8ec`. The release contains **216,808 recipes** in 11 variants. A projection over all recipes counted eligible operand combinations using visible positive observations, applicable task/variant rules, S1 approved class mappings and display restrictions. The projected counts below use one question per distinct eligible task instance. These are planning estimates. The full runtime census establishes exact targets for the revision actually loaded.

| Task | Estimated question target at one per instance |
|---|---:|
| Finding identification | 190,124 |
| Finding category | 123,977 |
| Evidence frame localization | 28,168 |
| Optional spatial localization | 4,136 |
| Verified reference comparison | 0 |
| Observed anchor order | 78,114 |
| Temporal relative evidence localization | 86,559 |
| **Total** | **511,078** |

The projection was cross-checked against native S1 binding, the S2 plan builder and semantic-ID deduplication on a stratified sample of 11,264 recipes (1,024 per variant). All 78,848 sampled recipe/task count comparisons agreed. The comparison task had an applicable policy cell, but its explicit normal-reference class had no approved binding in that cell. With two questions per eligible instance, the projected target is 1,022,156. The achieved number of accepted questions depends on generation and validation. The projection uses no OpenAI calls.


In [15]:
# Rebind the optional generation dependency when the mode changes in this live runtime.
engine["spacy"] = spacy
if MODE == "real_openai" and spacy is None:
    raise RuntimeError("Select real_openai in the first cell and run that cell before starting generation.")
config = {
    "mode": MODE, "output_dir": str(output_dir), "cache_dir": str(work / "cache"), "s8_dir": str(work / "s8"),
    "model": engine["API_MODEL"], "requested_count": QUESTIONS_PER_INSTANCE, "attempt_budget": ATTEMPT_BUDGET,
    "max_http_calls": MAX_HTTP_CALLS, "max_total_api_calls": MAX_TOTAL_API_CALLS,
    "interpretation_review_enabled": INTERPRETATION_REVIEW_ENABLED,
    "max_review_http_calls": MAX_REVIEW_HTTP_CALLS,
    "reviewer_model": REVIEWER_MODEL,
    "batch_size": BATCH_SIZE, "max_batches_per_run": MAX_BATCHES_PER_RUN,
    "inventory_window_size": INVENTORY_WINDOW_SIZE, "inventory_workers": INVENTORY_WORKERS,
    "checkpoint_recipes": CHECKPOINT_RECIPES, "snapshot_every_recipes": SNAPSHOT_EVERY_RECIPES, "tasks": [],
}
with sector2_progress(engine, config):
    result = engine["execute_sector2"](config, imported_inputs, component_hashes)
print(json.dumps(result, ensure_ascii=False, indent=2))


Verifying persistent inventory:   0%|                                                 | 0/1 [00:00<?, ?step/s]Inventory ID: fc0e15e47becbbbf75104a2b7a792bcd096ee168d5dac373a74235c069bf2bf4 | rebuilding | Missing inventory.json
VM working database: /content/mmvqa_phase3_sector2/snapshot_work/fc0e15e47becbbbf75104a2b7a792bcd096ee168d5dac373a74235c069bf2bf4/inventory.sqlite3
Persistent inventory manifest: /content/drive/MyDrive/MMVQA_Clinical/outputs/phase3/sector2/real_s8/inventories/fc0e15e47becbbbf75104a2b7a792bcd096ee168d5dac373a74235c069bf2bf4/inventory.json
Verified recipes available: 0
Reading S8 metadata:   0%|                                                            | 0/1 [00:00<?, ?step/s]Complete metadata table: source_evidence_manifest: 66,127 rows
Complete metadata table: visual_manifest: 184,328 rows
Complete metadata table: visual_source_manifest: 478,701 rows
Complete metadata table: benchmark_ground_truth_manifest: 47,239 rows
Complete metadata table: shared_source_grou

Verifying SQLite bytes:   0%|          | 0.00/344M [00:00<?, ?B/s]

Verifying SQLite bytes:   0%|          | 0.00/344M [00:00<?, ?B/s]

S8 recipes evaluated:  23%|2| 50000/216808 [06:45<24:43, 112.46recipe/s, VM: 50,000, persistent snapshot: 50,0S8 inventory: 50,000/216,808 recipes committed.
S8 recipes evaluated:  25%|2| 55000/216808 [07:25<17:59, 149.85recipe/s, VM: 55,000, persistent snapshot: 50,0S8 inventory: 55,000/216,808 recipes committed.
S8 recipes evaluated:  28%|2| 60000/216808 [08:04<24:02, 108.70recipe/s, VM: 60,000, persistent snapshot: 50,0S8 inventory: 60,000/216,808 recipes committed.
S8 recipes evaluated:  30%|2| 65000/216808 [08:44<19:19, 130.97recipe/s, VM: 65,000, persistent snapshot: 50,0S8 inventory: 65,000/216,808 recipes committed.
S8 recipes evaluated:  32%|3| 70000/216808 [09:25<26:46, 91.36recipe/s, VM: 70,000, persistent snapshot: 50,00S8 inventory: 70,000/216,808 recipes committed.
S8 recipes evaluated:  35%|3| 75000/216808 [10:03<12:20, 191.45recipe/s, VM: 75,000, persistent snapshot: 50,0S8 inventory: 75,000/216,808 recipes committed.
S8 recipes evaluated:  37%|3| 80000/216808 [10:43<18

Verifying SQLite bytes:   0%|          | 0.00/699M [00:00<?, ?B/s]

Verifying SQLite bytes:   0%|          | 0.00/699M [00:00<?, ?B/s]

... (6 outputs left out)

S8 recipes evaluated:  92%|9| 200000/216808 [35:23<01:40, 167.12recipe/s, VM: 200,000, persistent snapshot: 20S8 inventory: 200,000/216,808 recipes committed.
S8 recipes evaluated:  95%|9| 205000/216808 [36:15<01:21, 144.35recipe/s, VM: 205,000, persistent snapshot: 20S8 inventory: 205,000/216,808 recipes committed.
S8 recipes evaluated:  97%|9| 210000/216808 [36:58<00:33, 201.09recipe/s, VM: 210,000, persistent snapshot: 20S8 inventory: 210,000/216,808 recipes committed.
S8 recipes evaluated:  99%|9| 215000/216808 [37:39<00:11, 153.82recipe/s, VM: 215,000, persistent snapshot: 20S8 inventory: 215,000/216,808 recipes committed.
S8 recipes evaluated: 100%|#| 216808/216808 [37:54<00:00, 177.70recipe/s, VM: 216,808, persistent snapshot: 20S8 inventory: 216,808/216,808 recipes committed.


Verifying SQLite bytes:   0%|          | 0.00/1.48G [00:00<?, ?B/s]

Verifying SQLite bytes:   0%|          | 0.00/1.48G [00:00<?, ?B/s]

S8 recipes evaluated: 100%|#| 216808/216808 [42:19<00:00, 85.39recipe/s, VM: 216,808, persistent snapshot: 200
Preparing inventory report:   0%|                             | 0/1 [00:02<?, ?step/s, writing held instances]Recipes examined: 216808 | Unique eligible instances: 511078
evidence_frame_localization 28168 instances; 28168 target questions
finding_category 123977 instances; 123977 target questions
finding_identification 190124 instances; 190124 target questions
observed_anchor_order 78114 instances; 78114 target questions
optional_spatial_localization 4136 instances; 4136 target questions
temporal_relative_evidence_localization 86559 instances; 86559 target questions
verified_reference_comparison 0 instances; 0 target questions
Corpus question target: 511078
OpenAI calls this run: 100%|##########################| 12/12 [00:30<00:00,  2.54s/call, validating attempt 1]
{
  "mode": "real_openai",
  "build_id": "be9d4c65c69fd0b907f686075f4896e8c929d03d3683e59759a6ec28272a5b80",
  "

### Run outputs

The result prints the run directory, inventory checkpoint, exact per-task counts and question target. The generation summary separately reports API reservations, HTTP calls, finalized instances and accepted drafts. A plan for one question per eligible instance assumes the validator will eventually accept a wording for each instance. An API quota or exhausted retries can leave a shortfall.

| Location | Contents |
|---|---|
| `s8_vm_sync.json` in the run directory | Transfer and integrity checks for S8 metadata files, plus the counts of visual references matched across manifests and the number of visual files opened (zero) |
| `inventory_summary.json` in the run directory | Recipes examined and target questions by task |
| `inventory_database` in the run summary | Active SQLite inventory on the VM disk |
| `inventory_manifest` in the run summary | Verified identity of the census inputs and producer |
| `inventory_storage` in the run summary | Snapshot verification, build/reuse decision and publication progress |
| `held_instances.jsonl` in the persistent inventory directory | Pre-generation holds |
| `generation.sqlite3` in the campaign directory | Durable jobs, budget reservations, API responses, validation traces and deduplicated QA |
| `accepted_questions.jsonl`, `review.jsonl`, `rejected.jsonl` in the generation run | Finalized QA for this run and individual candidates for adjudication or rejection |
| `batches.jsonl`, `batch_progress.jsonl` and `uncertain_requests.jsonl` in the generation run | Per-instance attempt reports, local batch progress and submissions with an unknown outcome |
| `validated_drafts.jsonl` in the campaign directory | Accepted, deduplicated QA records accumulated across runs |
| `summary.json` in the run directory | Build, source, authorization, status, budget and output locations |


Read `inventory.json` at the path printed by the run. It records the inventory ID, snapshot SHA-256, relative database path, saved recipe count and completion flag. In `real_openai`, campaign attempts and budgets are scoped to the generation policy and the verified complete inventory. Source verification still runs when the snapshot is reused.


In [16]:
# Display the exact Drive snapshot selected or published by this run.
inventory_manifest_path = Path(result["inventory_manifest"])
published_inventory = json.loads(inventory_manifest_path.read_text(encoding="utf-8"))
if (published_inventory["inventory_id"] != result["inventory_id"]
        or published_inventory["database_sha256"] != result["inventory_database_sha256"]):
    raise ValueError("The published inventory changed since this run. Rerun the execution cell to verify it.")
print("Opening decision:", result["inventory_storage"]["mode"], "|", result["inventory_storage"]["reason"])
print("Inventory manifest:", inventory_manifest_path)
print("Inventory ID:", result["inventory_id"])
print("SQLite SHA-256:", published_inventory["database_sha256"])
print("Verified recipes:", result["inventory"]["recipes_examined"])
print("Eligible instances:", result["inventory"]["eligible_instances"])

run_directory = Path(result["run_directory"])
for name in ("accepted_questions.jsonl", "review.jsonl", "rejected.jsonl"):
    path = run_directory / name
    if path.is_file():
        with path.open(encoding="utf-8") as stream:
            first = next((json.loads(line) for line in stream if line.strip()), None)
        print("\n" + name)
        if first:
            print(first.get("question"))
            print(first.get("answer", first.get("reasons")))
        else:
            print("(empty)")
print("\nRun summary:", run_directory / "summary.json")


Opening decision: rebuilding | Missing inventory.json
Inventory manifest: /content/drive/MyDrive/MMVQA_Clinical/outputs/phase3/sector2/real_s8/inventories/fc0e15e47becbbbf75104a2b7a792bcd096ee168d5dac373a74235c069bf2bf4/inventory.json
Inventory ID: fc0e15e47becbbbf75104a2b7a792bcd096ee168d5dac373a74235c069bf2bf4
SQLite SHA-256: 66febfd1b896bdcea44b7bb4aea70eb264ee5b36e3519535d4fbdb7c71246641
Verified recipes: 216808
Eligible instances: 511078

accepted_questions.jsonl
Which one of the displayed panels shows lymphangiectasia?
image_1/panel_2

review.jsonl
What clinical group does the dataset assign to the finding lymphangiectasia in image_1/panel_2?
['full_meaning_not_established', 'speech_act_unresolved', 'unresolved_indirect_disclosure', 'unverified_english_wording', 'unverified_role_coverage']

rejected.jsonl
Could you name a finding visible in image 1, panel 2?
['different_named_visual_referent', 'speech_act_unresolved', 'unresolved_indirect_disclosure', 'unverified_english_wording'

In [17]:
def finalize_corpus(result, source):
    run = Path(result["run_directory"]).resolve()

    if result["mode"] != "real_openai":
        raise ValueError("Select a real generation campaign.")
    if (source.get("corpus_export_authorized") is not True
            or result["authorization"].get("sector1_export_authorized") is not True):
        raise PermissionError("The verified release does not authorize corpus export.")
    if any(source.get(k) != v for k, v in result["source"].items()):
        raise ValueError("The imported release differs from the generation release.")

    policy = result["generation_policy"]
    build = result["build_id"]
    if policy["requested_count"] != 1 or policy["build_id"] != build:
        raise ValueError("Expected one question per instance for this build.")

    campaign = run.parents[1] / "campaigns" / engine["wire_digest"](policy)
    corpus = campaign / "authorized_corpus.jsonl"
    db_path = campaign / "generation.sqlite3"
    if (db_path != Path(result["campaign_database"]).resolve()
            or corpus != Path(result["authorized_corpus"]).resolve()):
        raise ValueError("Export and database must belong to the selected campaign.")

    with ExitStack() as stack:
        def connect(path, mode):
            uri = Path(path).resolve().as_uri() + "?mode=" + mode
            db = sqlite3.connect(uri, uri=True, timeout=0)
            stack.callback(db.close)
            return db

        # Use the generator's lock so the export cannot change during the check.
        lock = connect(campaign / "writer.lock.sqlite3", "rw")
        lock.execute("BEGIN EXCLUSIVE")

        # Read the exact, completed database used to partition this campaign.
        vm = engine["_local_inventory_volume"](Path(result["inventory_database"]).parent)
        vm.mkdir(parents=True, exist_ok=True)
        temp = stack.enter_context(tempfile.TemporaryDirectory(dir=vm, prefix="finalize-"))
        snapshot, snapshot_stat = engine["finalized_inventory_snapshot"](result, Path(temp))
        inv = connect(snapshot, "ro&immutable=1")
        inv.execute("BEGIN")
        state = inv.execute("""
            SELECT build_id, complete, recipes_done = expected_recipes
            FROM inventory_state WHERE singleton = 1
        """).fetchone()
        if state != (result["inventory_id"], 1, 1):
            raise ValueError("Inventory is incomplete or belongs to another build.")

        eligible = {r[0] for r in inv.execute("SELECT instance_id FROM inventory_instances")}
        if not eligible or len(eligible) != result["inventory"]["eligible_instances"]:
            raise ValueError("Inventory size differs from the run summary or is zero.")

        # corpus_body is the exact JSON serialized by the authorized export path.
        db = connect(db_path, "ro")
        rows = db.execute("""
            SELECT q.instance, q.corpus_body
            FROM qa AS q
            WHERE q.corpus_body IS NOT NULL AND EXISTS (
                SELECT 1 FROM gen_attempts AS a
                WHERE a.instance_id = q.instance
                  AND a.audit_id = json_extract(q.corpus_body, '$.audit_id')
                  AND a.disposition = 'accepted'
            )
        """)
        approved = {(sid, hashlib.sha256(body.encode("utf-8")).digest())
                    for sid, body in rows}
        unexported = approved.copy()
        seen, valid, invalid_lines = Counter(), set(), []
        file_hash = hashlib.sha256()

        with corpus.open("rb") as stream:
            for line_no, raw in enumerate(stream, 1):
                file_hash.update(raw)
                try:
                    row = json.loads(raw)
                except ValueError:
                    row = None

                sid = row.get("semantic_instance_id") if isinstance(row, dict) else None
                if not isinstance(sid, str) or not sid:
                    invalid_lines.append(line_no)
                    continue

                seen[sid] += 1
                fingerprint = hashlib.sha256(raw.removesuffix(b"\n")).digest()
                if ((sid, fingerprint) in approved
                        and row.get("corpus_export_authorized") is True
                        and row.get("release_status") == "corpus_export"
                        and row.get("kind") == "verified_sector1"):
                    valid.add(sid)
                    unexported.discard((sid, fingerprint))
                else:
                    invalid_lines.append(line_no)

        if engine["_final_snapshot_stat"](snapshot.stat()) != snapshot_stat:
            raise ValueError("The verified database changed during finalization.")
        issues = {
            "missing_valid_ids": sorted(eligible - valid),
            "duplicate_ids": {sid: n for sid, n in seen.items() if n > 1},
            "unexpected_ids": sorted(seen.keys() - eligible),
            "invalid_lines": invalid_lines,
            "unexported_authorized_ids": sorted({sid for sid, _ in unexported}),
        }
        report = {
            "status": "incomplete" if any(issues.values()) else "ready",
            "build_id": build, "inventory_id": state[0],
            "inventory_database_sha256": policy["inventory_database_sha256"],
            "campaign_id": campaign.name,
            "corpus_file": str(corpus),
            "corpus_sha256": file_hash.hexdigest(),
            "eligible_instances": len(eligible),
            "covered_instances": len(eligible & valid),
            "counts": {key: len(value) for key, value in issues.items()},
        }
        engine["_save_json"](run / "corpus_coverage.json", {**report, "issues": issues})
        return report


# Enable only when checking the selected final campaign.
RUN_FINAL_COVERAGE_CHECK = False

if RUN_FINAL_COVERAGE_CHECK:
    coverage = finalize_corpus(result, imported_inputs["source"])
    print(json.dumps(coverage, indent=2))
    if coverage["status"] != "ready":
        raise RuntimeError("Corpus coverage is incomplete. See corpus_coverage.json.")
else:
    print("Final corpus coverage check skipped (RUN_FINAL_COVERAGE_CHECK=False).")


Final corpus coverage check skipped (RUN_FINAL_COVERAGE_CHECK=False).
